In [1]:
# Cell 1 — Cài đặt thư viện
# !pip install -q bm25s underthesea sentence-transformers==3.* transformers>=4.51.0 accelerate tqdm peft bitsandbytes


In [2]:
# Cell 1 - Chạy đầu tiên sau khi Restart Kernel
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2" 

import torch
print(torch.cuda.device_count()) # Lúc này chắc chắn sẽ ra 1

1


/mnt/mmlab2024nas/trantran/.venv/lib/python3.10/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [3]:
import os, torch
print(os.environ.get("CUDA_VISIBLE_DEVICES"))
print(torch.cuda.device_count())          # số GPU mà process THẤY
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

2
1
0 NVIDIA RTX A4000


In [4]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

import faulthandler
faulthandler.enable()

In [ ]:
# Cell 2 — Imports & Config
import os, json, glob, random, zipfile, re

import numpy as np
# SUA (moi): giam phan manh (fragmentation) VRAM tren server DUNG CHUNG - dat TRUOC khi torch
# khoi tao CUDA context, giup PyTorch de doi lai bo nho da giai phong hon, tranh OOM "gia" do
# fragmentation ngay ca khi con du tong dung luong trong.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# ---- Chỉnh lại các đường dẫn này cho khớp với dataset trên Kaggle ----
DATA_DIR = "data/raw"      # thư mục chứa train.json, warmup.json, ...

# ĐÃ SỬA: corpus giờ CHỈ nhận output ĐÃ TIỀN XỬ LÝ của pipeline 3 file main.py/preprocess.py/
# md_to_json.py -> thư mục "processed-contexts/", mỗi file context_*.json có field
# "passage_segments": list[dict] {"path", "content", "segment_type"}, KHÔNG còn field "passage"
# thô nữa. Corpus context_*.json/selected-contexts.zip CŨ (chưa qua pipeline này) KHÔNG còn
# được dùng trực tiếp — xem load_corpus() ở Cell 4.
CONTEXT_DIR = "data/processed/processed-contexts"        # thư mục chứa output của DSC-preprocessing (processed-contexts/context_*.json)
CONTEXTS_ZIP = "data/processed/processed-contexts.zip"   # (tuỳ chọn) nếu để dạng zip, sẽ tự giải nén ở Cell 3

TRAIN_PATH = os.path.join(DATA_DIR, "train.json")
WARMUP_PATH = os.path.join(DATA_DIR, "warmup.json")
PUBLIC_TEST_PATH = os.path.join(DATA_DIR, "public-official.json")
PRIVATE_TEST_PATH = os.path.join(DATA_DIR, "private-official.json")

TOP_K_STAGE1 = 30     # số ứng viên lấy ra ở stage-1 (BM25 / dense) để đưa vào reranker
TOP_K_FINAL = 5       # số document_id trả về cuối cùng (ràng buộc cứng của đề bài)
NEG_PER_POS = 10        # (ĐÃ TĂNG 7 -> 15) số negative cho MỖI positive — cũng chính là (kích
                        # thước group - 1): mỗi group = 1 positive + NEG_PER_POS negative CỦA
                        # CÙNG 1 CÂU HỎI (xem Cell 10d/Cell 12: dataloader group-based).
                        # LÝ DO TĂNG: mẫu số của softmax listwise (Cell 13/27) càng rộng thì
                        # tín hiệu tương phản càng gần điều kiện thật lúc inference (reranker
                        # phải phân biệt đúng trong top TOP_K_STAGE1=30 ứng viên, không phải
                        # chỉ 8) — đây là đòn bẩy LỚN NHẤT cho InfoNCE-style loss, quan trọng
                        # hơn hẳn việc tăng số group/step. Group giờ = 16 passage/forward (thay
                        # vì 8) — nếu OOM, giảm lại NEG_PER_POS hoặc tăng GRAD_ACCUM_STEPS
                        # (Cell 13) để bù batch hiệu dụng.

# ---- Mining negative TRỰC TIẾP trên SHORT-chunk BM25 index đã build sẵn ở Stage-1 (Cell 7/8) ----
# Không còn build BM25 index riêng trên long-chunk, và không còn pre-tokenize (underthesea) toàn
# bộ short-chunk của corpus nữa (đây là nguyên nhân chính gây OOM/CPU nặng trước đây, vì phải
# tokenize long-chunk của TOÀN BỘ corpus dù chỉ một phần nhỏ thực sự được dùng làm training pair).
# Cách mới (xem Section 4): với mỗi câu hỏi, dùng lại đúng bm25_scores_all() (short-chunk, đã có
# sẵn từ Stage-1) để tìm (a) short-chunk điểm cao nhất trong document gold (positive) và (b) các
# document khác có short-chunk điểm cao nhất (hard negative, tối đa NEG_MINING_POOL_K document khác
# nhau). Sau đó dùng trực tiếp short-chunk trúng tuyển
# (Cell 10c) — hàm này gọi thẳng short-chunk trực tiếp (Cell 6b) để ghép short-chunk trúng tuyển
# với path của chính nó ("path | short-chunk"), ĐÚNG Y HỆT cách
# select_relevant_chunks (Cell 6b) chọn short-chunk lúc inference thật (Cell 18) — training pairs và
# inference giờ dùng CHUNG một cách tạo long-chunk, không còn lệch nhau nữa.
# Vì giờ rẻ hơn nhiều, có thể tăng NEG_MINING_POOL_K (vd. 15-20) để có nhiều lựa chọn resample negative
# đa dạng hơn giữa các epoch mà không tốn thêm chi phí tokenize nào.
NEG_MINING_POOL_K = 100  # ĐÃ TĂNG (50 -> 100): pool tĩnh 50 document/câu hỏi bị tái sử
                          # dụng (chỉ re-score ở TẦNG 1.5, KHÔNG re-mine) suốt cả 3 epoch -> không
                          # gian negative quá hẹp so với corpus thật, dễ khiến model học thuộc đặc thù
                          # của đúng 50 candidate đó thay vì ranh giới tổng quát (nghi ngờ là một phần
                          # nguyên nhân dev recall giảm dần qua epoch). Tăng lên 100 hầu như KHÔNG tốn
                          # thêm chi phí ở TẦNG 1 (hybrid score đã tính cho TOÀN BỘ corpus, chỉ cắt
                          # top-K nhiều hơn từ mảng đã sort sẵn) — chi phí tăng thêm chỉ ở TẦNG 1.5
                          # (refresh_pool_scores forward pass thêm ~50 candidate/câu hỏi mỗi epoch),
                          # vẫn rẻ hơn hẳn TẦNG 1 (không có bm25.get_scores()/dense encode nào).
MINING_LAMBDA = 0.4      # trọng số BM25 trong HYBRID score dùng cho TOÀN BỘ candidate mining
                          # Score_mining = 0.4*BM25_norm + 0.6*Dense_norm

# ---- Config TẦNG 1.5 (MỚI) — refresh_pool_scores(): forward pass reranker (rẻ) trên candidate đã
# ---- có sẵn trong neg_pool (TẦNG 1) để cập nhật "độ khó hiện tại" theo checkpoint đang train. ----
POOL_SCORE_BATCH_SIZE = 128   # batch size khi rerank_score() toàn bộ neg_pool của 1 câu hỏi
NEG_HARD_TOP_FRAC = 0.6      # TẦNG 2 chỉ random.sample() negative trong top NEG_HARD_TOP_FRAC
                              # (theo điểm reranker MỚI NHẤT, TẦNG 1.5) của neg_pool -> ưu tiên hard-
                              # negative "khó" dưới góc nhìn hiện tại của model, nhưng vẫn random trong
                              # nhóm đó để có đa dạng giữa các epoch (không học tủ đúng 1 bộ cố định).
                              # Nếu qid chưa có điểm TẦNG 1.5 (chưa refresh) -> TẦNG 2 tự fallback về
                              # random.sample() trên toàn bộ neg_pool như hành vi gốc.

# (MỚI) MIX hard/easy negative thay vì chỉ đẩy khó dần: trước đây 100% n_neg lấy từ top
# NEG_HARD_TOP_FRAC (khó nhất theo điểm reranker HIỆN TẠI) -> có nguy cơ feedback loop (model tự
# chọn negative theo chính điểm số của nó, càng train càng thu hẹp trọng tâm vào 1 vùng hẹp, dễ
# mất khả năng tổng quát trên dev — nghi ngờ là nguyên nhân dev recall epoch cuối luôn thấp hơn).
# NEG_MIX_HARD_FRAC quyết định TỈ LỆ trong n_neg lấy từ hard pool; phần còn lại
# (1 - NEG_MIX_HARD_FRAC) lấy RANDOM ĐỀU trên toàn bộ eligible pool (bao gồm cả candidate "dễ"/hạng
# thấp), giữ model tiếp xúc với cả phổ độ khó thay vì chỉ vùng khó nhất mỗi epoch.
NEG_MIX_HARD_FRAC = 0.4      # 50% negative lấy từ hard pool (top NEG_HARD_TOP_FRAC), 50% còn lại
                              # random đều trên toàn bộ eligible pool (cả dễ lẫn khó).

# (MỚI — SỬA LỖI MẤT CÂN BẰNG HARD/EASY GIỮA CÁC GOLD CỦA CÙNG 1 CÂU HỎI)
# Vấn đề của bản cũ: _sample_negatives_for_qid trả về danh sách negative theo thứ tự
# [hard..., easy...] (hard luôn nằm ĐẦU), còn _groups_for_sample lại CẮT TUẦN TỰ thành từng khối
# liên tiếp neg_per_pos cho mỗi gold. Ví dụ câu hỏi có 2 gold, NEG_PER_POS=7 -> mine 14 negative
# (~6 hard + 8 easy) -> gold thứ 1 nhận gần như TOÀN BỘ hard negative, gold thứ 2 chỉ nhận easy.
# Đây là mất cân bằng KHÔNG CHỦ ĐÍCH: mục tiêu thật sự là kéo TẤT CẢ gold vào top-5 trên CÙNG 1
# candidate pool lúc inference, nên MỌI gold đều phải học cách thắng negative KHÓ, không riêng gold
# đầu tiên.
# Cách sửa (2 lớp, độc lập nhau):
#   (1) _sample_negatives_for_qid giờ SHUFFLE danh sách trả về -> không còn bất kỳ ý nghĩa thứ tự
#       hard/easy nào theo vị trí, mọi cách cắt về sau đều không bị thiên lệch.
#   (2) NEG_GROUP_SHARING quyết định cách gán negative cho từng gold:
#       - "shared"  (MẶC ĐỊNH — Hướng B): mine ĐÚNG neg_per_pos negative cho mỗi câu hỏi và DÙNG
#         CHUNG bộ đó cho MỌI gold của câu hỏi. Mỗi gold đều phải cạnh tranh với cùng một tập
#         negative (cùng tỉ lệ hard/easy) -> khớp đúng điều kiện inference: mọi gold cùng đua trên
#         1 candidate pool. Số group và tổng số pair KHÔNG đổi so với bản cũ.
#       - "block": giữ cơ chế cũ (mine neg_per_pos * n_pos rồi chia cho từng gold) NHƯNG chia theo
#         kiểu round-robin (stride) thay vì cắt khối liên tiếp, nên hard/easy được rải đều cho mọi
#         gold. Dùng khi muốn mỗi gold thấy negative KHÁC NHAU (đa dạng hơn, tốn nhiều passage hơn
#         trong pool top-30).
NEG_GROUP_SHARING = "shared"   # "shared" | "block"

# (ĐÃ SỬA — bỏ logic ưu tiên "vừa hard vừa top-K" ở trên, thay bằng cắt cứng): với loss InfoNCE/
# listwise (Cell 27), negative dùng để TÍNH LOSS chỉ nên lấy trong đúng phạm vi candidate mà pipeline
# thật sự đưa vào reranker ở inference (top TOP_K_STAGE1 = 30 ứng viên stage-1), để train/inference
# khớp phân phối negative với nhau — không còn cần đến logic "ưu tiên" (sample nhóm A trước, thiếu
# mới bù nhóm B) như bản trước, vì giờ toàn bộ pool để chọn hard/easy đã bị CẮT THẲNG về top-K này
# ngay từ đầu (xem NEG_TRAIN_POOL_TOP_K, _sample_negatives_for_qid — Cell 10d).
# neg_pool vẫn được MINE đầy đủ NEG_MINING_POOL_K=100 document/câu hỏi (TẦNG 1, Cell 10c) — không
# đổi gì ở khâu mining. NHƯNG vì negative dùng để train/tính loss giờ chỉ lấy trong top
# NEG_TRAIN_POOL_TOP_K, TẦNG 1.5 (refresh_pool_scores, Cell 10c2) ĐÃ SỬA để CHỈ re-score đúng top
# NEG_TRAIN_POOL_TOP_K candidate đầu của neg_pool mỗi epoch, KHÔNG re-score cả 100 nữa (re-score cả
# 100 là dư thừa — 70 candidate còn lại không bao giờ được chọn làm negative train nên forward pass
# trên chúng chỉ tốn compute vô ích). neg_pool đã được sort giảm dần theo hybrid/dense score GỐC
# ngay lúc mine (_hybrid_negative_pool, Cell 10c), nên top NEG_TRAIN_POOL_TOP_K vị trí ĐẦU luôn đúng
# là top-30 ứng viên stage-1 (dense retrieval) ban đầu. PHẦN 70 candidate còn lại (vị trí 30-99) vẫn
# nằm trong neg_pool GỐC (giữ nguyên trong fit_candidate_pools, phòng khi cần tăng lại
# NEG_TRAIN_POOL_TOP_K sau này mà không phải mine lại) nhưng KHÔNG còn được re-score/dùng làm
# negative train/tính loss nữa.
NEG_TRAIN_POOL_TOP_K = TOP_K_STAGE1   # = 30: chỉ cắt/tính loss trong top 30 của neg mining pool
                                       # (mining pool vẫn giữ nguyên kích thước 100, xem
                                       # NEG_MINING_POOL_K ở trên).

# ---- Config lọc "nghi ngờ false-negative" bằng XÁC NHẬN CHÉO 2 nguồn ĐỘC LẬP (MỚI) ----
# Vấn đề: nếu chỉ dùng điểm reranker hiện tại (TẦNG 1.5) để loại negative "quá giống positive",
# model đang train sẽ tự quyết định xem candidate nào là false-negative -> feedback loop (model sai
# lại tự "xác nhận" cái sai đó, càng train càng lệch). Cách khắc phục: CHỈ loại 1 candidate ra khỏi
# neg_pool khi CẢ 2 nguồn ĐỘC LẬP cùng đồng thuận nó có khả năng là positive thật:
#   (a) điểm reranker hiện tại (TẦNG 1.5, model ĐANG train) rất cao, VÀ
#   (b) hybrid score gốc (BM25+Dense, TẦNG 1, KHÔNG phụ thuộc model đang train) cũng xếp hạng rất cao
#       trong neg_pool (neg_pool vốn đã được sort giảm dần theo hybrid score lúc mine — Cell 10c/
#       _hybrid_negative_pool — nên chỉ cần dùng VỊ TRÍ trong neg_pool làm proxy cho hybrid rank).
# Nếu chỉ 1 trong 2 nguồn "nghi ngờ" (vd. reranker cao nhưng hybrid rank thấp, hoặc ngược lại) ->
# GIỮ LẠI làm negative bình thường (thận trọng thiên về giữ, không thiên về loại) — tránh việc lọc
# quá tay làm mất hard-negative thật.
# LƯU Ý (ĐÃ SỬA): giá trị 0.9 dưới đây chỉ là NGƯỠNG KHỞI TẠO, dùng ở epoch 1 (lúc chưa có đủ
# known-positive score để tính percentile). Từ epoch 2 trở đi, Cell 14 tự ghi đè biến này bằng
# compute_dynamic_false_neg_threshold() (Cell 10c3) — percentile p10 (làm mượt qua EMA) của
# điểm reranker hiện tại trên known-positive, tự khớp theo độ tự tin thực tế của checkpoint ở
# từng epoch thay vì dùng mãi 1 số cố định.
FALSE_NEG_RERANK_THRESHOLD = 0.9   # (a) NGƯỠNG DỰ PHÒNG (fallback) khi câu hỏi chưa có known-
                                    # positive score riêng (epoch 1, hoặc positive không chunk được).
                                    # ĐÃ SỬA (quan trọng): với loss LISTWISE (Cell 27), sigmoid(logit)
                                    # chỉ đảm bảo ĐÚNG THỨ TỰ trong nội bộ group 8 passage của TỪNG
                                    # câu hỏi — KHÔNG có ràng buộc nào ép giá trị tuyệt đối của nó phải
                                    # nhất quán GIỮA các câu hỏi khác nhau (câu hỏi A có thể có positive
                                    # "quen" quanh 0.7, câu hỏi B quanh 0.95, hoàn toàn hợp lệ dưới
                                    # listwise). Vì vậy 1 ngưỡng TUYỆT ĐỐI dùng chung cho MỌI câu hỏi
                                    # (kể cả bản percentile "động" cũ, compute_dynamic_false_neg_threshold
                                    # — Cell 10c3) là KHÔNG công bằng giữa các câu hỏi: nó sẽ gần như
                                    # không bao giờ lọc được false-negative ở câu hỏi có thang điểm thấp,
                                    # và lọc quá tay ở câu hỏi có thang điểm cao. Giá trị 0.9 ở đây GIỜ
                                    # CHỈ còn là NGƯỠNG DỰ PHÒNG toàn cục (dùng khi câu hỏi chưa có đủ dữ
                                    # liệu riêng) — ngưỡng THẬT SỰ dùng để lọc mỗi câu hỏi được tính
                                    # RIÊNG theo positive-score của CHÍNH câu hỏi đó, xem
                                    # local_false_neg_threshold() (Cell 10c3) và cách nó được gọi trong
                                    # _suspected_false_negatives (Cell 10d).
FALSE_NEG_HYBRID_TOP_FRAC = 0.3    # (b) VÀ candidate phải nằm trong top 30% ĐẦU neg_pool theo hybrid
                                    # rank gốc (TẦNG 1) — tức "được cả BM25/Dense lẫn reranker đồng
                                    # thuận là rất liên quan", không chỉ riêng model đang train nghĩ vậy.
                                    # (Điều kiện (b) này KHÔNG bị ảnh hưởng bởi vấn đề calibration của
                                    # listwise loss vì hybrid rank không phụ thuộc model đang train.)
FALSE_NEG_LOCAL_MARGIN = 0.0       # (MỚI) trừ thêm margin này (thang sigmoid [0,1]) khỏi positive-
                                    # floor RIÊNG của từng câu hỏi khi tính local_false_neg_threshold
                                    # (Cell 10c3), để không lọc quá sát ngay tại positive yếu nhất của
                                    # chính câu hỏi đó. 0.0 = ngưỡng = đúng bằng positive-floor (mặc
                                    # định, đơn giản, thiên về lọc); tăng lên (vd 0.05) nếu muốn khoan
                                    # dung hơn (loại ít candidate hơn, giữ lại nhiều hard-negative hơn).

# ---- Config LISTWISE LOSS (MỚI — thay cho BCE pointwise ở Cell 13) ----
# Loss giờ là softmax cross-entropy TRÊN TỪNG GROUP (1 positive + NEG_PER_POS negative CỦA CÙNG 1
# CÂU HỎI, xem Cell 10d/Cell 12):
#     L = -log( exp(s_pos/T) / sum_j exp(s_j/T) )
# tức model chỉ cần học ĐÚNG THỨ TỰ trong nội bộ mỗi câu hỏi (positive phải cao hơn mọi negative
# của chính câu hỏi đó), thay vì phải ép logit từng cặp về đúng 0/1 tuyệt đối như BCE. Đây chính là
# thứ mà metric của đề bài (recall/precision trên top-5 XẾP HẠNG) đo, nên tín hiệu train khớp với
# tín hiệu đánh giá hơn hẳn.
#
# HỆ QUẢ QUAN TRỌNG (đọc trước khi tune threshold): softmax cross-entropy chỉ ràng buộc HIỆU
# giữa các logit trong cùng group, KHÔNG ràng buộc giá trị tuyệt đối -> sigmoid(logit) sau khi
# train listwise KHÔNG còn là xác suất hiệu chỉnh (calibrated) nữa, và sẽ KHÔNG còn dồn về sát
# 1.0 như khi train BCE. Thứ tự xếp hạng không đổi (sigmoid đơn điệu) nên pipeline top-k vẫn chạy
# đúng, nhưng RERANK_THRESHOLD = 0.99 bên dưới là NGƯỠNG CỦA MODEL TRAIN BẰNG BCE — phải tune lại
# (Cell 19b) nếu muốn bật lại bước lọc theo ngưỡng. Hiện apply_rerank_filter (Cell 13) đang TẮT lọc
# theo ngưỡng (chỉ cắt top_k_final) nên mặc định không bị ảnh hưởng.
LISTWISE_TEMPERATURE = 1.0   # T trong công thức trên. T < 1 (vd 0.5) làm phân phối "nhọn" hơn ->
                              # phạt nặng hơn khi negative lọt lên trên positive; T > 1 làm mượt hơn.
                              # Với cross-encoder logit thang ~[-10, 10] thì T = 1.0 là mặc định an
                              # toàn; chỉ hạ T khi thấy loss giảm quá chậm.
LISTWISE_BCE_WEIGHT = 0.0    # (TUỲ CHỌN) trọng số của một số hạng BCE pointwise cộng thêm vào loss
                              # listwise: L = L_listwise + w * L_BCE. Đặt w > 0 (vd 0.1) nếu bạn MUỐN
                              # giữ lại tính hiệu chỉnh của sigmoid(logit) để còn lọc theo
                              # RERANK_THRESHOLD; đặt 0.0 (mặc định) nếu chỉ quan tâm thứ hạng.

# ---- Config Stage-1 HYBRID retrieval (BM25 + Dense, theo mục 2.2.1) ----
EMBED_MODEL_NAME = "AITeamVN/Vietnamese_Embedding_v2"   # dense encoder cho stage-1
EMBED_BATCH_SIZE = 32
LAMBDA_HYBRID = 0.4   # trọng số BM25 trong Score = lambda*BM25_norm + (1-lambda)*Dense_norm
                       # (giá trị mặc định, sẽ được tune lại trên tập dev ở Cell 8c)

# ---- Config Stage-2 filtering (mục 2.2.2): giữ nguyên reranker BAAI/bge-reranker-v2-m3, chỉ ----
# ---- thêm bước lọc theo threshold sau khi rerank.                                        ----
RERANK_TOP_N = 10        # chỉ xét top-10 document điểm reranker cao nhất trước khi lọc
RERANK_THRESHOLD = 0.99  # ngưỡng "tự tin cao": chỉ giữ candidate có điểm reranker > ngưỡng này
RERANK_FALLBACK_K = 2    # nếu không candidate nào vượt ngưỡng -> lấy top-2 theo điểm reranker
                          # (tránh trả về tập rỗng). Cả 3 giá trị trên sẽ được tune lại trên
                          # tập dev ở Cell 19b, sau khi đã có pipeline hoàn chỉnh.

# ---- Config chunking 2 TẦNG theo đúng paper ViDRILL ----
# Corpus có văn bản dài tới hàng triệu ký tự (median ~23.000, xem phân tích EDA), vượt xa context
# 2304 token của BAAI/bge-reranker-v2-m3 -> bắt buộc chunk trước khi đưa vào cả retrieval lẫn reranker.
# Cả 2 tầng đều dựa trên cấu trúc pháp lý phân cấp Điều -> Khoản -> Điểm (giữ nguyên logic cũ),
# chỉ khác nhau về kích thước mục tiêu, có/không overlap, và có/không được phép gộp nhiều đơn vị lại.

# Long chunking (Stage-2 / reranker): các đơn vị Điều/Khoản/Điểm liên tiếp được GỘP lại tới ~2000
# ký tự để cung cấp đủ ngữ cảnh cho cross-encoder. Overlap giữa 2 chunk liên tiếp = ĐÚNG 1 CÂU HOÀN
# CHỈNH (câu cuối của chunk trước lặp lại làm câu đầu chunk sau), không bao giờ cắt đứt giữa câu.
LONG_CHUNK_CHARS = 3000

# Short chunking (Stage-1 / retrieval): mỗi passage_segment (dict {"path", "content", "segment_type"}
# — output của md_to_json.py/preprocess.py, xem load_corpus() ở Cell 4) là 1 đơn vị ĐỘC LẬP, tối đa
# 2000 ký tự (ĐÃ TĂNG từ 1000 -> 2000; path prefix + content, riêng segment "structured" còn được
# gộp thêm root_preamble ở đầu — xem ROOT_PREAMBLE_CHARS và chunk_document_short_from_segments ở
# Cell 4b), KHÔNG gộp nhiều segment lại, KHÔNG overlap. Nội dung nào vượt quá ngân sách còn lại thì
# tách tiếp xuống Khoản rồi Điểm, và nếu xuống tới Điểm vẫn còn dài thì mới cắt theo câu hoàn chỉnh
# (clause-based segmentation, split_legal_unit_recursive — Cell 4b), giống hệt logic cũ, chỉ đổi
# ngân sách ký tự.
SHORT_CHUNK_CHARS = 2000  # NGAN SACH TONG toi da cua 1 short-chunk Stage-1 (content + path + root_preamble
                          # gop them, neu co) -- dung de dat ten file cache (Cell 7) va cho ham
                          # chunk_document_short() CU (khong con duoc goi trong luong chinh, chi giu lai
                          # de tuong thich nguoc). Luong chinh (chunk_document_short_from_segments -- Cell
                          # 4b) gio dung 3 ngan sach RIENG theo segment_type ben duoi, duoc thiet ke sao
                          # cho TONG DO DAI short-chunk cuoi cung cua MOI segment_type deu <= SHORT_CHUNK_CHARS
                          # (2000) nay:
                          #   - "root_preamble": content duoc cat toi da 2000 ky tu, KHONG gan them gi
                          #     (chinh no da la root_preamble cua document, khong can gop root_preamble
                          #     vao chinh no) -> tong toi da 2000.
                          #   - "unstructured_whole": content duoc cat toi da 1500 ky tu, sau do GAN
                          #     THEM ROOT_PREAMBLE_CHARS (500) ky tu root_preamble cua document vao DAU
                          #     -> tong toi da 1500 + 500 = 2000.
                          #   - "structured": content duoc cat toi da 1000 ky tu, GAN path (neu hop le,
                          #     < MAX_SEGMENT_PATH_CHARS = 500 ky tu) roi GAN THEM ROOT_PREAMBLE_CHARS
                          #     (500) ky tu root_preamble cua document vao DAU -> tong toi da
                          #     1000 + 500 (path) + 500 (root_preamble) = 2000. Neu path KHONG hop le
                          #     (>= MAX_SEGMENT_PATH_CHARS) thi path do da duoc GOP THANG vao content
                          #     TRUOC KHI cat (xem chunk_document_short_from_segments -- Cell 4b), nen
                          #     khong cong don them ngoai ngan sach 1000 ky tu content o tren.

# Ngan sach cat CONTENT (KHONG tinh path/root_preamble gan them) o Stage-1, rieng theo tung
# segment_type -- xem giai thich chi tiet o SHORT_CHUNK_CHARS phia tren va
# chunk_document_short_from_segments (Cell 4b).
SHORT_CHUNK_CHARS_ROOT_PREAMBLE = 2000    # segment_type == "root_preamble"
SHORT_CHUNK_CHARS_UNSTRUCTURED = 2000     # segment_type == "unstructured_whole" (+ 500 root_preamble gan them)
SHORT_CHUNK_CHARS_STRUCTURED = 1000       # segment_type == "structured" (+ path + 500 root_preamble gan them)


# Root-preamble dùng khi tạo short-chunk Stage-1: 1000 ký tự ĐẦU TIÊN của đoạn mở đầu văn bản trước
# heading đầu tiên (segment_type == "root_preamble", xem load_corpus() ở Cell 4) của DOCUMENT
# chứa short-chunk đại diện. Nếu có, đã nằm trong short-chunk ở Stage-1 (xem
# short-chunk trực tiếp — Cell 6b), CÙNG với path của short-chunk đại diện đó rồi mới tới
# chính short-chunk đại diện. Nếu document không có segment root_preamble -> chuỗi rỗng.
ROOT_PREAMBLE_CHARS = 500

# (MỚI) Khi short-chunk ĐẠI DIỆN của short-chunk chính là 1 mảnh thuộc segment_type
# "root_preamble" (đoạn mở đầu văn bản trước heading đầu tiên, KHÔNG có path/cấu trúc phân cấp
# Chương/Điều/Khoản), long-chunk không còn ghép root_preamble(1000 ký tự đầu) + path + short-chunk
# như trường hợp thường nữa (vì root_preamble không có path để bám vào) mà MỞ RỘNG NGỮ CẢNH ngay
# trong chính root_preamble: short-chunk = ROOT_PREAMBLE_EXTEND_CHARS ký tự TRƯỚC short-chunk đại
# diện + short-chunk đại diện + ROOT_PREAMBLE_EXTEND_CHARS ký tự SAU nó (xem short-chunk trực tiếp
# — Cell 6b). Các trường hợp short-chunk đại diện KHÔNG thuộc root_preamble giữ nguyên logic cũ.
ROOT_PREAMBLE_EXTEND_CHARS = 500

# "path" (vd "Chương I > Điều 2 > Khoản 1") bình thường chỉ vài chục ký tự. Nếu path DÀI HƠN
# ngưỡng này (bất thường — coi là NHIỄU của bước preprocessing, ví dụ tách sai path/content ở
# md_to_json.py) thì KHÔNG dùng nó làm prefix cấu trúc nữa: gộp thẳng path đó vào phần content
# (coi như text thường), rồi áp dụng ĐÚNG quy tắc cắt như content — chỉ cắt khi TỔNG vượt quá
# SHORT_CHUNK_CHARS (1000), luôn cắt tại ranh giới câu hoàn chỉnh — xem
# chunk_document_short_from_segments ở Cell 4b. Path vẫn được đưa đi retrieve bình thường, chỉ
# không còn được ưu tiên giữ nguyên như 1 prefix cấu trúc nữa.
MAX_SEGMENT_PATH_CHARS = 500

MAX_CHUNKS_PER_DOC_FOR_RERANK = 1   # số short-chunk liên quan nhất/doc được đưa vào reranker khi rerank ứng viên
TOP_K_STAGE1_CHUNKS = 300           # số short-chunk thô lấy từ BM25 trước khi gộp điểm về cấp document

# ---- Giới hạn số tiến trình song song (server DÙNG CHUNG với người khác) ----
# KHÔNG dùng hết toàn bộ CPU (os.cpu_count()) cho các bước tokenize/mining song song hoá
# (Cell 6a, Cell 7, Cell 11) — mặc định chỉ dùng tối đa MAX_WORKERS tiến trình, chừa lại core cho
# các job khác đang chạy chung server. Có thể override bằng biến môi trường MAX_WORKERS trước khi
# start kernel (vd. `export MAX_WORKERS=8`), hoặc chỉnh số bên dưới tuỳ mức độ rảnh của server.
MAX_WORKERS = int(os.environ.get("MAX_WORKERS", 4))


def get_n_workers(cap: int = MAX_WORKERS) -> int:
    """Số tiến trình song song an toàn: không vượt quá cap và không vượt quá (tổng CPU - 1)."""
    return max(1, min(cap, (os.cpu_count() or 1) - 1))


def _lower_priority():
    """Initializer cho worker: hạ priority (nice) của tiến trình con để nhường CPU cho các job
    khác đang chạy cùng server, thay vì cạnh tranh CPU ngang hàng với chúng."""
    try:
        os.nice(10)
    except (AttributeError, OSError):
        pass  # os.nice không có trên Windows, hoặc không đủ quyền chỉnh nice -> bỏ qua


Device: cuda


/mnt/mmlab2024nas/trantran/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load dữ liệu (queries + corpus)

In [6]:
# Cell 3 — Giải nén corpus ĐÃ TIỀN XỬ LÝ (processed-contexts.zip, nếu để dạng zip) và load train.json
if not os.path.isdir(CONTEXT_DIR) and os.path.exists(CONTEXTS_ZIP):
    os.makedirs(CONTEXT_DIR, exist_ok=True)
    with zipfile.ZipFile(CONTEXTS_ZIP, "r") as zf:
        zf.extractall(CONTEXT_DIR)
    print("Đã giải nén processed-context corpus vào", CONTEXT_DIR)

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

# train.json có dạng: {"<qid>": {"question": "...", "answer": ["177504", ...]}, ...}
train_raw = load_json(TRAIN_PATH)
print("Số câu hỏi trong train:", len(train_raw))

sample_qid = next(iter(train_raw))
print(sample_qid, "->", train_raw[sample_qid])


Số câu hỏi trong train: 7000
86666 -> {'question': 'Thời hạn cấp đăng ký xe máy của người nước ngoài làm việc tại Việt Nam là bao lâu?', 'answer': ['280282']}


In [7]:
# Cell 4 — Load toàn bộ corpus ĐÃ TIỀN XỬ LÝ (processed-contexts/*.json, output THẬT của pipeline
# 3 file main.py -> preprocess.py -> md_to_json.py) vào dict: doc_id (str) -> {"name":..., "link":...,
# "passage_segments": [...]}. KHÔNG còn dùng field "passage" (văn bản thô) nữa.
#
# ĐÃ SỬA (khớp đúng build_segments_json/md_to_segments trong md_to_json.py): mỗi phần tử của
# "passage_segments" giờ là 1 DICT {"path": str, "content": str, "segment_type": str}, KHÔNG còn là
# string dạng "path | content" nữa. "segment_type" chỉ nhận 1 trong 3 giá trị mà md_to_json.py sinh
# ra: "structured" (segment thường, có path phân cấp Chương/Mục/Điều/Khoản/Điểm), "root_preamble"
# (đoạn mở đầu văn bản trước heading đầu tiên, path rỗng) và "unstructured_whole" (toàn văn bản không
# có heading nào, path rỗng). Cả 3 loại đều mang nội dung thật (không phải cấu trúc rỗng) nên đều
# được giữ lại để đưa vào Stage-1 — xem ALLOWED_SEGMENT_TYPES bên dưới.

from typing import Optional

ALLOWED_SEGMENT_TYPES = {"structured", "root_preamble", "unstructured_whole"}

_RE_MULTI_SPACE = re.compile(r" {2,}")   # nhiều dấu cách (" ") liên tiếp
_RE_LONG_DOTS = re.compile(r"\.{5,}")    # dãy dấu chấm "." DÀI HƠN 4 ký tự (tức >= 5 dấu liền nhau)
# Dãy dấu gạch ngang "-" DÀI HƠN 3 ký tự liên tiếp (tức >= 4 dấu liền nhau, vd "--------") -> rút
# gọn còn ĐÚNG 3 ký tự ("---"). Ngưỡng >=4 (thay vì >=2) để KHÔNG đụng tới các dấu "-" đơn lẻ hay
# đôi vốn thường xuất hiện trong công thức/phép trừ/khoảng số (vd "a - b", "5-10", "Điều 5-7").
_RE_LONG_DASHES = re.compile(r"-{4,}")
# Chuỗi quốc hiệu/tiêu ngữ lặp lại ở đầu rất nhiều văn bản pháp luật, không mang giá trị retrieval
# -> xoá hẳn. Dùng \s+ giữa các từ (thay vì khoảng trắng literal) để khớp cả khi có xuống dòng hoặc
# nhiều khoảng trắng xen giữa (trước khi _RE_MULTI_SPACE chạy), tránh xoá sót do lệch whitespace.
_RE_NATIONAL_HEADER = re.compile(
    r"CỘNG\s+HÒA\s+XÃ\s+HỘI\s+CHỦ\s+NGHĨA\s+VIỆT\s+NAM\s+"
    r"Độc\s+lập\s*-\s*Tự\s+do\s*-\s*Hạnh\s+phúc\s*",
    re.UNICODE,
)


def _clean_text(text: str) -> str:
    """Tiền xử lý bổ sung cho MỖI field text (path hoặc content) lấy từ processed-context:
    1) Xoá chính xác chuỗi quốc hiệu/tiêu ngữ "CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM Độc lập - Tự do -
       Hạnh phúc " (nếu có) — không mang thông tin pháp lý, chỉ là header lặp lại ở đầu văn bản.
    2) Gộp nhiều dấu cách liên tiếp ("    ", ...) thành đúng 1 dấu cách " ".
    3) Dãy dấu gạch ngang dài hơn 3 ký tự (vd "----------", dùng để ngăn cách hình thức trong văn
       bản gốc) -> rút gọn còn ĐÚNG 3 ký tự ("---"). Chỉ áp dụng cho dãy >=4 dấu "-" liền nhau nên
       KHÔNG đụng tới dấu "-" đơn/đôi trong công thức, phép trừ, khoảng số (vd "a - b", "5-10").
    4) Dãy dấu chấm dài hơn 4 ký tự (vd "......") -> rút gọn còn ĐÚNG 3 ký tự ("...").
    """
    text = _RE_NATIONAL_HEADER.sub("", text or "")
    text = _RE_MULTI_SPACE.sub(" ", text)
    text = _RE_LONG_DASHES.sub("---", text)
    text = _RE_LONG_DOTS.sub("...", text)
    return text.strip()


def clean_passage_segment(seg: dict) -> Optional[dict]:
    """Làm sạch 1 passage_segment dict {"path", "content", "segment_type"} (output của
    md_to_json.py). Trả về None nếu segment_type không hợp lệ hoặc content rỗng sau khi clean
    (segment không mang thông tin gì để đi retrieve)."""
    if not isinstance(seg, dict):
        return None
    if seg.get("segment_type") not in ALLOWED_SEGMENT_TYPES:
        return None
    content = _clean_text(seg.get("content", ""))
    if not content:
        return None
    return {
        "path": _clean_text(seg.get("path", "")),
        "content": content,
        "segment_type": seg["segment_type"],
    }


def _extract_root_preamble(segments):
    """Nối nội dung các segment "root_preamble" (đoạn mở đầu văn bản trước heading đầu tiên,
    xem ALLOWED_SEGMENT_TYPES ở trên) của 1 document, rồi cắt lấy ĐÚNG ROOT_PREAMBLE_CHARS
    (1000) ký tự đầu tiên. Nếu có, đã nằm trong short-chunk ở Stage-1 (short-chunk trực tiếp,
    Cell 6b). Trả về "" nếu document không có segment root_preamble nào (vd. văn bản
    "unstructured_whole" hoặc không có đoạn mở đầu trước heading)."""
    parts = [s["content"] for s in (segments or []) if s.get("segment_type") == "root_preamble" and s.get("content")]
    if not parts:
        return ""
    return " ".join(parts)[:ROOT_PREAMBLE_CHARS]


def _extract_root_preamble_full(segments):
    """Giống _extract_root_preamble() ở trên nhưng KHÔNG cắt còn ROOT_PREAMBLE_CHARS — trả về TOÀN
    BỘ nội dung root_preamble (nối các segment segment_type == "root_preamble" bằng dấu cách).
    Dùng làm nguồn để MỞ RỘNG NGỮ CẢNH (500 ký tự trước/sau) khi short-chunk đại diện của long-chunk
    chính là 1 mảnh thuộc root_preamble (xem ROOT_PREAMBLE_EXTEND_CHARS ở Cell 2 và
    short-chunk trực tiếp ở Cell 6b) — root_preamble không có path/cấu trúc phân cấp nên không thể
    bám theo path như các segment "structured" khác, phải mở rộng trực tiếp trong chính văn bản
    root_preamble. Trả về "" nếu document không có segment root_preamble nào."""
    parts = [s["content"] for s in (segments or []) if s.get("segment_type") == "root_preamble" and s.get("content")]
    if not parts:
        return ""
    return " ".join(parts)


def load_corpus(context_dir):
    corpus = {}
    files = sorted(glob.glob(os.path.join(context_dir, "**", "context_*.json"), recursive=True))
    assert len(files) > 0, (
        f"Không tìm thấy file context_*.json (processed-context) trong {context_dir}. "
        f"Corpus giờ CHỈ nhận output của pipeline main.py/preprocess.py/md_to_json.py (thư mục "
        f"processed-contexts/, mỗi file có field \"passage_segments\": list[dict]), KHÔNG còn dùng "
        f"corpus context_*.json thô (field \"passage\") nữa."
    )
    for fp in tqdm(files, desc="Loading processed-context corpus files"):
        data = load_json(fp)
        # một file context_*.json có thể chứa 1 văn bản (dict) hoặc nhiều văn bản (list)
        items = data if isinstance(data, list) else [data]
        for item in items:
            doc_id = str(item["id"])
            raw_segments = item.get("passage_segments", []) or []
            segments = [s for s in (clean_passage_segment(seg) for seg in raw_segments) if s]
            corpus[doc_id] = {
                "name": item.get("name", ""),
                "passage_segments": segments,
                "link": item.get("link", ""),
            }
    return corpus

corpus = load_corpus(CONTEXT_DIR)
print("Số văn bản trong corpus (trước khi lọc):", len(corpus))

# ---- Lọc các văn bản không còn passage_segments nào (rỗng), tương đương bước lọc "passage rỗng" cũ ----
empty_doc_ids = [d for d, v in corpus.items() if not v["passage_segments"]]
if empty_doc_ids:
    gold_ids_all = set()
    for v in train_raw.values():
        gold_ids_all.update(str(a) for a in v["answer"])
    overlap_empty_gold = set(empty_doc_ids) & gold_ids_all
    print(f"Cảnh báo: {len(empty_doc_ids)} văn bản có passage_segments RỖNG -> sẽ loại khỏi corpus.")
    if overlap_empty_gold:
        print(f"  !! {len(overlap_empty_gold)} trong số này lại là gold answer trong train.json: "
              f"{sorted(overlap_empty_gold)[:10]}{'...' if len(overlap_empty_gold) > 10 else ''}")
        print("  -> các câu hỏi có gold answer rỗng này sẽ KHÔNG BAO GIỜ đạt recall=1 dù pipeline đúng, "
              "cần kiểm tra lại nguồn dữ liệu gốc nếu muốn khắc phục.")
    for d in empty_doc_ids:
        del corpus[d]

print("Số văn bản trong corpus (sau khi lọc):", len(corpus))
doc_ids_all = list(corpus.keys())

# ---- doc_id -> 1000 ký tự đầu của root_preamble (đoạn mở đầu văn bản), dùng để ghép
# short-chunk ở Stage-1 (xem short-chunk trực tiếp, Cell 6b) ----
doc_root_preamble_of = {d: _extract_root_preamble(v["passage_segments"]) for d, v in corpus.items()}

Loading processed-context corpus files: 100%|██████████| 8532/8532 [00:33<00:00, 252.33it/s]

Số văn bản trong corpus (trước khi lọc): 8532
Cảnh báo: 21 văn bản có passage_segments RỖNG -> sẽ loại khỏi corpus.
  !! 6 trong số này lại là gold answer trong train.json: ['10533', '131890', '149317', '263763', '288457', '55497']
  -> các câu hỏi có gold answer rỗng này sẽ KHÔNG BAO GIỜ đạt recall=1 dù pipeline đúng, cần kiểm tra lại nguồn dữ liệu gốc nếu muốn khắc phục.
Số văn bản trong corpus (sau khi lọc): 8511


## 1b. Chunk văn bản dài (dành riêng cho Stage-2 / reranker)

BAAI/bge-reranker-v2-m3 chỉ hỗ trợ tối đa 2304 token, trong khi corpus có văn bản dài tới hàng triệu ký tự
(xem phân tích phân bố độ dài — median ~23.000 ký tự, một số file >300.000 ký tự là các bộ luật gộp
nhiều điều/chương vào 1 file).

**Chiến lược 3 tầng ưu tiên khi chunk 1 văn bản:**
1. Nếu văn bản có cấu trúc `"Điều X."` rõ ràng (≥2 lần xuất hiện) → tách theo ranh giới Điều, rồi gộp các
   Điều ngắn lại gần `TARGET_CHUNK_CHARS` để không sinh ra quá nhiều chunk vụn.
2. Nếu không có cấu trúc Điều (văn bản thường: bản án, công văn, quyết định...) → tách theo đoạn văn
   (`\n\n`, fallback `\n`), rồi gộp tương tự.
3. Nếu 1 đơn vị (Điều hoặc đoạn văn) vẫn tự nó dài hơn `HARD_MAX_CHUNK_CHARS` → buộc cắt cứng theo ký tự,
   nhưng ưu tiên cắt tại vị trí gần dấu kết câu (`.`, `?`, `!`, `;`) để tránh đứt giữa câu, có overlap để
   không mất ngữ cảnh ở ranh giới.

**Lưu ý quan trọng:** Stage-1 (BM25) vẫn giữ nguyên hoạt động ở cấp **toàn văn bản** như baseline gốc —
BM25 là phương pháp bag-of-words nên không bị giới hạn context như model transformer, chunking ở đây chỉ
cần thiết cho Stage-2 (reranker).


In [8]:
# Cell 4b — Chunking cho pipeline (đã bỏ nhánh LONG-chunking cố định — xem ghi chú bên dưới):
#  - Short chunking (tối đa SHORT_CHUNK_CHARS=1000 ký tự, KHÔNG overlap) -> dùng cho Stage-1 (BM25/retrieval).
#  - Stage-2 (reranker) dùng trực tiếp short-chunk, không tạo long-chunk
#    (chunk_document_long/chunk_long_with_sentence_overlap/get_doc_chunks_long — đã XOÁ vì không
#    còn nơi nào gọi tới nữa). Stage-2 không ghép short-chunk thành đơn vị mới
#    lúc cần dùng (select_relevant_chunks — Cell 6b — cho pipeline chính/eval; short-chunk trực tiếp
#    — Cell 10c — cho mining training pairs), nên không cần tạo long-chunk ở bất kỳ bước nào.
# Việc tách theo cấu trúc pháp lý Điều -> Khoản -> Điểm (các hàm split_* bên dưới) vẫn dùng chung cho
# short-chunking và cho các bước tách câu khác trong notebook.

DIEU_SPLIT_RE = re.compile(r"(?=\n\s*Điều\s+\d+[\.:])", re.UNICODE)
DIEU_COUNT_RE = re.compile(r"(?:^|\n)\s*Điều\s+\d+[\.:]", re.UNICODE)
KHOAN_SPLIT_RE = re.compile(r"(?=\n\s*\d+\.\s)", re.UNICODE)          # ranh giới Khoản: "\n1. ", "\n2. ", ...
DIEM_SPLIT_RE = re.compile(r"(?=\n\s*[a-zđ]\)\s)", re.UNICODE)        # ranh giới Điểm: "\na) ", "\nb) ", ...
_SENTENCE_SPLIT_RE = re.compile(r"(?<=[\.!?;:])\s+|\n+", re.UNICODE)  # ranh giới câu hoàn chỉnh / dòng

# ---- Chuẩn hoá xuống dòng + dấu chấm/chấm lửng lặp, DÙNG CHUNG CHO CẢ NOTEBOOK ----
# Quy tắc:
#  1) "\r\n", "\r" (đơn lẻ) hoặc "\n" (đơn lẻ) đều CHỈ chuyển thành ĐÚNG 1 khoảng trắng " ".
#  2) Dãy dấu chấm DÀI HƠN 3 dấu (vd "......", tức >=4 dấu "." liền nhau) -> thay bằng " ".
#  3) Dãy dấu chấm lửng Unicode "…" lặp lại, xen khoảng trắng (vd "… … …") -> thay bằng " ".
#     (Đây là 2 dạng KHÁC NHAU: dạng 2 là nhiều ký tự "." liền nhau; dạng 3 là ký tự "…" đơn lặp
#     lại nhiều lần, có thể xen khoảng trắng giữa các "…". Không gộp chung logic.)
#  4) Sau các bước trên, nếu còn nhiều dấu " " liên tiếp thì gộp lại thành 1 dấu " " duy nhất.
# áp dụng ĐỒNG BỘ ở mọi nơi tạo ra chunk text CUỐI CÙNG (short-chunk: chunk_document_short
# ghép từ short-chunk: short-chunk trực tiếp ở Cell 6b, short-chunk trực tiếp ở Cell 10c) để
# chunk text đưa vào BM25/dense/reranker không còn lẫn xuống dòng thô hay dãy chấm/chấm lửng thừa,
# dù được tạo ra từ hàm nào.
# CHỈ dùng hàm này ở bước CUỐI (sau khi đã tách xong theo Điều/Khoản/Điểm/đoạn văn/câu) —
# KHÔNG dùng trước hoặc trong lúc tách, vì các bước tách (split_by_dieu/khoan/diem/paragraph,
# split_into_sentences) vẫn cần \n gốc để nhận diện đúng ranh giới.
_NEWLINE_NORM_RE = re.compile(r"\r\n|\r")
_NEWLINE_RUN_RE = re.compile(r"[ \t]*\n[ \t]*(?:\n[ \t]*)*")
_DOT_RUN_RE = re.compile(r"\.{4,}")                    # dãy "." dài hơn 3 dấu (>=4 dấu liền nhau)
_ELLIPSIS_RUN_RE = re.compile(r"…(?:[ \t]*…)+")         # "… …", "… … …", ... (>=2 lần, xen khoảng trắng)
_MULTI_SPACE_RE = re.compile(r" {2,}")                  # nhiều dấu cách liên tiếp -> 1 dấu cách


def normalize_newlines(text):
    """Chuẩn hoá xuống dòng + dãy dấu chấm/chấm lửng lặp, DÙNG CHUNG CHO CẢ NOTEBOOK:
    1) "\r\n"/"\r" -> "\n" trước, sau đó GỘP cả cụm \n liên tiếp gần nhau (kể cả xen khoảng trắng/tab)
       thành ĐÚNG 1 khoảng trắng " ".
    2) Dãy dấu chấm dài hơn 3 dấu ("......", >=4 dấu "." liền nhau) -> thay bằng " ".
    3) Dãy dấu chấm lửng Unicode "…" lặp lại, có thể xen khoảng trắng ("… … …") -> thay bằng " ".
       (Khác với (2): đây là ký tự "…" đơn lặp lại nhiều lần, không phải nhiều dấu "." liền nhau.)
    4) Gộp mọi dấu " " liên tiếp còn sót lại (do (1)/(2)/(3) sinh ra hoặc đã có sẵn) thành 1 dấu " ".
    Dùng ở bước cuối khi trả về chunk text, KHÔNG dùng trước khi split theo cấu trúc pháp lý."""
    text = _NEWLINE_NORM_RE.sub("\n", text or "")
    text = _NEWLINE_RUN_RE.sub(" ", text)
    text = _DOT_RUN_RE.sub(" ", text)
    text = _ELLIPSIS_RUN_RE.sub(" ", text)
    text = _MULTI_SPACE_RE.sub(" ", text)
    return text.strip()


def split_into_sentences(text):
    """Tách text thành list câu hoàn chỉnh, dựa trên dấu kết câu (. ! ? ; :) và ranh giới dòng."""
    text = (text or "").strip()
    if not text:
        return []
    return [p.strip() for p in _SENTENCE_SPLIT_RE.split(text) if p.strip()]


def split_by_dieu(text):
    return [p.strip() for p in DIEU_SPLIT_RE.split(text) if p.strip()]


def split_by_khoan(text):
    return [p.strip() for p in KHOAN_SPLIT_RE.split(text) if p.strip()]


def split_by_diem(text):
    return [p.strip() for p in DIEM_SPLIT_RE.split(text) if p.strip()]


def split_by_paragraph(text):
    parts = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    if len(parts) < 2:
        parts = [p.strip() for p in text.split("\n") if p.strip()]
    return parts


def split_top_level_units(text):
    """Tách văn bản thành các đơn vị Điều (nếu có cấu trúc Điều rõ ràng, >=2 Điều), nếu không thì
    theo đoạn văn — giống hệt logic gốc, chỉ đổi tên cho rõ vai trò "top-level" trong pipeline 2 tầng."""
    n_dieu = len(DIEU_COUNT_RE.findall(text))
    if n_dieu >= 2:
        return split_by_dieu(text)
    units = split_by_paragraph(text)
    if len(units) < 2:
        units = [text]
    return units


def split_by_sentence_hard(text, max_chars):
    """Cắt 1 đoạn text không còn tách được theo Khoản/Điểm thành các đoạn <= max_chars, LUÔN cắt tại
    ranh giới câu hoàn chỉnh (không bao giờ cắt giữa câu), KHÔNG overlap — dùng làm bước cuối cùng khi
    đã xuống tới cấp thấp nhất của cấu trúc pháp lý mà vẫn còn dài."""
    sentences = split_into_sentences(text)
    if not sentences:
        return [text[:max_chars].strip()] if text.strip() else []

    chunks, buf = [], ""
    for sent in sentences:
        if len(sent) > max_chars:
            if buf:
                chunks.append(buf); buf = ""
            start = 0
            while start < len(sent):
                chunks.append(sent[start:start + max_chars].strip())
                start += max_chars
            continue
        if len(buf) + len(sent) + 1 <= max_chars:
            buf = (buf + " " + sent).strip() if buf else sent
        else:
            chunks.append(buf)
            buf = sent
    if buf:
        chunks.append(buf)
    return [c for c in chunks if c]


def split_legal_unit_recursive(unit_text, max_chars):
    """Tách đệ quy 1 đơn vị (Điều/đoạn văn) xuống Khoản rồi Điểm nếu vẫn dài hơn max_chars. Nếu
    xuống tới Điểm vẫn còn dài -> cắt theo câu hoàn chỉnh (split_by_sentence_hard)."""
    if len(unit_text) <= max_chars:
        return [unit_text]

    khoan_parts = split_by_khoan(unit_text)
    if len(khoan_parts) >= 2:
        out = []
        for kp in khoan_parts:
            out.extend(split_legal_unit_recursive(kp, max_chars))
        return out

    diem_parts = split_by_diem(unit_text)
    if len(diem_parts) >= 2:
        out = []
        for dp in diem_parts:
            out.extend(split_legal_unit_recursive(dp, max_chars))
        return out

    return split_by_sentence_hard(unit_text, max_chars)


# =========================================================================================
# SHORT CHUNKING (Stage-1 / retrieval, và là được dùng trực tiếp cho Stage-2 — xem
# select_relevant_chunks ở Cell 6b, short-chunk trực tiếp ở Cell 10c) — mỗi Điều/Khoản/Điểm
# là 1 đơn vị ĐỘC LẬP, tối đa SHORT_CHUNK_CHARS ký tự, KHÔNG gộp, KHÔNG overlap. Đơn vị dài quá thì
# tách tiếp Khoản -> Điểm -> câu.
# =========================================================================================
def chunk_document_short(doc_id, text):
    text = (text or "").strip()
    if not text:
        return []
    top_units = split_top_level_units(text)
    chunks = []
    for u in top_units:
        chunks.extend(split_legal_unit_recursive(u, SHORT_CHUNK_CHARS))
    # Chuẩn hoá xuống dòng -> khoảng trắng Ở BƯỚC CUỐI (xem normalize_newlines phía trên) — đồng bộ
    # với short-chunk trực tiếp
    # (Cell 10c).
    return [(f"{doc_id}#s{i}", normalize_newlines(c)) for i, c in enumerate(chunks) if c.strip()]


# ĐÃ SỬA: corpus giờ load từ processed-context THẬT (Cell 4, output của main.py/preprocess.py/
# md_to_json.py) nên MỖI văn bản đã có sẵn "passage_segments" — list[dict] {"path", "content",
# "segment_type"} (xem collect_segments()/md_to_segments() trong md_to_json.py), KHÔNG còn là
# string "path | content" nữa. Hàm dưới đây thay thế chunk_document_short() ở Stage-1 (Cell 7 build
# BM25 index): dùng THẲNG từng passage_segment làm 1 đơn vị short-chunk gốc (mọi segment_type trong
# ALLOWED_SEGMENT_TYPES — Cell 4 — đều đã được lọc/giữ lại từ lúc load_corpus).
#
# path (vd "Chương I > Điều 2 > Khoản 1") bình thường chỉ vài chục ký tự. Nếu path DÀI HƠN
# MAX_SEGMENT_PATH_CHARS (Cell 2, mặc định 500) thì coi đây là BẤT THƯỜNG/NHIỄU của bước
# preprocessing (vd md_to_json.py tách sai ranh giới path/content) -> KHÔNG dùng nó làm path riêng
# nữa mà GỘP THẲNG vào content, coi như text thường (path == "" trong trường hợp này).
#
# ĐÃ SỬA (quan trọng, MỚI): việc CẮT content vẫn xảy ra TRƯỚC, CHỈ dựa trên content thuần (KHÔNG
# tính path vào ngân sách cắt) — dùng ĐÚNG logic cũ (Khoản -> Điểm -> câu hoàn chỉnh,
# split_legal_unit_recursive phía trên, LUÔN cắt tại ranh giới câu hoàn chỉnh — không bao giờ cắt
# giữa câu). NHƯNG NGAY SAU KHI CẮT XONG (ở mỗi mảnh con sinh ra), nếu path hợp lệ
# (< MAX_SEGMENT_PATH_CHARS, tức không bị coi là nhiễu) thì path đó được GẮN NGAY vào ĐẦU mảnh con
# bằng " | " ("path | content-đã-cắt") để tạo thành short-chunk text CUỐI CÙNG — chính text này mới
# được đưa vào BM25/dense index ở Stage-1 (Cell 7). Nhờ vậy, khi 1 short-chunk được chọn làm đại
# diện để đưa vào Stage-2/reranker, KHÔNG cần gắn path thêm lần nữa (xem short-chunk trực tiếp ở
# Cell 6b) — path đã có sẵn ngay trong short-chunk text rồi. path gốc vẫn được trả về riêng (xem
# short_chunk_path_of ở Cell 7) để tham khảo/debug, không còn là thành phần bắt buộc ở Stage-2 nữa.

def chunk_document_short_from_segments(doc_id, segments):
    """Trả về list[(chunk_id, text, path, segment_type)]:
      - text: content ĐÃ CẮT theo ngân sách RIÊNG cho từng segment_type (xem
        SHORT_CHUNK_CHARS_ROOT_PREAMBLE/UNSTRUCTURED/STRUCTURED — Cell 2), sau đó GẮN THÊM prefix
        vào ĐẦU bằng " | " tuỳ segment_type — đây là text thật sự đưa vào index Stage-1 (Cell 7) VÀ
        cũng chính là short-chunk dùng thẳng ở Stage-2 (Cell 6b), không cần gắn lại gì nữa.
        ĐÃ SỬA (MỚI, quan trọng): mỗi segment_type có 1 công thức cắt/gắn RIÊNG, được thiết kế để
        TỔNG ĐỘ DÀI short-chunk cuối cùng LUÔN <= 2000 ký tự (SHORT_CHUNK_CHARS) — đây là ngân sách
        chung mà CẢ BM25 lẫn dense index ở Stage-1 đều nhận:
          * "root_preamble": content cắt tối đa 2000 ký tự (SHORT_CHUNK_CHARS_ROOT_PREAMBLE), KHÔNG
            gắn thêm gì (chính segment này đã LÀ root_preamble của document) -> tổng tối đa 2000.
          * "unstructured_whole": content cắt tối đa 1500 ký tự (SHORT_CHUNK_CHARS_UNSTRUCTURED),
            sau đó GẮN THÊM ROOT_PREAMBLE_CHARS (500) ký tự root_preamble của document (đã
            precompute ở doc_root_preamble_of — Cell 4) vào ĐẦU -> tổng tối đa 1500 + 500 = 2000.
          * "structured": content cắt tối đa 1000 ký tự (SHORT_CHUNK_CHARS_STRUCTURED), gắn path
            (nếu hợp lệ, < MAX_SEGMENT_PATH_CHARS) rồi GẮN THÊM ROOT_PREAMBLE_CHARS (500) ký tự
            root_preamble vào ĐẦU (trước path) -> tổng tối đa 1000 + 500 (path) + 500 (root_preamble)
            = 2000. Nếu path KHÔNG hợp lệ (>= MAX_SEGMENT_PATH_CHARS) thì path đó đã được GỘP THẲNG
            vào content TRƯỚC KHI cắt (xem đoạn xử lý MAX_SEGMENT_PATH_CHARS bên dưới) nên vẫn nằm
            trong ngân sách 1000 ký tự content, không cộng dồn thêm.
      - path: path gốc của segment nếu path < MAX_SEGMENT_PATH_CHARS, ngược lại "" (path bất thường
        đã bị gộp thẳng vào content ở trên). Giữ lại riêng để tham khảo/debug.
      - segment_type (MỚI): segment_type gốc của segment sinh ra mảnh này ("structured",
        "root_preamble", "unstructured_whole", hoặc "" nếu seg là string fallback định dạng cũ
        không mang segment_type). Dùng để nhận biết short-chunk nào thuộc root_preamble (không có
        path/cấu trúc phân cấp) -> Stage-2 dùng trực tiếp short-chunk bằng
        ROOT_PREAMBLE_EXTEND_CHARS ký tự trước/sau thay vì dùng root_preamble + short-chunk như
        segment "structured". Logic Cell 6b GIỮ NGUYÊN không đổi.
    """
    # ĐÃ SỬA (MỚI): root_preamble (ROOT_PREAMBLE_CHARS ký tự đầu) của CHÍNH document đang chunk —
    # cần cho việc gộp vào short-chunk "structured" bên dưới. doc_root_preamble_of được precompute
    # ở Cell 4 (chạy trước Cell 7, nơi hàm này thực sự được gọi) nên luôn có sẵn trong session; nếu
    # vì lý do nào đó doc_id chưa có trong dict (vd. gọi hàm này độc lập ngoài luồng chính) thì coi
    # như không có root_preamble ("").
    doc_root_preamble = doc_root_preamble_of.get(doc_id, "") if "doc_root_preamble_of" in globals() else ""

    chunks = []
    for seg in segments or []:
        if not seg:
            continue
        # segments đã là dict {"path", "content", "segment_type"} đã clean sẵn từ load_corpus (Cell 4);
        # vẫn hỗ trợ fallback string "path | content" (định dạng cũ) để không phá vỡ pickle cache cũ.
        if isinstance(seg, dict):
            path, content = seg.get("path", ""), seg.get("content", "")
            seg_type = seg.get("segment_type", "")
        else:
            seg = seg.strip()
            if " | " in seg:
                path, content = seg.split(" | ", 1)
            else:
                path, content = "", seg
            seg_type = ""
        path = (path or "").strip()
        content = (content or "").strip()

        # Path >= MAX_SEGMENT_PATH_CHARS (500) là BẤT THƯỜNG (coi là nhiễu của bước preprocessing,
        # ví dụ md_to_json.py tách sai ranh giới path/content) -> KHÔNG dùng nó làm path riêng nữa,
        # mà GỘP THẲNG vào content (coi như text thường). Vẫn đưa đi retrieve như content bình thường,
        # chỉ khác là không còn giữ nguyên như 1 "path" độc lập -> path = "" từ đây trở đi.
        if len(path) >= MAX_SEGMENT_PATH_CHARS:
            content = f"{path} {content}".strip() if content else path
            path = ""

        if not content:
            continue

        # ĐÃ SỬA (MỚI): ngân sách cắt CONTENT (KHÔNG tính path/root_preamble gắn thêm) giờ RIÊNG
        # theo từng segment_type (xem SHORT_CHUNK_CHARS_ROOT_PREAMBLE/UNSTRUCTURED/STRUCTURED — Cell
        # 2), thay vì dùng chung 1 SHORT_CHUNK_CHARS như trước:
        #   - "root_preamble": budget = SHORT_CHUNK_CHARS_ROOT_PREAMBLE (2000) — không gắn thêm gì ở
        #     bước build text bên dưới nên tổng short-chunk cũng tối đa 2000.
        #   - "unstructured_whole": budget = SHORT_CHUNK_CHARS_UNSTRUCTURED (1500) — sẽ gắn thêm
        #     ROOT_PREAMBLE_CHARS (500) ký tự root_preamble ở bước build text bên dưới -> tổng tối đa
        #     1500 + 500 = 2000.
        #   - "structured" (hoặc "" fallback định dạng cũ): budget = SHORT_CHUNK_CHARS_STRUCTURED
        #     (1000) — sẽ gắn thêm path (< MAX_SEGMENT_PATH_CHARS = 500) và ROOT_PREAMBLE_CHARS (500)
        #     ký tự root_preamble ở bước build text bên dưới -> tổng tối đa 1000 + 500 + 500 = 2000.
        # Nhờ vậy, MỌI short-chunk cuối cùng đưa vào BM25/dense index ở Stage-1 (Cell 7) đều đảm bảo
        # tối đa 2000 ký tự, bất kể segment_type. path, segment_type (và root_preamble gắn lúc build
        # text ở dưới) được gắn NGUYÊN VẸN vào cùng MỌI mảnh con tách ra từ content này.
        if seg_type == "root_preamble":
            _content_budget = SHORT_CHUNK_CHARS_ROOT_PREAMBLE
        elif seg_type == "unstructured_whole":
            _content_budget = SHORT_CHUNK_CHARS_UNSTRUCTURED
        else:
            _content_budget = SHORT_CHUNK_CHARS_STRUCTURED
        for part in split_legal_unit_recursive(content, _content_budget):
            chunks.append((part, path, seg_type))

    out = []
    for i, (text, path, seg_type) in enumerate(chunks):
        if not text.strip():
            continue
        # ĐÃ SỬA (MỚI): root_preamble (ROOT_PREAMBLE_CHARS = 500 ký tự đầu của document, xem
        # doc_root_preamble_of — Cell 4) giờ được gộp thêm vào ĐẦU cho CẢ "structured" LẪN
        # "unstructured_whole" (trước đây chỉ "structured"):
        #   - "structured" -> "root_preamble(500) | path(<500) | content-đã-cắt(<=1000)" (path chỉ
        #     gắn nếu hợp lệ, xem đoạn lọc MAX_SEGMENT_PATH_CHARS phía trên).
        #   - "unstructured_whole" -> "root_preamble(500) | content-đã-cắt(<=1500)" (không có path).
        #   - "root_preamble" -> giữ NGUYÊN "content-đã-cắt(<=2000)", KHÔNG gộp gì thêm (chính segment
        #     này đã LÀ root_preamble của document, gộp thêm chính nó là vô nghĩa).
        prefix_parts = []
        if seg_type in ("structured", "unstructured_whole") and doc_root_preamble:
            prefix_parts.append(doc_root_preamble)
        if path:
            prefix_parts.append(path)
        prefix = " | ".join(prefix_parts)
        final_text = f"{prefix} | {text}" if prefix else text
        # ĐÃ SỬA (MỚI): gắn path (và root_preamble nếu có) vào ĐẦU mảnh con (đã cắt) bằng " | "
        # NGAY TẠI ĐÂY, SAU KHI cắt xong -> đây là short-chunk text cuối cùng, dùng cho CẢ Stage-1
        # (BM25/dense index) LẪN Stage-2 (short-chunk = root_preamble + short-chunk này, xem Cell 6b
        # — logic Cell 6b GIỮ NGUYÊN không đổi), không cần gắn path thêm lần nào nữa ở bước sau.
        out.append((f"{doc_id}#s{i}", normalize_newlines(final_text), path, seg_type))
    return out


In [9]:
# Cell 5 — Chuẩn hoá train.json thành list các sample (qid, question, list[answer_id])
def flatten_queries(raw):
    samples = []
    for qid, v in raw.items():
        samples.append({
            "qid": str(qid),
            "question": v["question"],
            "answers": [str(a) for a in v["answer"]],
        })
    return samples

train_samples = flatten_queries(train_raw)
print(train_samples[0])

# Tách train/dev nội bộ để đánh giá offline (đề bài không cho nhãn của public/private test)
random.shuffle(train_samples)
n_dev = max(1, int(0.1 * len(train_samples)))
dev_samples = train_samples[:n_dev]
fit_samples = train_samples[n_dev:]
print(f"fit: {len(fit_samples)} | dev (offline eval): {len(dev_samples)}")


{'qid': '86666', 'question': 'Thời hạn cấp đăng ký xe máy của người nước ngoài làm việc tại Việt Nam là bao lâu?', 'answers': ['280282']}
fit: 6300 | dev (offline eval): 700


## 2. Stage 1 — Hybrid retrieval (BM25 + Dense)

Bài toán DRiLL (VLSP 2025) được đặt ra như một bài toán truy hồi điều luật: cho một câu
hỏi pháp lý dạng ngôn ngữ tự nhiên (query) và một kho ngữ liệu lớn gồm các điều luật của
Việt Nam (mỗi điều/khoản/điểm tương ứng 1 provision), hệ thống cần trả về danh sách các
điều luật liên quan, có thể trả lời hoặc bao hàm câu hỏi đó.

Hình thức hoá: cho câu hỏi *q* và kho ngữ liệu *C = {a₁, a₂, ..., a_N}* gồm các điều
luật (ở đây là các **short-chunk** — xem mục 1b), hệ thống truy hồi cần học một hàm
chấm điểm *f(q, a)* sao cho các điều luật liên quan nhận điểm cao hơn.

Để tối đa hoá recall, ta dùng **hybrid retrieval**, kết hợp cả tín hiệu sparse (từ vựng)
và dense (ngữ nghĩa):

- **Sparse retrieval (BM25).** BM25 (Robertson & Zaragoza, 2009) tính điểm liên quan dựa
  trên khớp từ vựng chính xác giữa các từ trong câu hỏi và trong điều luật, có điều chỉnh
  theo tần suất từ (TF) và tần suất nghịch đảo văn bản (IDF):

  $$\mathrm{BM25}(q, a) = \sum_{t \in q} \mathrm{IDF}(t) \cdot
  \frac{f(t, a) \cdot (k_1 + 1)}{f(t, a) + k_1 \cdot \left(1 - b + b \cdot \frac{|a|}{\mathrm{avgdl}}\right)}$$

  trong đó *f(t, a)* là tần suất xuất hiện của từ *t* trong điều luật *a*, *|a|* là độ dài
  điều luật, *avgdl* là độ dài trung bình, *k₁, b* là các siêu tham số (dùng mặc định của
  `rank_bm25`).

- **Dense retrieval.** Ta dùng mô hình **Vietnamese_Embedding_v2 (AITeamVN)** để encode cả câu hỏi *q*
  và điều luật *a* thành các vector dense h_q, h_a ∈ ℝ^d. Điểm liên quan là cosine
  similarity giữa 2 vector (tương đương dot-product vì embedding đã được chuẩn hoá về độ
  dài 1):

  $$\mathrm{Dense}(q, a) = \cos(h_q, h_a) = \frac{h_q \cdot h_a}{\lVert h_q \rVert \, \lVert h_a \rVert}$$

- **Kết hợp 2 tín hiệu.** Vì thang điểm của BM25 và cosine similarity rất khác nhau, ta
  min-max normalize từng loại điểm về [0, 1] trên cùng tập ứng viên rồi cộng có trọng số:

  $$\mathrm{Score}(q, a) = \lambda \cdot \mathrm{BM25_{norm}}(q, a) + (1 - \lambda) \cdot \mathrm{Dense_{norm}}(q, a)$$

  với λ ∈ [0, 1] là siêu tham số được **tune trên tập validation (dev)** — xem Cell 8c.
  Chiến lược đơn giản nhưng hiệu quả này cân bằng giữa độ chính xác của sparse retrieval
  và khả năng khái quát hoá ngữ nghĩa của dense retrieval. Hybrid retriever trả về top-k
  điều luật ứng viên cho reranker ở stage-2 (giữ nguyên BAAI/bge-reranker-v2-m3, không đổi).

> **Lưu ý về thứ tự chạy cell:** phần định nghĩa hàm `bm25_retrieve` / `dense_retrieve` /
> `hybrid_retrieve` nằm ở đây (mục 2), nhưng việc **load model dense lên GPU và encode**
> (và bước tune λ đi kèm) được đặt **sau** mục 4 (mining negative) và mục 5 (fine-tune
> reranker) — ngay trước mục 6 (Pipeline) — để tránh xung đột `fork()` (multiprocessing,
> dùng khi mining) với CUDA context (xem lưu ý chi tiết ở cell encode dense).


In [10]:
# Cell 6 — Tokenize tiếng Việt
# CHỈ dùng underthesea cho MỌI nơi tokenize trong notebook (đã bỏ fast tokenizer bằng regex).
# vi_tokenize_short (dùng cho BM25 short-chunk, Cell 7/8) và vi_tokenize (dùng cho phần còn lại:
# chọn short-chunk đại diện, chunk_token_cache, ...) giờ trỏ về CÙNG 1 hàm underthesea bên dưới.
# Giữ nguyên 2 tên biến để không phải sửa các cell downstream đang gọi vi_tokenize_short/vi_tokenize.
from underthesea import word_tokenize as _underthesea_tokenize


def vi_tokenize(text: str):
    return _underthesea_tokenize((text or "").lower())


# ---- Mọi nơi (BM25 short-chunk Stage-1, short-chunk mining, select_best_chunk, chunk_token_cache,
#      ...): LUÔN dùng underthesea, không còn fast/regex tokenizer nữa ----
vi_tokenize_short = vi_tokenize

import bm25s  # thay cho rank_bm25 -> BM25Okapi: bm25s dùng backend numpy/scipy tối ưu (numba/JIT nếu
                 # có sẵn), tính điểm nhanh hơn đáng kể trên corpus lớn, giảm chi phí tính toán ở Stage-1


In [ ]:
# Cell 6a — Tokenize sẵn TẤT CẢ câu hỏi (train_samples), cache ra pickle.
# ĐÃ SỬA: chỉ còn 1 tokenizer (underthesea) cho toàn bộ notebook, nên chỉ cần tokenize câu hỏi
# đúng 1 LẦN duy nhất, lưu ra đĩa theo qid -> list[token]. Cell 11/11b sau đó chỉ cần TRA CACHE,
# không tokenize lại nữa. `_question_token_cache_fast` được GIỮ LẠI làm alias trỏ tới CÙNG dict với
# `_question_token_cache` (không tokenize 2 lần nữa) để không phải sửa tên biến ở các cell downstream
# (mining, Cell 11/11b) vẫn đang tham chiếu `_question_token_cache_fast`.
import pickle

_QUESTION_TOKEN_CACHE_PATH_ACCURATE = "cache/question_token_cache_underthesea.pkl"


def _load_or_init_token_cache(path, label):
    if os.path.exists(path):
        with open(path, "rb") as f:
            cache = pickle.load(f)
        print(f"Đã load question token cache ({label}) từ {path}: {len(cache)} câu hỏi.")
    else:
        cache = {}
        print(f"Chưa có question token cache ({label}) tại {path} -> sẽ tokenize mới và lưu lại.")
    return cache


_question_token_cache = _load_or_init_token_cache(_QUESTION_TOKEN_CACHE_PATH_ACCURATE, "underthesea")
# Alias: các cell downstream (Cell 11/11b) vẫn dùng tên `_question_token_cache_fast` -> trỏ về
# CÙNG dict underthesea ở trên, không cần cache/tokenize riêng nữa.
_question_token_cache_fast = _question_token_cache

_qid2question = {s["qid"]: s["question"] for s in train_samples}  # gồm cả fit_samples + dev_samples

_tokenize_fn_for_pool = None

def _tokenize_questions_batch(qid_batch):
    return [(qid, _tokenize_fn_for_pool(_qid2question[qid])) for qid in qid_batch]


def _set_tokenizer_fn(fn):
    global _tokenize_fn_for_pool
    _tokenize_fn_for_pool = fn


def _tokenize_missing_questions(cache, tokenize_fn, cache_path, label, allow_parallel):
    """Tokenize các câu hỏi còn thiếu trong `cache` bằng `tokenize_fn`, lưu lại vào `cache_path`.
    allow_parallel=True dùng multiprocessing.Pool (an toàn ở đây vì CUDA/model lớn chưa init) —
    bật multiprocessing để tăng tốc underthesea (tokenizer duy nhất còn lại, chậm hơn regex)."""
    missing_qids = [qid for qid in _qid2question if qid not in cache]
    if not missing_qids:
        print(f"Tất cả câu hỏi (fit + dev) đã có sẵn trong cache ({label}), không cần tokenize lại.")
        return

    n_workers = min(get_n_workers(), len(missing_qids))  # tránh tạo worker thừa nếu số câu hỏi ít
    if not allow_parallel or n_workers == 1 or len(missing_qids) < 32:
        for qid in tqdm(missing_qids, desc=f"Tokenize câu hỏi ({label})"):
            cache[qid] = tokenize_fn(_qid2question[qid])
    else:
        # An toàn để Pool ở đây (chưa có CUDA/model lớn nào được load trong process này).
        from multiprocessing import Pool

        batch_size = min(64, max(8, len(missing_qids) // (n_workers * 4)))
        qid_batches = [missing_qids[i:i + batch_size] for i in range(0, len(missing_qids), batch_size)]

        ctx = mp.get_context("spawn")
        with ctx.Pool(n_workers, initializer=_set_tokenizer_fn, initargs=(tokenize_fn,)) as pool:
            for batch_result in tqdm(pool.imap_unordered(_tokenize_questions_batch, qid_batches),
                                     total=len(qid_batches), desc=f"Tokenize câu hỏi ({label}, song song)"):
                for qid, toks in batch_result:
                    cache[qid] = toks

    with open(cache_path, "wb") as f:
        pickle.dump(cache, f)
    print(f"Đã tokenize {len(missing_qids)} câu hỏi mới ({label}), lưu cache tại {cache_path} "
          f"(tổng {len(cache)} câu hỏi).")


# Underthesea: tokenizer duy nhất còn lại -> tận dụng multiprocessing để tăng tốc.
_tokenize_missing_questions(_question_token_cache, vi_tokenize,
                             _QUESTION_TOKEN_CACHE_PATH_ACCURATE, "underthesea", allow_parallel=True)


Đã load question token cache (underthesea) từ /mnt/mmlab2024nas/trantran/question_token_cache_underthesea.pkl: 7000 câu hỏi.
Tất cả câu hỏi (fit + dev) đã có sẵn trong cache (underthesea), không cần tokenize lại.


In [12]:
# Cell 6b — Chọn SHORT-chunk liên quan nhất ở Stage-2 bằng CÙNG SCORING với Stage-1.
#
# QUY TẮC PIPELINE MỚI:
# - Stage-1 đã chọn TOP_K_STAGE1 document ở cấp document.
# - Với MỖI document ứng viên, Stage-2 chỉ chọn tối đa MAX_CHUNKS_PER_DOC_FOR_RERANK
#   short-chunk có điểm Stage-1 cao nhất.
# - SHORT-CHUNK ĐƯỢC ĐƯA THẲNG VÀO RERANKER. KHÔNG expand, KHÔNG ghép chunk lân cận,
#   KHÔNG tạo long-chunk.
#
# `chunk_scores` là score của TOÀN BỘ short-chunk cho câu hỏi hiện tại, dùng cùng scoring
# với Stage-1 (HYBRID mặc định). Hàm này chỉ làm nhiệm vụ chọn top-N short-chunk trong
# MỘT document; text trả về chính là `short_chunk_text_of` của chunk đó.

def select_relevant_chunks(doc_id, chunk_scores, top_n=None):
    """Trả về tối đa top_n SHORT-chunk của `doc_id` để đưa thẳng vào Stage-2 reranker.

    Không có bất kỳ bước expand/ghép nào ở đây. `chunk_scores` là score của toàn bộ
    short-chunk và được tính bằng cùng scoring function với Stage-1.

    Return: list[(chunk_id, short_chunk_text)].
    """
    if top_n is None:
        top_n = MAX_CHUNKS_PER_DOC_FOR_RERANK
    idxs = _doc_to_short_chunk_idx.get(doc_id)
    if not idxs or top_n <= 0:
        return []

    ranked_global_idx = sorted(idxs, key=lambda i: -chunk_scores[i])[:top_n]
    out = []
    for global_idx in ranked_global_idx:
        cid = short_chunk_ids[global_idx]
        text = short_chunk_text_of[cid]
        if text and text.strip():
            out.append((cid, text))
    return out


In [ ]:
# Cell 7 — Short-chunk toàn bộ corpus (Stage-1 / retrieval) rồi build BM25 index TRÊN CÁC SHORT-CHUNK
# (song song hoá + cache ra đĩa), thay vì trên toàn văn bản như trước. Đây là thay đổi kiến trúc chính
# để khớp đúng paper ViDRILL: Stage-1 hoạt động ở cấp chunk ngắn (450 ký tự, clause-based), không phải
# cấp document.
import pickle
import multiprocessing as mp
from multiprocessing import Pool, cpu_count
import gc, ctypes

# ---- Bước 1: short-chunk toàn bộ corpus (mỗi văn bản -> nhiều short-chunk độc lập) ----
# ĐÃ SỬA: chunk_document_short_from_segments (Cell 4b) giờ trả về (chunk_id, text, path, segment_type),
# trong đó text ĐÃ GẮN SẴN path ở đầu (nếu path hợp lệ, gắn NGAY SAU KHI cắt content) -> text này vừa
# là short-chunk index ở Stage-1 (BM25/dense) VỪA là short-chunk dùng thẳng ở Stage-2 (Cell 6b, không
# cần gắn path lại). Đổi hậu tố tên file cache ("_pathinchunk") để không lỡ load nhầm cache ĐỊNH DẠNG
# CŨ (chunk_text CHƯA gắn path, hoặc thiếu segment_type) — nếu có cache cũ, cell này sẽ tự short-chunk
# lại.
_SHORT_CHUNK_CORPUS_PATH = os.path.join(
    "cache",
    f"short_chunk_corpus_typed_{SHORT_CHUNK_CHARS_STRUCTURED}_{SHORT_CHUNK_CHARS_UNSTRUCTURED}_"
    f"{SHORT_CHUNK_CHARS_ROOT_PREAMBLE}_7.pkl",
)  # ĐÃ SỬA: đổi tên cache theo 3 ngân sách RIÊNG từng segment_type (Cell 2) để không lỡ load
   # nhầm cache cũ (tạo bằng 1 SHORT_CHUNK_CHARS chung) — nếu chưa có cache mới, cell này sẽ tự
   # short-chunk lại toàn bộ corpus theo logic mới.

if os.path.exists(_SHORT_CHUNK_CORPUS_PATH):
    with open(_SHORT_CHUNK_CORPUS_PATH, "rb") as f:
        short_chunk_corpus = pickle.load(f)  # list[(chunk_id, doc_id, chunk_text, path, segment_type)]
    print(f"Đã load short-chunk corpus từ {_SHORT_CHUNK_CORPUS_PATH}: {len(short_chunk_corpus)} chunk.")
else:
    short_chunk_corpus = []
    for _doc_id in tqdm(doc_ids_all, desc="Short-chunking toàn bộ corpus (Stage-1)"):
        # ĐÃ SỬA: dùng passage_segments (processed-context, Cell 4) thay vì "passage" thô
        for _cid, _ctext, _path, _segtype in chunk_document_short_from_segments(_doc_id, corpus[_doc_id]["passage_segments"]):
            short_chunk_corpus.append((_cid, _doc_id, _ctext, _path, _segtype))
    with open(_SHORT_CHUNK_CORPUS_PATH, "wb") as f:
        pickle.dump(short_chunk_corpus, f)
    print(f"Đã short-chunk {len(doc_ids_all)} văn bản -> {len(short_chunk_corpus)} chunk, "
          f"lưu tại {_SHORT_CHUNK_CORPUS_PATH}")

short_chunk_ids = [c[0] for c in short_chunk_corpus]
short_chunk_doc_of = {c[0]: c[1] for c in short_chunk_corpus}   # chunk_id -> doc_id
short_chunk_text_of = {c[0]: c[2] for c in short_chunk_corpus}  # chunk_id -> text ĐÃ GẮN path ở đầu
                                                                  # (nếu hợp lệ) — dùng cho CẢ Stage-1
                                                                  # (BM25/dense index) LẪN Stage-2
                                                                  # (short-chunk = root_preamble +
                                                                  # short-chunk này, xem Cell 6b)
short_chunk_path_of = {c[0]: c[3] for c in short_chunk_corpus}  # chunk_id -> path gốc ("" nếu không
                                                                  # có/bất thường) -- CHỈ để tham
                                                                  # khảo/debug, KHÔNG còn dùng lại ở
                                                                  # Stage-2 (path đã nằm sẵn trong
                                                                  # short_chunk_text_of rồi)
short_chunk_segtype_of = {c[0]: c[4] for c in short_chunk_corpus}  # chunk_id -> segment_type gốc
                                                                     # ("structured"/"root_preamble"/
                                                                     # "unstructured_whole"/"") --
                                                                     # dùng để nhận biết short-chunk
                                                                     # thuộc root_preamble (Cell 6b)
del short_chunk_corpus
gc.collect()
# ---- Bước 2: tokenize từng short-chunk (song song + cache ra đĩa) ----
# ĐÃ SỬA: chỉ còn underthesea (đã bỏ fast/regex tokenizer) -> đổi hậu tố tên file cache thành
# "underthesea" để không lỡ load nhầm cache tokenize kiểu regex cũ. ĐÃ SỬA THÊM (MỚI): nội dung
# short_chunk_text_of (short_chunk_corpus[idx][2]) giờ đã gắn path ở đầu (nếu hợp lệ, xem Cell 4b) ->
# đổi thêm hậu tố "_pathinchunk" để không lỡ load nhầm cache tokenize CŨ (tokenize trên text CHƯA
# gắn path), buộc tokenize lại cho khớp text mới.
TOKENIZED_CACHE_PATH = os.path.join(
    "cache",
    f"bm25_tokenized_short_chunks_typed_{SHORT_CHUNK_CHARS_STRUCTURED}_{SHORT_CHUNK_CHARS_UNSTRUCTURED}_"
    f"{SHORT_CHUNK_CHARS_ROOT_PREAMBLE}_underthesea_7.pkl",
)  # ĐÃ SỬA: đổi tên cache tương ứng với _SHORT_CHUNK_CORPUS_PATH ở trên, tránh tokenize nhầm
   # cache cũ (short-chunk text đã đổi theo ngân sách mới -> phải tokenize lại).

def _tokenize_one_chunk(idx):
    # BM25 trên short-chunk dùng vi_tokenize_short, giờ LUÔN là underthesea (xem Cell 6).
    return vi_tokenize_short(short_chunk_corpus[idx][2])

if os.path.exists(TOKENIZED_CACHE_PATH):
    print("Đã tìm thấy cache, load lại thay vì tokenize lại:", TOKENIZED_CACHE_PATH)
    with open(TOKENIZED_CACHE_PATH, "rb") as f:
        bm25_tokenized_corpus = pickle.load(f)
else:
    n_workers = get_n_workers()  # giới hạn bởi MAX_WORKERS (Cell 2) -> không chiếm hết CPU server
    n_chunks = len(short_chunk_corpus)
    print(f"Tokenizing {n_chunks} short-chunk bằng {n_workers} tiến trình song song...")
    if n_workers > 1:
        if __name__ == "__main__":          # <-- thêm guard ở đây
            ctx = mp.get_context("spawn")
            with ctx.Pool(n_workers, initializer=_lower_priority) as pool:
                bm25_tokenized_corpus = list(
                    tqdm(pool.imap(_tokenize_one_chunk, range(n_chunks), chunksize=200),
                         total=n_chunks, desc="Tokenizing short-chunk corpus")
                )
    else:
        bm25_tokenized_corpus = [_tokenize_one_chunk(i) for i in tqdm(range(n_chunks), desc="Tokenizing short-chunk corpus")]

    with open(TOKENIZED_CACHE_PATH, "wb") as f:
        pickle.dump(bm25_tokenized_corpus, f)
    print("Đã lưu cache tokenize tại:", TOKENIZED_CACHE_PATH)

# ---- Build BM25 index bằng bm25s (thay cho rank_bm25.BM25Okapi) ----
# bm25s nhận trực tiếp corpus đã tokenize sẵn (list[list[str]]) giống hệt rank_bm25, nên không cần
# tokenize lại. Đây là thay đổi duy nhất cần thiết: bm25s dùng cấu trúc dữ liệu sparse (scipy) +
# tính điểm vector hoá, giúp build index và tính get_scores() nhanh hơn nhiều so với rank_bm25
# (pure Python), giảm đáng kể chi phí tính toán Stage-1 khi corpus lớn.
bm25 = bm25s.BM25()

if 'model' in dir():
    del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

bm25.index(bm25_tokenized_corpus)
print("BM25 (bm25s) index sẵn sàng với", len(short_chunk_ids), "short-chunk trên", len(doc_ids_all), "văn bản")

# ---- Dọn RAM: bm25_tokenized_corpus và short_chunk_corpus chỉ dùng để build index/dict ở trên,
# không còn được tham chiếu ở bất kỳ cell nào phía sau (bm25 đã tự lưu cấu trúc riêng sau .index(),
# và short_chunk_text_of/short_chunk_doc_of/... đã tách ra từ short_chunk_corpus rồi).
del bm25_tokenized_corpus
gc.collect()
ctypes.CDLL("libc.so.6").malloc_trim(0)

print("Đã giải phóng bm25_tokenized_corpus và short_chunk_corpus khỏi RAM.")

Đã load short-chunk corpus từ /mnt/mmlab2024nas/trantran/short_chunk_corpus_typed_1000_2000_2000_7.pkl: 442255 chunk.
Đã tìm thấy cache, load lại thay vì tokenize lại: /mnt/mmlab2024nas/trantran/bm25_tokenized_short_chunks_typed_1000_2000_2000_underthesea_7.pkl


Building index from tokens


BM25 (bm25s) index sẵn sàng với 442255 short-chunk trên 8511 văn bản
Đã giải phóng bm25_tokenized_corpus và short_chunk_corpus khỏi RAM.


In [14]:
# Cell 8 — Hàm retrieve top-k bằng BM25, hoạt động ở cấp SHORT-CHUNK rồi gộp điểm về cấp document.
# ĐÃ SỬA (bỏ bước cắt top-K_STAGE1_CHUNKS thô ở cấp CHUNK trước khi gộp): pipeline CŨ là
#   toàn bộ chunk -> cắt top-300 chunk điểm cao nhất -> gộp max-per-doc TRÊN 300 chunk đó
# có nhược điểm: 1 document có NHIỀU chunk liên quan vừa phải (không chunk nào đủ mạnh để lọt
# top-300 toàn corpus) sẽ bị loại hoàn toàn, dù xét riêng document đó nó xứng đáng có mặt trong
# top-k. Pipeline MỚI:
#   toàn bộ chunk -> gộp max-per-doc TRÊN TOÀN BỘ chunk (không cắt trước) -> sort document theo
#   điểm đã gộp -> cắt top-k document cuối cùng
# tức là bước "cắt top-k" giờ chỉ xảy ra ĐÚNG 1 LẦN, ở CẤP DOCUMENT (sau khi đã gộp), thay vì cắt
# ở cấp CHUNK trước rồi mới gộp. Giao diện hàm (trả về list[(doc_id, score)]) giữ nguyên như
# trước để không phải sửa các cell downstream (mining, eval, pipeline 2 tầng).

def bm25_scores_all(question: str = None, q_tokens=None):
    """Trả về np.array điểm BM25 của TẤT CẢ short-chunk (dùng chung cho bm25_retrieve
    và hybrid_scores_all để không phải tính 2 lần).
    q_tokens (nếu truyền vào) PHẢI được tokenize bằng vi_tokenize_short (underthesea, xem Cell 6),
    để khớp đúng tokenizer đã dùng khi build BM25 index trên short-chunk ở Cell 7.
    bm25.get_scores(...) ở đây là API của bm25s (không phải rank_bm25 nữa, xem Cell 6/7) nhưng
    giữ nguyên chữ ký (nhận list[str] token đơn 1 câu hỏi, trả về np.array điểm cho toàn bộ
    corpus) nên không cần sửa gì thêm ở đây."""
    if q_tokens is None:
        q_tokens = vi_tokenize_short(question)
    return bm25.get_scores(q_tokens)


# ---- Precompute (1 LẦN) ánh xạ short-chunk -> document, dùng để gộp max-per-doc bằng numpy
# ---- (np.maximum.at) trên TOÀN BỘ chunk mỗi câu hỏi, thay vì cắt top-K chunk thô trước rồi
# ---- mới gộp (cách cũ). Vector hoá để việc xét TOÀN BỘ chunk (thay vì chỉ 300 chunk) mỗi
# ---- query vẫn đủ nhanh, không lặp bằng Python thuần trên hàng trăm nghìn chunk.
_doc_id_list_for_agg = sorted(set(short_chunk_doc_of.values()))
_doc_id_to_idx_for_agg = {d: i for i, d in enumerate(_doc_id_list_for_agg)}
_chunk_doc_idx_for_agg = np.array(
    [_doc_id_to_idx_for_agg[short_chunk_doc_of[cid]] for cid in short_chunk_ids], dtype=np.int64
)
_n_docs_for_agg = len(_doc_id_list_for_agg)


def _aggregate_scores_to_doc(scores: np.ndarray, top_k: int):
    """np.array điểm của TOÀN BỘ short-chunk (cùng thứ tự với short_chunk_ids, KHÔNG cắt trước)
    -> top_k document_id: list[(doc_id, score)], score = điểm CHUNK CAO NHẤT/document, gộp trên
    TOÀN BỘ chunk của document đó (không giới hạn trong 1 tập con top-K chunk thô như cách cũ).
    Dùng np.maximum.at để gộp max-per-doc vector hoá (nhanh dù xét toàn bộ chunk mỗi query)."""
    best_score_per_doc = np.full(_n_docs_for_agg, -np.inf, dtype=np.float64)
    np.maximum.at(best_score_per_doc, _chunk_doc_idx_for_agg, scores)
    top_doc_idx = np.argsort(best_score_per_doc)[::-1][:top_k]
    return [(_doc_id_list_for_agg[i], float(best_score_per_doc[i])) for i in top_doc_idx]


def bm25_retrieve(question: str, top_k: int = TOP_K_STAGE1, q_tokens=None):
    """Trả về top_k document_id: list[(doc_id, score)], score = điểm short-chunk cao nhất của
    document đó trên TOÀN BỘ short-chunk corpus (đã bỏ bước cắt top-K_STAGE1_CHUNKS thô ở cấp
    chunk trước khi gộp, xem ghi chú đầu cell).
    q_tokens: nếu đã tokenize sẵn câu hỏi (vd. lấy từ _question_token_cache ở Cell 6a khi mining),
    truyền vào đây để tránh tokenize lại bằng underthesea."""
    scores = bm25_scores_all(question, q_tokens=q_tokens)
    return _aggregate_scores_to_doc(scores, top_k)


In [15]:
# Cell 8b — Hàm dense_retrieve (song song với bm25_retrieve, cùng cấp SHORT-CHUNK) và
# hybrid_retrieve (BM25 + Dense theo công thức mục 2.2.1: min-max normalize rồi weighted sum).
# ĐÃ SỬA (đồng bộ với Cell 8): bỏ bước cắt top-K_STAGE1_CHUNKS thô ở cấp CHUNK trước khi gộp về
# document -- gộp max-per-doc trên TOÀN BỘ chunk rồi mới sort/cắt top-k ở CẤP DOCUMENT (dùng chung
# _aggregate_scores_to_doc, Cell 8).

def _minmax_normalize(scores: np.ndarray) -> np.ndarray:
    lo, hi = float(scores.min()), float(scores.max())
    if hi - lo < 1e-12:
        return np.zeros_like(scores)
    return (scores - lo) / (hi - lo)


def dense_scores_all(question: str = None, q_emb: np.ndarray = None):
    """Trả về np.array điểm cosine similarity (dot product, embeddings đã normalize) của
    câu hỏi với TẤT CẢ short-chunk. Dùng chung cho dense_retrieve và hybrid_scores_all."""
    if q_emb is None:
        # ĐÃ SỬA: Vietnamese_Embedding_v2 (AITeamVN, nền BGE-M3) không định nghĩa sẵn prompt "query"
        # như Qwen3-Embedding -> bỏ prompt_name="query" (nếu giữ lại sẽ lỗi vì model không có
        # prompts config này). Encode câu hỏi trực tiếp, không cần prefix đặc biệt.
        q_emb = embed_model.encode(
            [question], convert_to_numpy=True, normalize_embeddings=True
        )[0]
    return chunk_embeddings @ q_emb


def dense_retrieve(question: str, top_k: int = TOP_K_STAGE1, q_emb=None):
    """Dense-only stage-1 (dùng để ablation / so sánh với BM25 và hybrid). Gộp max-per-doc trên
    TOÀN BỘ short-chunk corpus (không cắt top-K chunk thô trước, xem ghi chú đầu cell)."""
    scores = dense_scores_all(question, q_emb=q_emb)
    return _aggregate_scores_to_doc(scores, top_k)


def hybrid_scores_all(question: str = None, lam: float = None, q_tokens=None, q_emb=None):
    """Trả về hybrid score của TOÀN BỘ short-chunk corpus.
    Dùng CHUNG cho Stage-1 và chọn representative chunk trong Stage-2."""
    if lam is None:
        lam = LAMBDA_HYBRID
    bm25_raw = bm25_scores_all(question, q_tokens=q_tokens)
    dense_raw = dense_scores_all(question, q_emb=q_emb)
    return lam * _minmax_normalize(bm25_raw) + (1 - lam) * _minmax_normalize(dense_raw)


def hybrid_retrieve(question: str, top_k: int = TOP_K_STAGE1, lam: float = None, q_tokens=None, q_emb=None):
    """Stage-1 HYBRID mặc định: kết hợp BM25 (sparse) + Vietnamese_Embedding_v2 (AITeamVN) (dense).
    Score(q, a) = lambda * BM25_norm(q, a) + (1 - lambda) * Dense_norm(q, a), tính trên TOÀN BỘ
    short-chunk corpus rồi min-max normalize từng loại điểm về [0, 1] trước khi cộng (BM25 và
    cosine similarity có thang điểm khác nhau nên không thể cộng trực tiếp). Sau đó gộp max-per-doc
    trên TOÀN BỘ chunk (không cắt top-K chunk thô trước) rồi mới sort/cắt top_k ở CẤP DOCUMENT.
    Giao diện hàm giống hệt bm25_retrieve/dense_retrieve (list[(doc_id, score)]) nên có thể truyền
    thẳng vào two_stage_retrieve(stage1_fn=hybrid_retrieve) không cần sửa gì ở downstream."""
    combined = hybrid_scores_all(question, lam=lam, q_tokens=q_tokens, q_emb=q_emb)
    return _aggregate_scores_to_doc(combined, top_k)


## 3. Hàm đánh giá Recall / Precision (đúng theo công thức của đề bài)

In [16]:
# Cell 9 — Recall@k / Precision@k theo đúng định nghĩa trong đề bài
# (trung bình theo từng câu hỏi; nếu hệ thống không trả về gì thì precision câu đó = 0)
def recall_precision(pred_ids, gold_ids):
    gold = set(gold_ids)
    pred = set(pred_ids)
    if len(pred) == 0:
        precision = 0.0
    else:
        precision = len(gold & pred) / len(pred)
    recall = len(gold & pred) / len(gold) if len(gold) > 0 else 0.0
    return recall, precision

def evaluate(retrieve_fn, samples, top_k=TOP_K_FINAL, desc="eval"):
    # retrieve_fn(question) -> list[doc_id] (đã sắp xếp theo độ liên quan giảm dần)
    recalls, precisions = [], []
    for s in tqdm(samples, desc=desc):
        pred = retrieve_fn(s["question"])[:top_k]   # luôn cắt đúng top_k <= 5
        r, p = recall_precision(pred, s["answers"])
        recalls.append(r); precisions.append(p)
    return {
        "recall": float(np.mean(recalls)),
        "precision": float(np.mean(precisions)),
        "n": len(samples),
    }

In [ ]:
# LƯU Ý (ĐÃ SỬA): mining (Cell 10c) giờ KHÔNG còn dùng multiprocessing/fork nữa (chỉ còn numpy +
# dict lookup), nên ràng buộc "phải load CUDA sau khi fork mining" không còn bắt buộc như trước.
# Nhờ vậy Cell 8c (tune lambda) đã được CHUYỂN LÊN ngay sau cell này (thay vì đặt sau toàn bộ
# mining + fine-tune reranker như bản cũ) — đúng với mặt lý thuyết (mục 2: hybrid retrieval là
# Stage-1), và quan trọng hơn: đảm bảo LAMBDA_HYBRID đã được tune xong TRƯỚC khi training loop
# (Cell 14) dùng nó để eval dev mỗi epoch / chọn best checkpoint (xem note đầu Cell 8c).
# Cell 7b — Encode toàn bộ short-chunk corpus bằng Vietnamese_Embedding_v2 (AITeamVN) (dense, cho stage-1)
# Cùng cấp chunk với BM25 (Cell 7) để 2 tín hiệu có thể kết hợp (hybrid) ở mức chunk, sau đó
# mới gộp điểm về cấp document giống hệt cách BM25 đang làm.
from sentence_transformers import SentenceTransformer

torch.cuda.empty_cache()  # dọn VRAM còn sót lại từ lần chạy OOM trước (nếu có)

embed_model = SentenceTransformer(
    EMBED_MODEL_NAME,
    device=DEVICE,
    model_kwargs={"torch_dtype": torch.float16},  # giảm ~50% VRAM, gần như không đổi chất lượng
)
# ĐÃ SỬA: theo đúng khuyến nghị chính thức của AITeamVN/Vietnamese_Embedding_v2 (max sequence length
# huấn luyện = 2048 token) — short-chunk (<= SHORT_CHUNK_CHARS=1000 ký tự) vẫn nằm rất sâu trong
# ngân sách này nên không đổi hành vi, chỉ đảm bảo đúng cấu hình model.
embed_model.max_seq_length = 2048

_CHUNK_EMB_CACHE_PATH = os.path.join(
    "cache", f"vn_embed_v2_short_chunks_{SHORT_CHUNK_CHARS}_7.npy"
)

if os.path.exists(_CHUNK_EMB_CACHE_PATH):
    chunk_embeddings = np.load(_CHUNK_EMB_CACHE_PATH)
    print(f"Đã load dense embeddings từ {_CHUNK_EMB_CACHE_PATH}: shape={chunk_embeddings.shape}")
else:
    short_chunk_texts = [short_chunk_text_of[cid] for cid in short_chunk_ids]
    chunk_embeddings = embed_model.encode(
        short_chunk_texts,
        batch_size=EMBED_BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype(np.float16)
    np.save(_CHUNK_EMB_CACHE_PATH, chunk_embeddings)
    print(f"Đã encode {len(short_chunk_ids)} short-chunk -> embeddings shape={chunk_embeddings.shape}, "
          f"lưu tại {_CHUNK_EMB_CACHE_PATH}")


Loading weights: 100%|██████████| 391/391 [00:17<00:00, 22.76it/s]
/mnt/mmlab2024nas/trantran/.venv/lib/python3.10/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


Đã load dense embeddings từ /mnt/mmlab2024nas/trantran/vn_embed_v2_short_chunks_2000_7.npy: shape=(442255, 1024)


In [18]:
# Cell 8c — Tune lambda (trọng số BM25 trong hybrid) trên tập VALIDATION (dev)
# ĐÃ CHUYỂN LÊN ĐÚNG VỊ TRÍ (trước đây cell này nằm SAU toàn bộ mining + fine-tune reranker,
# tức SAU Cell 15 — lý do cũ là lo ngại fork+CUDA khi mining còn dùng multiprocessing, xem note
# đã lỗi thời ở cell encode dense phía trên). Mining giờ không còn multiprocessing/fork nữa nên
# ràng buộc đó không còn, và quan trọng hơn: ĐỂ CELL NÀY SAU Cell 14 (training loop) là 1 BUG
# THẬT — vòng lặp training đánh giá dev sau mỗi epoch bằng hybrid_retrieve(), hàm này dùng
# LAMBDA_HYBRID mặc định (0.4, Cell 2) NẾU lambda chưa được tune, vì Cell 8c (nơi ghi đè
# LAMBDA_HYBRID bằng giá trị tune xong) khi đó CHƯA chạy -> checkpoint 'tốt nhất' (best_epoch,
# BEST_DIR) bị chọn dựa trên Stage-1 CHƯA tune, có thể không thật sự là checkpoint tốt nhất dưới
# Stage-1 đã tune (dùng ở Cell 18 sau đó). Đặt Cell 8c NGAY SAU khi build xong Stage-1 (BM25 +
# Dense, Cell 7b) và TRƯỚC mining/fine-tune reranker đảm bảo LAMBDA_HYBRID đã đúng giá trị tune
# trước khi bất kỳ cell nào (kể cả training loop) dùng tới.
# Grid search theo Recall@TOP_K_STAGE1 của stage-1 hybrid_retrieve trên dev_samples (proxy tốt cho
# chất lượng ứng viên đưa vào reranker — recall càng cao thì reranker càng có cơ hội tìm ra đúng
# điều luật liên quan trong top_k_stage1 ứng viên).
#
# TỐI ƯU: bm25_raw/dense_raw của 1 câu hỏi KHÔNG đổi theo lambda (chỉ cách kết hợp 2 điểm đổi) —
# nhưng bản gốc gọi hybrid_retrieve(question, lam=...) cho MỖI lambda trong LAMBDA_GRID, khiến
# bm25_scores_all()/dense_scores_all() (encode câu hỏi qua GPU + tính điểm toàn corpus) bị tính lại
# 11 LẦN/câu hỏi một cách dư thừa. Ở đây ta PRECOMPUTE bm25_norm/dense_norm 1 LẦN/câu hỏi (min-max
# normalize sẵn), cache lại, rồi mỗi lambda chỉ cần cộng có trọng số (rẻ, không đụng GPU/BM25 nữa).
#
# ĐÃ SỬA (đồng bộ với Cell 8/8b): KHÔNG còn cắt top-K_STAGE1_CHUNKS chunk thô trước khi gộp về
# document nữa. stage1_recall_at_lambda giờ gộp max-per-doc trên TOÀN BỘ short-chunk (dùng chung
# _aggregate_scores_to_doc, Cell 8) rồi mới sort/cắt top_k ở CẤP DOCUMENT — ĐÚNG KHỚP với cách
# hybrid_retrieve (Cell 8b) hoạt động lúc inference thật, nên kết quả tune lambda ở đây phản ánh
# đúng recall thực tế của Stage-1 sau khi sửa.

def _precompute_stage1_norm_scores(samples):
    """Tính trước bm25_norm/dense_norm (đã min-max normalize) cho từng câu hỏi trong `samples`, dùng
    chung cho toàn bộ LAMBDA_GRID bên dưới thay vì tính lại mỗi lambda. Trả về list[(bm25_norm,
    dense_norm)] cùng thứ tự với `samples`."""
    cached = []
    for s in tqdm(samples, desc="Precompute BM25+Dense scores (dùng chung cho mọi lambda)"):
        bm25_raw = bm25_scores_all(s["question"])
        dense_raw = dense_scores_all(s["question"])
        cached.append((_minmax_normalize(bm25_raw), _minmax_normalize(dense_raw)))
    return cached


def stage1_recall_at_lambda(lam, samples, cached_norm_scores, top_k=TOP_K_STAGE1):
    """Chỉ cộng có trọng số 2 mảng điểm ĐÃ NORMALIZE SẴN (cached_norm_scores, TOÀN BỘ short-chunk)
    rồi gộp max-per-doc trên TOÀN BỘ chunk (_aggregate_scores_to_doc, Cell 8) để lấy top_k document
    — không gọi lại bm25_scores_all/dense_scores_all nữa (xem _precompute_stage1_norm_scores), và
    không cắt top-K chunk thô trước khi gộp (khớp đúng hybrid_retrieve, Cell 8b)."""
    recalls = []
    for s, (bm25_norm, dense_norm) in zip(samples, cached_norm_scores):
        combined = lam * bm25_norm + (1 - lam) * dense_norm
        pred = [d for d, _ in _aggregate_scores_to_doc(combined, top_k)]
        r, _ = recall_precision(pred, s["answers"])
        recalls.append(r)
    return float(np.mean(recalls))


# _dev_stage1_norm_scores = _precompute_stage1_norm_scores(dev_samples)

# LAMBDA_GRID = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
# lambda_search_results = {
#     lam: stage1_recall_at_lambda(lam, dev_samples, _dev_stage1_norm_scores)
#     for lam in tqdm(LAMBDA_GRID, desc="Tuning lambda")
# }

# for lam, r in sorted(lambda_search_results.items()):
#     print(f"  lambda={lam:.1f} -> stage-1 recall@{TOP_K_STAGE1}={r:.4f}"
#           + ("  (BM25-only)" if lam == 1.0 else "  (Dense-only)" if lam == 0.0 else ""))

# LAMBDA_HYBRID = max(lambda_search_results, key=lambda_search_results.get)
# MINING_LAMBDA = LAMBDA_HYBRID  # dùng cho mining (Cell 10c) và fine-tune reranker (Cell 14)
# print(f"\n-> Chọn lambda={LAMBDA_HYBRID:.1f} "
#       f"(stage-1 recall@{TOP_K_STAGE1}={lambda_search_results[LAMBDA_HYBRID]:.4f}) cho hybrid_retrieve.")


## 4. Mining positive/negative samples bằng HYBRID để fine-tune cross-encoder (5:1)

Với mỗi câu hỏi: positive và hard-negative đều được chọn từ **cùng một HYBRID scoring** trên
short-chunk, với `MINING_LAMBDA=0.2`: `0.2*BM25_norm + 0.8*Dense_norm`. Như vậy bước mining
không còn dùng BM25-only cho positive hoặc negative. Sau khi chọn short-chunk đại diện, cả positive
và negative đều được giữ nguyên short-chunk bằng đúng `short-chunk trực tiếp` như pipeline
chính.

**Cách mining (short-chunk HYBRID + mở rộng theo Điều), tách 2 tầng để tận dụng cache:**

- **Tầng 1 — `mine_candidate_pools()` (ĐẮT, cache ra đĩa):** với mỗi câu hỏi, tính BM25 + Dense trên
  toàn bộ short-chunk, min-max normalize từng tín hiệu rồi cộng `MINING_LAMBDA * BM25_norm +
  (1-MINING_LAMBDA) * Dense_norm`. Từ cùng mảng điểm hybrid này: (a) chọn short-chunk điểm cao nhất
  trong từng document gold làm positive representative, và (b) chọn short-chunk điểm cao nhất của
  tối đa `NEG_MINING_POOL_K` document khác nhau làm hard-negative pool.
- **Tầng 2 — `build_training_pairs_from_pool()` (RẺ, chạy lại mỗi epoch):** `random.sample()` từ
  negative pool đã mine sẵn ở tầng 1, rồi dùng trực tiếp short-chunk bằng đúng
  `short-chunk trực tiếp` / `short-chunk trực tiếp`, giống pipeline chính.

Do đó positive và negative trong training đều được mining bởi **một scoring function duy nhất**;
không còn trường hợp positive = BM25-only nhưng negative = hybrid.


In [ ]:
# Cell 10c — Mining positive/negative trên SHORT-chunk index — CHỈ TẦNG 1.
#
# Candidate pool lưu (doc_id, global_idx) để có thể tái sử dụng đúng short-chunk đã mine.
# Không còn hàm short-chunk trực tiếp(): training và inference đều dùng THẲNG
# short_chunk_text_of[chunk_id] làm passage cho reranker.
from collections import defaultdict

# ---- Lập chỉ mục doc_id -> list[index vào short_chunk_ids] MỘT LẦN ----
_doc_to_short_chunk_idx = defaultdict(list)
_short_global_to_local_pos = {}
for _idx, _cid in enumerate(short_chunk_ids):
    _doc = short_chunk_doc_of[_cid]
    _short_global_to_local_pos[_idx] = len(_doc_to_short_chunk_idx[_doc])
    _doc_to_short_chunk_idx[_doc].append(_idx)
print(f"Đã lập chỉ mục short-chunk theo document: {len(_doc_to_short_chunk_idx)} document.")


def _hybrid_mining_scores(question, q_tokens=None, q_emb=None, lam=MINING_LAMBDA):
    """Tính HYBRID score trên TOÀN BỘ short-chunk cho mining."""
    bm25_raw = bm25_scores_all(question, q_tokens=q_tokens)
    dense_raw = dense_scores_all(question, q_emb=q_emb)
    return lam * _minmax_normalize(bm25_raw) + (1.0 - lam) * _minmax_normalize(dense_raw)


def _hybrid_negative_pool(question, gold, pool_k, q_tokens=None, q_emb=None):
    """Trả về tối đa pool_k document KHÁC gold, xếp theo HYBRID score.
    Mỗi document giữ global_idx của short-chunk điểm cao nhất."""
    scores = _hybrid_mining_scores(question, q_tokens=q_tokens, q_emb=q_emb, lam=MINING_LAMBDA)
    order = np.argsort(scores)[::-1]
    seen_docs = set()
    pool = []
    for idx in order:
        if len(pool) >= pool_k:
            break
        cid = short_chunk_ids[idx]
        d = short_chunk_doc_of[cid]
        if d in gold or d in seen_docs:
            continue
        seen_docs.add(d)
        pool.append((d, int(idx)))
    return pool, scores


def _best_short_chunk_idx_in_doc(doc_id, scores):
    """global_idx của short-chunk có HYBRID score cao nhất trong document."""
    idxs = _doc_to_short_chunk_idx.get(doc_id)
    if not idxs:
        return None
    return max(idxs, key=lambda i: scores[i])


_CANDIDATE_POOL_CACHE_PATH = os.path.join(
    "cache",
    f"7_vn_embed_v2_mining_candidate_pools_{SHORT_CHUNK_CHARS}_pk{NEG_MINING_POOL_K}_lam{MINING_LAMBDA:.1f}.pkl"
)


def mine_candidate_pools(samples, pool_k=NEG_MINING_POOL_K,
                          cache_path=_CANDIDATE_POOL_CACHE_PATH, force_remine=False):
    """TẦNG 1: mining positive + negative bằng CÙNG HYBRID score trên short-chunk.

    Với mỗi sample: tính BM25 + Dense trên toàn bộ short-chunk; chọn short-chunk điểm cao
    nhất trong mỗi gold document làm positive representative; ngoài gold chọn tối đa pool_k
    document khác nhau, mỗi document lấy short-chunk điểm cao nhất.
    """
    current_qids = {s["qid"] for s in samples}
    if not force_remine and os.path.exists(cache_path):
        with open(cache_path, "rb") as f:
            cached = pickle.load(f)
        if set(cached.keys()) == current_qids:
            print(f"Đã load HYBRID mining candidate pool từ cache: {cache_path} ({len(cached)} câu hỏi).")
            return cached
        print(f"Cache {cache_path} không khớp bộ câu hỏi hiện tại ({len(cached)} vs {len(current_qids)}) -> mine lại.")

    pools = {}
    n_skipped = 0
    for sample in tqdm(samples, desc=f"Mining candidate pool (HYBRID lambda={MINING_LAMBDA:.1f})"):
        qid, question, gold = sample["qid"], sample["question"], set(sample["answers"])
        q_tokens = _question_token_cache_fast.get(qid)
        if q_tokens is None:
            q_tokens = vi_tokenize_short(question)

        neg_pool, hybrid_scores = _hybrid_negative_pool(
            question, gold, pool_k, q_tokens=q_tokens, q_emb=None
        )

        positives = []
        for g in gold:
            if g not in corpus:
                n_skipped += 1
                continue
            global_idx = _best_short_chunk_idx_in_doc(g, hybrid_scores)
            if global_idx is None:
                n_skipped += 1
                continue
            positives.append((g, int(global_idx)))

        pools[qid] = {"positives": positives, "neg_pool": neg_pool}

    if n_skipped:
        print(f"Bỏ qua {n_skipped} document gold không tìm được short-chunk đại diện.")

    with open(cache_path, "wb") as f:
        pickle.dump(pools, f)
    print(f"Đã mine xong HYBRID candidate pool cho {len(pools)} câu hỏi, lưu tại {cache_path}")
    return pools


fit_candidate_pools = mine_candidate_pools(fit_samples, pool_k=NEG_MINING_POOL_K)
print("Số câu hỏi có pool:", len(fit_candidate_pools))

# ---- Dọn RAM: corpus (text gốc chưa chunk) không còn được dùng ở bất kỳ cell nào phía sau ----
del corpus
gc.collect()
ctypes.CDLL("libc.so.6").malloc_trim(0)


Đã lập chỉ mục short-chunk theo document: 8511 document.
Đã load HYBRID mining candidate pool từ cache: /mnt/mmlab2024nas/trantran/7_vn_embed_v2_mining_candidate_pools_2000_pk100_lam0.4.pkl (6300 câu hỏi).
Số câu hỏi có pool: 6300


1

In [20]:
# Cell 10c2 — TẦNG 1.5 (MỚI, RẺ — chạy mỗi epoch, hoặc mỗi N step nếu gọi thủ công với `qids`):
# refresh_pool_scores(pools) — forward pass reranker (điểm "yes") trên TỪNG candidate ĐÃ CÓ SẴN
# trong neg_pool (mine ở TẦNG 1 / Cell 10c, KHÔNG quét lại corpus, KHÔNG gọi lại BM25/dense) để cập
# nhật "độ khó hiện tại" của mỗi candidate DƯỚI GÓC NHÌN của chính checkpoint đang train.
#
# ĐÃ SỬA (InfoNCE — khớp với Cell 10d): negative dùng để TRAIN/TÍNH LOSS giờ chỉ lấy trong top
# NEG_TRAIN_POOL_TOP_K (=30, Cell 2) đầu của neg_pool, KHÔNG còn dùng tới 70 candidate còn lại
# (vị trí 30-99) của mining pool 100 nữa -> re-score CẢ 100 candidate mỗi epoch là dư thừa, tốn thêm
# ~3.3x forward pass vô ích. Hàm này giờ CHỈ rerank_score() trên đúng NEG_TRAIN_POOL_TOP_K candidate
# đầu tiên của neg_pool (neg_pool đã sort giảm dần theo hybrid/dense score GỐC lúc mine — Cell 10c —
# nên top NEG_TRAIN_POOL_TOP_K đầu luôn là đúng tập được dùng để train). Mining pool 100 (Cell 10c)
# thì KHÔNG đổi — chỉ khâu re-score (TẦNG 1.5) này bị thu hẹp lại theo đúng phạm vi thật sự dùng.
#
# Yêu cầu: Cell 10c đã chạy xong (fit_candidate_pools trong session) VÀ
# Cell 12 (load reranker_model + rerank_score) đã chạy. Ở vị trí notebook hiện tại (trước Cell 12),
# reranker_model CHƯA tồn tại -> cell này chỉ ĐỊNH NGHĨA hàm, KHÔNG gọi ngay; refresh_pool_scores()
# thật sự được gọi mỗi epoch trong training loop (Cell 14), SAU KHI Cell 12/13 đã chạy.
#
# _pool_neg_scores: dict qid -> np.ndarray điểm reranker hiện tại, ĐỘ DÀI <= NEG_TRAIN_POOL_TOP_K,
# CÙNG THỨ TỰ với NEG_TRAIN_POOL_TOP_K phần tử ĐẦU của fit_candidate_pools[qid]["neg_pool"]. Cell
# 10d (TẦNG 2) đọc dict này để chọn negative theo rank MỚI NHẤT; nếu qid chưa có trong dict (chưa
# refresh lần nào), Cell 10d tự fallback về random.sample() trên top NEG_TRAIN_POOL_TOP_K của
# neg_pool giống hành vi gốc.

_pool_neg_scores = {}  # qid -> np.ndarray, cùng thứ tự với pools[qid]["neg_pool"]


@torch.no_grad()
def refresh_pool_scores(pools, samples=None, batch_size=POOL_SCORE_BATCH_SIZE, qids=None, desc=None):
    """TẦNG 1.5: forward pass reranker_model (checkpoint HIỆN TẠI — dù đang train dở hay vừa load)
    trên từng candidate trong TOP NEG_TRAIN_POOL_TOP_K của neg_pool (ĐÃ SỬA — không còn re-score cả
    neg_pool 100 nữa, chỉ đúng phạm vi thật sự dùng để train, xem comment đầu Cell) của từng câu hỏi,
    lưu điểm 'yes' (xác suất liên quan) vào _pool_neg_scores[qid]. Chỉ tính lại rerank_score() trên
    candidate ĐÃ CÓ SẴN trong pool (TẦNG 1) -> rẻ hơn hẳn TẦNG 1 (không có bm25.get_scores()/dense
    encode nào), an toàn để gọi mỗi epoch.

    pools: fit_candidate_pools (hoặc tập con) từ mine_candidate_pools (Cell 10c).
    samples: nếu truyền vào (vd. fit_samples), dùng để tra `question` theo qid; None -> dùng
        qid_to_question (map dựng sẵn từ fit_samples ngay dưới đây).
    qids: giới hạn chỉ refresh một tập con qid (vd. muốn refresh mỗi N step thay vì mỗi epoch, chỉ
        cần truyền các qid vừa xuất hiện trong N step gần nhất) thay vì toàn bộ pools.
    """
    was_training = reranker_model.training
    reranker_model.eval()

    target_qids = list(qids) if qids is not None else list(pools.keys())
    q_lookup = qid_to_question if samples is None else {s["qid"]: s["question"] for s in samples}

    for qid in tqdm(target_qids, desc=desc or "Refresh pool scores (TẦNG 1.5)"):
        pool_entry = pools.get(qid)
        if pool_entry is None or not pool_entry["neg_pool"]:
            continue
        question = q_lookup.get(qid)
        if question is None:
            continue

        # ĐÃ SỬA: chỉ score top NEG_TRAIN_POOL_TOP_K đầu của neg_pool (đúng phạm vi Cell 10d dùng để
        # train, xem comment đầu Cell) thay vì toàn bộ neg_pool (100) — giảm ~3.3x forward pass/câu
        # hỏi mỗi lần refresh mà không mất thông tin nào (70 candidate còn lại không được dùng làm
        # negative train nên không cần re-score).
        neg_pool = pool_entry["neg_pool"][:NEG_TRAIN_POOL_TOP_K]
        texts, valid_idx = [], []
        for i, (d, global_idx) in enumerate(neg_pool):
            cid = short_chunk_ids[global_idx]
            short_text = short_chunk_text_of[cid]
            if not short_text or not short_text.strip():
                continue
            texts.append(short_text)
            valid_idx.append(i)
        if not texts:
            continue

        scores = rerank_score(question, texts, batch_size=batch_size)

        arr = np.full(len(neg_pool), -np.inf, dtype=np.float32)
        for i, sc in zip(valid_idx, scores):
            arr[i] = sc
        _pool_neg_scores[qid] = arr

    if was_training:
        reranker_model.train()


qid_to_question = {s["qid"]: s["question"] for s in fit_samples}

# KHÔNG gọi refresh_pool_scores() ở đây: reranker_model chưa được load ở vị trí này trong notebook
# (Cell 12 nằm SAU). Lần refresh ĐẦU TIÊN diễn ra trong training loop (Cell 14), trước epoch 2 trở
# đi (sau khi checkpoint đã được fine-tune ít nhất 1 epoch) — xem đoạn "refresh_pool_scores(...)"
# được chèn vào đầu vòng lặp epoch ở Cell 14.
print("Đã định nghĩa refresh_pool_scores() (TẦNG 1.5). Sẽ được gọi mỗi epoch trong training loop "
      "(Cell 14), sau khi Cell 12/13 đã chạy.")


Đã định nghĩa refresh_pool_scores() (TẦNG 1.5). Sẽ được gọi mỗi epoch trong training loop (Cell 14), sau khi Cell 12/13 đã chạy.


In [21]:
# Cell 10c3 — TẦNG 1.5 (bổ sung): refresh_pool_pos_scores() + ngưỡng false-negative ĐỘNG
# Vấn đề với FALSE_NEG_RERANK_THRESHOLD tĩnh (Cell 2, = 0.9): ngưỡng "điểm reranker cao" mang ý
# nghĩa khác nhau tuỳ epoch — model mới train 1 epoch thường tự tin thấp hơn hẳn model đã train
# nhiều epoch, nên 1 ngưỡng cố định có thể QUÁ CHẶT ở epoch đầu (không lọc được false-negative nào)
# hoặc QUÁ LỎNG ở epoch cuối (lọc nhầm cả hard-negative thật). Cách khắc phục: lấy percentile thấp
# (mặc định p10) của điểm reranker HIỆN TẠI trên chính các known-positive — tức "ngưỡng đúng" là
# mức điểm mà ~90% positive thật đạt được ở checkpoint đang xét, tự khớp theo độ tự tin thực tế của
# model tại từng epoch, không cần đoán số cố định.
#
# refresh_pool_pos_scores() giống hệt refresh_pool_scores() (Cell 10c2) nhưng chấm điểm reranker
# HIỆN TẠI cho `positives` thay vì `neg_pool` — RẺ hơn nhiều vì mỗi câu hỏi chỉ có 1-3 positive
# (so với NEG_MINING_POOL_K=50 negative), nên gọi thêm mỗi epoch gần như không tốn chi phí đáng kể.
#
# Yêu cầu: giống Cell 10c2, hàm short-chunk trực tiếp/rerank_score/qid_to_question phải đã có
# sẵn trong session (Cell 10c, Cell 10c2, Cell 12). Ở vị trí notebook hiện tại (trước Cell 12),
# reranker_model CHƯA tồn tại -> cell này chỉ ĐỊNH NGHĨA hàm, KHÔNG gọi ngay; refresh_pool_pos_scores()
# thật sự được gọi mỗi epoch trong training loop (Cell 14), NGAY SAU refresh_pool_scores().

_pool_pos_scores = {}  # qid -> np.ndarray điểm reranker hiện tại trên known-positive của câu hỏi đó
                        # (không nhất thiết cùng độ dài với pools[qid]["positives"] nếu có positive
                        # bị bỏ qua do không xử lý được short-chunk — chỉ dùng để gộp lại tính
                        # percentile toàn dataset, không cần giữ alignment vị trí như _pool_neg_scores)


@torch.no_grad()
def refresh_pool_pos_scores(pools, samples=None, batch_size=POOL_SCORE_BATCH_SIZE, qids=None, desc=None):
    """TẦNG 1.5 (positives): forward pass reranker_model (checkpoint HIỆN TẠI) trên từng known-
    positive của từng câu hỏi, lưu điểm 'yes' vào _pool_pos_scores[qid]. Dùng cho
    compute_dynamic_false_neg_threshold() ngay dưới đây. RẺ hơn hẳn refresh_pool_scores (Cell 10c2)
    vì mỗi câu hỏi chỉ có 1-3 positive thay vì cả neg_pool.

    pools: fit_candidate_pools (hoặc tập con) từ mine_candidate_pools (Cell 10c).
    samples/qids: giống hệt refresh_pool_scores (Cell 10c2)."""
    was_training = reranker_model.training
    reranker_model.eval()

    target_qids = list(qids) if qids is not None else list(pools.keys())
    q_lookup = qid_to_question if samples is None else {s["qid"]: s["question"] for s in samples}

    for qid in tqdm(target_qids, desc=desc or "Refresh positive scores (TẦNG 1.5 - positives)"):
        pool_entry = pools.get(qid)
        if pool_entry is None or not pool_entry["positives"]:
            continue
        question = q_lookup.get(qid)
        if question is None:
            continue

        positives = pool_entry["positives"]
        texts = []
        for g, global_idx in positives:
            cid = short_chunk_ids[global_idx]
            short_text = short_chunk_text_of[cid]
            if not short_text or not short_text.strip():
                continue
            texts.append(short_text)
        if not texts:
            continue

        scores = rerank_score(question, texts, batch_size=batch_size)
        _pool_pos_scores[qid] = np.array(scores, dtype=np.float32)

    if was_training:
        reranker_model.train()


_threshold_ema = None  # EMA của threshold TOÀN CỤC (fallback), làm mượt qua các epoch — xem
                        # compute_dynamic_false_neg_threshold. CHỈ dùng làm DỰ PHÒNG cho câu hỏi
                        # chưa có known-positive score riêng ở epoch hiện tại; ngưỡng THẬT SỰ dùng để
                        # lọc mỗi câu hỏi nằm ở local_false_neg_threshold() ngay bên dưới.


def compute_dynamic_false_neg_threshold(low_percentile=10, min_thr=0.5, max_thr=0.999, min_n=50,
                                         ema_alpha=0.3):
    """Threshold TOÀN CỤC (fallback) = percentile thấp (mặc định p10) của điểm known-positive GỘP
    TOÀN BỘ dataset ở checkpoint HIỆN TẠI (_pool_pos_scores, vừa refresh bởi
    refresh_pool_pos_scores() ngay phía trên).

    LƯU Ý (ĐÃ SỬA — quan trọng): với loss LISTWISE (Cell 27), sigmoid(logit) chỉ đảm bảo ĐÚNG THỨ TỰ
    trong nội bộ group 8 passage của TỪNG câu hỏi, KHÔNG có ràng buộc nào ép giá trị tuyệt đối của nó
    phải nhất quán GIỮA các câu hỏi khác nhau (vd. câu hỏi A có thể có positive "quen" quanh 0.7, câu
    hỏi B quanh 0.95 — cả 2 đều hợp lệ dưới listwise). Vì vậy 1 giá trị percentile GỘP TOÀN DATASET
    như hàm này trả về KHÔNG "công bằng" khi áp dụng chung cho mọi câu hỏi: dễ bỏ sót false-negative
    ở câu hỏi có thang điểm thấp, và lọc nhầm cả hard-negative thật ở câu hỏi có thang điểm cao.
    Ngưỡng THẬT SỰ dùng để lọc mỗi câu hỏi giờ được tính RIÊNG bằng local_false_neg_threshold() (dựa
    trên chính known-positive của CÙNG câu hỏi đó — so sánh NỘI BỘ trong đúng group mà loss listwise
    tối ưu, nên công bằng hơn nhiều). Hàm compute_dynamic_false_neg_threshold() này chỉ còn được giữ
    lại để tính NGƯỠNG DỰ PHÒNG toàn cục — dùng khi 1 câu hỏi CHƯA có known-positive score riêng ở
    epoch hiện tại (positive không chunk được, hoặc epoch 1 trước khi refresh_pool_pos_scores() chạy
    lần đầu) — xem local_false_neg_threshold() ngay bên dưới và _suspected_false_negatives (Cell 10d).

    min_thr/max_thr: kẹp threshold trong khoảng an toàn (tránh percentile ra giá trị quá thấp/quá
    cao nếu phân bố điểm positive lệch bất thường ở vài epoch đầu).
    min_n: số known-positive score tối thiểu cần có (gộp toàn dataset) mới tin percentile — nếu
    chưa đủ (vd. epoch 1, refresh_pool_pos_scores() chưa chạy lần nào) -> trả về None, nơi gọi hàm
    này (Cell 14) sẽ tự giữ nguyên FALSE_NEG_RERANK_THRESHOLD tĩnh cũ.
    ema_alpha: hệ số làm mượt EMA qua các epoch, tránh threshold nhảy loạn nếu percentile của 1
    epoch nào đó bị nhiễu (vd. ít known-positive xử lý được short-chunk hơn bình thường).
    """
    global _threshold_ema
    all_pos = np.concatenate([v for v in _pool_pos_scores.values() if len(v) > 0]) \
        if _pool_pos_scores else np.array([])
    if len(all_pos) < min_n:
        return None  # chưa đủ dữ liệu -> giữ nguyên threshold tĩnh cũ (Cell 14 tự bỏ qua khi None)

    raw_thr = float(np.clip(np.percentile(all_pos, low_percentile), min_thr, max_thr))
    _threshold_ema = raw_thr if _threshold_ema is None \
        else ema_alpha * raw_thr + (1 - ema_alpha) * _threshold_ema
    return float(_threshold_ema)


# ============================================================================================
# (MỚI) Threshold false-negative RIÊNG CHO TỪNG CÂU HỎI — thay cho việc dùng 1 threshold TUYỆT ĐỐI
# chung cho mọi câu hỏi (compute_dynamic_false_neg_threshold ở trên, giờ chỉ còn là fallback).
#
# Ý tưởng: so sánh điểm của 1 candidate trong neg_pool với điểm known-positive CỦA CHÍNH câu hỏi đó
# (_pool_pos_scores[qid]) — đây là so sánh NỘI BỘ trong cùng group mà softmax cross-entropy (Cell 27)
# thực sự tối ưu, nên "công bằng" bất kể thang điểm tuyệt đối của câu hỏi này cao hay thấp hơn câu
# hỏi khác. Cụ thể: 1 candidate bị nghi ngờ false-negative khi điểm reranker của nó cao ngang (hoặc
# hơn) chính positive YẾU NHẤT mà câu hỏi này đang có — tức nó "khó phân biệt với positive thật của
# CHÍNH câu hỏi này" chứ không phải so với 1 con số chung chung của toàn dataset.
#
# Dùng MIN thay vì percentile trên _pool_pos_scores[qid] vì mỗi câu hỏi thường chỉ có 1-3 positive —
# quá ít để tin 1 percentile riêng; lấy giá trị thấp nhất (positive "yếu nhất" mà câu hỏi này có) làm
# mốc thận trọng hơn. EMA theo TỪNG qid (không gộp chung 1 EMA toàn cục) để làm mượt qua các epoch mà
# vẫn giữ đặc trưng riêng của câu hỏi đó.
_per_qid_threshold_ema = {}  # qid -> EMA của threshold RIÊNG cho câu hỏi đó


def local_false_neg_threshold(qid, margin=None, min_thr=0.3, max_thr=0.999, ema_alpha=0.3,
                               fallback_thr=None):
    """Trả về threshold false-negative RIÊNG cho câu hỏi `qid`, dựa trên known-positive score CỦA
    CHÍNH câu hỏi đó (_pool_pos_scores[qid]) thay vì 1 giá trị tuyệt đối chung cho mọi câu hỏi.

    threshold = clip(min(_pool_pos_scores[qid]) - margin, min_thr, max_thr), làm mượt qua EMA riêng
    theo từng qid (_per_qid_threshold_ema) để tránh nhảy loạn giữa các epoch.

    margin: (mặc định đọc từ FALSE_NEG_LOCAL_MARGIN, Cell 2) trừ thêm khỏi positive yếu nhất của câu
        hỏi này để không lọc quá sát — margin càng lớn càng khoan dung (loại ít candidate hơn).
    fallback_thr: threshold TOÀN CỤC dùng khi câu hỏi `qid` CHƯA có known-positive score ở epoch
        hiện tại (vd. epoch 1, hoặc positive của câu hỏi này không chunk được) — thường truyền vào
        FALSE_NEG_RERANK_THRESHOLD (đã được Cell 14 ghi đè bằng compute_dynamic_false_neg_threshold()
        mỗi epoch, xem Cell 13/14). Trả về đúng fallback_thr này (có thể là None) khi thiếu dữ liệu.
    """
    if margin is None:
        margin = FALSE_NEG_LOCAL_MARGIN

    pos_scores = _pool_pos_scores.get(qid)
    if pos_scores is None or len(pos_scores) == 0:
        return fallback_thr  # không đủ dữ liệu riêng cho câu hỏi này -> dùng ngưỡng dự phòng toàn cục

    pos_floor = float(np.min(pos_scores))  # positive "yếu nhất" mà CHÍNH câu hỏi này đang có
    raw_thr = float(np.clip(pos_floor - margin, min_thr, max_thr))

    prev = _per_qid_threshold_ema.get(qid)
    ema = raw_thr if prev is None else ema_alpha * raw_thr + (1 - ema_alpha) * prev
    _per_qid_threshold_ema[qid] = ema
    return ema


print("Đã định nghĩa refresh_pool_pos_scores(), compute_dynamic_false_neg_threshold() (fallback "
      "toàn cục) và local_false_neg_threshold() (ngưỡng RIÊNG cho từng câu hỏi — xem _suspected_"
      "false_negatives, Cell 10d). Cả hai đều được gọi mỗi epoch trong training loop (Cell 14), "
      "ngay sau refresh_pool_scores().")

Đã định nghĩa refresh_pool_pos_scores(), compute_dynamic_false_neg_threshold() (fallback toàn cục) và local_false_neg_threshold() (ngưỡng RIÊNG cho từng câu hỏi — xem _suspected_false_negatives, Cell 10d). Cả hai đều được gọi mỗi epoch trong training loop (Cell 14), ngay sau refresh_pool_scores().


In [ ]:
# Cell 10d — TẦNG 2 (RẺ — dùng trực tiếp SHORT-chunk): build_training_pairs_from_pool() -> gọi lại mỗi epoch (Cell 14/rebuild_train_loader)
# để có bộ negative khác nhau, không cần tính lại BM25. Yêu cầu Cell 10c (mine_candidate_pools) và
# Cell 10c2 (refresh_pool_scores, định nghĩa _pool_neg_scores) đã chạy, và Cell 6b đã chạy (định
# nghĩa short-chunk trực tiếp mà short-chunk trực tiếp gọi).
#
# ĐÃ SỬA (so với bản chỉ dùng random.sample() thuần): chọn negative giờ gồm 2 bước.
#   Bước 1 (MỚI) — lọc "nghi ngờ false-negative" bằng XÁC NHẬN CHÉO 2 NGUỒN ĐỘC LẬP
#     (_suspected_false_negatives): chỉ loại candidate khi CẢ hybrid rank GỐC (TẦNG 1, BM25+Dense —
#     KHÔNG phụ thuộc model đang train) LẪN điểm reranker HIỆN TẠI (TẦNG 1.5, checkpoint đang train)
#     đều đồng thuận là rất liên quan -> tránh dùng chính model đang train để tự quyết định false-
#     negative (feedback loop: model lệch rồi tự "xác nhận" cái lệch đó). Nếu chỉ 1 trong 2 nguồn
#     nghi ngờ -> GIỮ LẠI làm negative bình thường (thận trọng thiên về giữ, không thiên về loại).
#   Bước 2 — trong phần còn lại, ưu tiên rank TẦNG 1.5 (_pool_neg_scores — điểm reranker mới nhất):
#     sort giảm dần theo điểm đó, lấy top NEG_HARD_TOP_FRAC candidate khó nhất làm "hard pool", rồi
#     random.sample() trong đó để vẫn có bộ negative khác nhau mỗi epoch (tránh học tủ 1 bộ cố định).
# Nếu qid CHƯA có _pool_neg_scores (chưa refresh lần nào, vd. epoch đầu tiên) -> bỏ qua bước 1 (không
# đủ 2 nguồn để xác nhận chéo) và bước 2 fallback random.sample() trên toàn bộ neg_pool, giống hệt
# hành vi gốc.
#
# Bản streaming: ghi pairs ra đĩa theo batch pickle nối tiếp thay vì giữ toàn bộ list `pairs`
# trong RAM khi build (đỉnh RAM ở bản cũ = build xong + shuffle xong cùng lúc tồn tại 2 bản).
import gc
import pickle


# Thống kê số candidate bị loại vì nghi ngờ false-negative (2 nguồn đồng thuận) — reset mỗi lần
# build_training_pairs_from_pool*/được gọi, chỉ để in ra theo dõi, KHÔNG ảnh hưởng logic.
_false_neg_stats = {"n_checked_qid": 0, "n_suspected_total": 0}


def _suspected_false_negatives(qid, neg_pool):
    """Trả về set các VỊ TRÍ (index vào neg_pool) bị nghi ngờ là false-negative (thật ra có thể là
    positive chưa được gán nhãn), dựa trên XÁC NHẬN CHÉO 2 nguồn ĐỘC LẬP — KHÔNG chỉ dùng điểm của
    model đang train, để tránh feedback loop (model tự lệch rồi tự "xác nhận" cái lệch đó):

      (a) điểm reranker HIỆN TẠI (TẦNG 1.5, _pool_neg_scores — phụ thuộc checkpoint đang train)
          >= threshold RIÊNG cho câu hỏi qid (ĐÃ SỬA — xem local_false_neg_threshold, Cell 10c3):
          với loss LISTWISE (Cell 27), sigmoid(logit) chỉ đảm bảo đúng thứ tự NỘI BỘ trong group 8
          passage của TỪNG câu hỏi, KHÔNG ép giá trị tuyệt đối nhất quán GIỮA các câu hỏi khác nhau
          (câu hỏi A có thể "quen" quanh 0.7, câu hỏi B quanh 0.95 — cả 2 đều hợp lệ) -> 1 threshold
          TUYỆT ĐỐI dùng chung cho mọi câu hỏi (kể cả bản percentile "động" gộp toàn dataset cũ) là
          KHÔNG công bằng. Threshold giờ được tính RIÊNG cho qid này, dựa trên known-positive score
          của CHÍNH câu hỏi đó (so sánh nội bộ, đúng thang mà loss thực sự tối ưu); chỉ fallback về
          ngưỡng toàn cục FALSE_NEG_RERANK_THRESHOLD khi câu hỏi này chưa có known-positive score
          riêng ở epoch hiện tại. VÀ
      (b) hybrid rank GỐC (TẦNG 1, BM25+Dense — KHÔNG phụ thuộc model đang train) nằm trong top
          FALSE_NEG_HYBRID_TOP_FRAC đầu của neg_pool. neg_pool đã được _hybrid_negative_pool (Cell
          10c) sort giảm dần theo hybrid score ngay lúc mine, nên chỉ cần dùng VỊ TRÍ trong neg_pool
          làm proxy cho hybrid rank, không cần lưu thêm điểm hybrid nào. (b) không bị ảnh hưởng bởi
          vấn đề calibration của listwise loss vì hybrid rank không phụ thuộc model đang train.

    CHỈ loại khi CẢ (a) VÀ (b) cùng đúng (thận trọng thiên về GIỮ). Nếu chưa có _pool_neg_scores cho
    qid (TẦNG 1.5 chưa refresh) -> không đủ 2 nguồn để xác nhận chéo -> trả về set rỗng (không loại
    gì cả), giữ nguyên hành vi gốc.
    """
    # ĐÃ SỬA (InfoNCE — cắt neg_pool về top NEG_TRAIN_POOL_TOP_K trước khi train, xem
    # _sample_negatives_for_qid): `neg_pool` truyền vào đây có thể là bản ĐÃ CẮT (prefix) của
    # neg_pool GỐC đã mine (100 candidate). _pool_neg_scores[qid] luôn được refresh_pool_scores()
    # tính trên neg_pool GỐC (đủ 100), nên chỉ cần lấy ĐÚNG PREFIX cùng độ dài với `neg_pool` truyền
    # vào (2 mảng cùng thứ tự ngay từ lúc mine) là khớp index 1-1, không cần độ dài bằng nhau.
    scores_arr_full = _pool_neg_scores.get(qid)
    if scores_arr_full is None or len(scores_arr_full) < len(neg_pool):
        return set()
    scores_arr = scores_arr_full[:len(neg_pool)]

    # (a) ngưỡng RIÊNG cho câu hỏi này (Cell 10c3), fallback về ngưỡng toàn cục nếu câu hỏi chưa có
    # known-positive score ở epoch hiện tại.
    local_thr = local_false_neg_threshold(qid, fallback_thr=FALSE_NEG_RERANK_THRESHOLD)
    if local_thr is None:
        local_thr = FALSE_NEG_RERANK_THRESHOLD

    hybrid_cutoff = max(1, int(len(neg_pool) * FALSE_NEG_HYBRID_TOP_FRAC))
    suspected = {
        i for i in range(hybrid_cutoff)   # (b) nằm trong top hybrid rank gốc
        if scores_arr[i] >= local_thr     # (a) VÀ điểm reranker hiện tại rất cao SO VỚI CHÍNH câu hỏi này
    }
    if suspected:
        _false_neg_stats["n_suspected_total"] += len(suspected)
    _false_neg_stats["n_checked_qid"] += 1
    return suspected


def _sample_negatives_for_qid(qid, neg_pool, gold, n_neg):
    """Chọn tối đa n_neg negative (doc_id, global_idx) từ neg_pool cho câu hỏi qid.

    Bước 0 (MỚI — InfoNCE): CẮT neg_pool (mine đủ NEG_MINING_POOL_K=100 document, Cell 10c) về đúng
    top NEG_TRAIN_POOL_TOP_K (=30, mặc định = TOP_K_STAGE1) vị trí ĐẦU trước khi làm bất cứ điều gì
    khác. neg_pool đã được _hybrid_negative_pool (Cell 10c) sort giảm dần theo hybrid/dense score GỐC
    ngay lúc mine, nên top-30 đầu chính là top-30 ứng viên stage-1 (dense retrieval) ban đầu — ĐÚNG
    phạm vi candidate mà pipeline thật sự đưa vào reranker lúc inference. Toàn bộ negative dùng để
    TÍNH LOSS (InfoNCE/listwise) chỉ được lấy trong phạm vi 30 này; 70 document còn lại của mining
    pool (vị trí 30-99) vẫn nằm trong neg_pool GỐC (fit_candidate_pools, không bị xoá) nhưng KHÔNG
    còn được TẦNG 1.5 re-score (refresh_pool_scores, Cell 10c2, ĐÃ SỬA để chỉ score đúng top
    NEG_TRAIN_POOL_TOP_K) và KHÔNG còn được chọn làm negative train nữa — bỏ hẳn compute lãng phí
    trên phần không dùng tới. (ĐÃ BỎ logic "ưu tiên" nhóm hard vừa-nằm-trong-top-K ở bản trước —
    không còn cần thiết vì giờ toàn bộ hard_pool vốn đã nằm gọn trong top-30 này luôn.)

    Bước 1 — lọc nghi ngờ false-negative bằng xác nhận chéo 2 nguồn (xem
    _suspected_false_negatives, hoạt động trên phần đã cắt ở Bước 0): loại các candidate mà CẢ hybrid
    rank gốc (TẦNG 1) LẪN điểm reranker hiện tại (TẦNG 1.5) đều đồng thuận là rất liên quan, tránh
    dùng chính unlabeled true-positive làm negative để train (feedback loop). Nếu chỉ 1 nguồn nghi
    ngờ -> vẫn giữ, không loại.

    Bước 2 — MIX hard/easy thay vì lấy 100% từ hard pool: chia n_neg theo NEG_MIX_HARD_FRAC (Cell 2).
      - n_hard = round(n_neg * NEG_MIX_HARD_FRAC): random.sample() trong top NEG_HARD_TOP_FRAC
        candidate khó nhất (theo điểm reranker TẦNG 1.5 mới nhất) CỦA PHẦN ĐÃ CẮT ở Bước 0.
      - n_easy = n_neg - n_hard: sample RANDOM ĐỀU trên toàn bộ eligible pool (đã cắt), loại trừ
        candidate đã chọn ở phần hard, để model vẫn tiếp xúc với negative "dễ"/hạng thấp mỗi epoch
        thay vì chỉ toàn candidate khó nhất -> giảm nguy cơ feedback loop.
      - Nếu chưa có điểm TẦNG 1.5 cho qid này (vd. epoch đầu tiên, trước lần refresh_pool_scores()
        đầu tiên) -> fallback random.sample() trên toàn bộ eligible pool (đã cắt), giống hệt hành vi
        gốc (chưa có điểm để xếp hạng hard/easy).

    Nếu vẫn thiếu (n_neg > số candidate khả dụng TRONG top-30 đã cắt), bổ sung thêm document ngẫu
    nhiên ngoài gold/đã chọn (không giới hạn top-30 — chỉ là cơ chế fallback hiếm khi cần tới), giống
    cơ chế fallback gốc."""
    train_pool_size = min(len(neg_pool), NEG_TRAIN_POOL_TOP_K)
    neg_pool_train = neg_pool[:train_pool_size]  # Bước 0: cắt về top NEG_TRAIN_POOL_TOP_K

    suspected_idx = _suspected_false_negatives(qid, neg_pool_train)
    eligible_idx = [i for i in range(len(neg_pool_train)) if i not in suspected_idx]
    if not eligible_idx:
        # Hiếm khi xảy ra (toàn bộ pool đã cắt bị nghi ngờ) -> không để pool rỗng, dùng lại toàn bộ
        # neg_pool_train gốc (thận trọng thiên về giữ, thà lẫn vài false-negative còn hơn mất hết
        # negative).
        eligible_idx = list(range(len(neg_pool_train)))

    scores_arr_full = _pool_neg_scores.get(qid)
    scores_arr = scores_arr_full[:train_pool_size] \
        if scores_arr_full is not None and len(scores_arr_full) >= train_pool_size else None
    if scores_arr is not None:
        order = sorted(eligible_idx, key=lambda i: scores_arr[i], reverse=True)  # khó nhất trước
        hard_k = max(1, int(len(order) * NEG_HARD_TOP_FRAC))
        hard_pool = order[:hard_k]

        # ---- Bước 2a: phần "hard" — random.sample() thẳng trong hard_pool (đã nằm trong top-30 đã
        # cắt ở Bước 0, không cần logic ưu tiên thêm nữa như bản trước). ----
        n_hard = min(n_neg, round(n_neg * NEG_MIX_HARD_FRAC), len(hard_pool))
        hard_chosen = random.sample(hard_pool, n_hard) if n_hard > 0 else []

        # ---- Bước 2b: phần "easy/mix" — sample random đều trên eligible pool, loại trừ hard đã chọn ----
        n_easy = n_neg - len(hard_chosen)
        hard_chosen_set = set(hard_chosen)
        easy_candidates = [i for i in eligible_idx if i not in hard_chosen_set]
        easy_chosen = random.sample(easy_candidates, min(n_easy, len(easy_candidates))) if n_easy > 0 else []

        chosen_idx = hard_chosen + easy_chosen
        # ĐÃ SỬA (QUAN TRỌNG): SHUFFLE để thứ tự trả về KHÔNG còn mang ý nghĩa hard/easy theo vị trí.
        # Bản cũ trả về [hard..., easy...] và _groups_for_sample cắt tuần tự từng khối liên tiếp ->
        # gold đầu tiên nhận gần hết hard negative, gold sau chỉ nhận easy (mất cân bằng không chủ
        # đích). Sau khi shuffle, mọi cách cắt/chia về sau đều nhận được tỉ lệ hard/easy như nhau.
        random.shuffle(chosen_idx)
    else:
        # Chưa refresh TẦNG 1.5 cho qid này (vd. epoch đầu tiên, trước khi Cell 14 gọi
        # refresh_pool_scores lần đầu) -> fallback rank hybrid ban đầu, giống hệt bản gốc.
        chosen_idx = random.sample(eligible_idx, min(n_neg, len(eligible_idx)))

    negs = [neg_pool_train[i] for i in chosen_idx]

    if len(negs) < n_neg:
        picked_docs = {d for d, _ in negs}
        extra_docs = [d for d in doc_ids_all if d not in gold and d not in picked_docs]
        random.shuffle(extra_docs)
        for d in extra_docs[: n_neg - len(negs)]:
            idxs = _doc_to_short_chunk_idx.get(d)
            if idxs:
                negs.append((d, idxs[0]))
    return negs


def build_training_pairs_from_pool(pools, samples, neg_per_pos=NEG_PER_POS,
                                    desc="Resample negative từ candidate pool (TẦNG 2 - rẻ)"):
    """Sinh training pairs (question, chunk_id, chunk_text, label) từ candidate pool đã mine sẵn ở
    TẦNG 1 (mine_candidate_pools, Cell 10c). KHÔNG tính lại BM25 — chỉ chọn negative theo rank TẦNG
    1.5 (_pool_neg_scores, xem _sample_negatives_for_qid/Cell 10d) rồi mở rộng short-chunk -> long-
    chunk BẰNG CÁCH GẮN PATH VÀO ĐẦU SHORT-CHUNK ("path | short-chunk", KHÔNG ghép thêm chunk lân cận,
    short-chunk trực tiếp/Cell 10c — ĐÚNG cách select_relevant_chunks dùng trong pipeline
    chính). Gọi lại hàm này mỗi epoch (random.seed khác nhau trước khi gọi, VÀ refresh_pool_scores()
    đã chạy cho epoch đó ở Cell 14) để có bộ negative "khó" khác nhau, cập nhật theo checkpoint mới
    nhất, mà không tốn thêm chi phí BM25 nào.
    Trả về list trong RAM như bản gốc — dùng cho samples nhỏ/vừa. Với samples lớn, dùng
    build_training_pairs_from_pool_streaming bên dưới để tránh đỉnh RAM."""
    pairs = []
    n_skipped = 0
    for sample in tqdm(samples, desc=desc):
        qid, question, gold = sample["qid"], sample["question"], set(sample["answers"])
        pool_entry = pools.get(qid)
        if pool_entry is None:
            continue

        for g, global_idx in pool_entry["positives"]:
            cid = short_chunk_ids[global_idx]
            short_text = short_chunk_text_of[cid]
            if not short_text or not short_text.strip():
                n_skipped += 1
                continue
            pairs.append((question, cid, short_text, 1))

        neg_pool = pool_entry["neg_pool"]
        n_pos = len(pool_entry["positives"]) or len(gold) or 1
        n_neg = neg_per_pos * n_pos
        negs = _sample_negatives_for_qid(qid, neg_pool, gold, n_neg)

        for d, global_idx in negs:
            cid = short_chunk_ids[global_idx]
            short_text = short_chunk_text_of[cid]
            if not short_text or not short_text.strip():
                n_skipped += 1
                continue
            pairs.append((question, cid, short_text, 0))

    random.shuffle(pairs)
    if n_skipped:
        print(f"Bỏ qua {n_skipped} pair do văn bản không chunk được (passage rỗng).")
    return pairs


def build_training_pairs_from_pool_streaming(pools, samples, out_path, neg_per_pos=NEG_PER_POS,
                                              batch_size=2000,
                                              desc="Resample negative từ candidate pool (TẦNG 2 - streaming)"):
    """Giống build_training_pairs_from_pool nhưng ghi ra đĩa theo batch pickle nối tiếp thay vì
    giữ toàn bộ `pairs` trong RAM. Dùng khi số lượng samples/pairs đủ lớn để bản in-RAM gây OOM.
    Negative cũng được chọn theo rank TẦNG 1.5 qua _sample_negatives_for_qid (xem docstring
    build_training_pairs_from_pool). Đọc lại bằng load_pairs_streaming()."""
    buf = []
    n_written = 0
    n_skipped = 0
    with open(out_path, "wb") as fout:
        for sample in tqdm(samples, desc=desc):
            qid, question, gold = sample["qid"], sample["question"], set(sample["answers"])
            pool_entry = pools.get(qid)
            if pool_entry is None:
                continue

            for g, global_idx in pool_entry["positives"]:
                cid = short_chunk_ids[global_idx]
                short_text = short_chunk_text_of[cid]
                if not short_text or not short_text.strip():
                    n_skipped += 1
                    continue
                buf.append((question, cid, short_text, 1))

            neg_pool = pool_entry["neg_pool"]
            n_pos = len(pool_entry["positives"]) or len(gold) or 1
            n_neg = neg_per_pos * n_pos
            negs = _sample_negatives_for_qid(qid, neg_pool, gold, n_neg)

            for d, global_idx in negs:
                cid = short_chunk_ids[global_idx]
                short_text = short_chunk_text_of[cid]
                if not short_text or not short_text.strip():
                    n_skipped += 1
                    continue
                buf.append((question, cid, short_text, 0))

            if len(buf) >= batch_size:
                pickle.dump(buf, fout)
                n_written += len(buf)
                buf = []
                gc.collect()  # ép giải phóng ngay, tránh dồn ứ RAM trước lần dump kế tiếp

        if buf:
            pickle.dump(buf, fout)
            n_written += len(buf)

    if n_skipped:
        print(f"Bỏ qua {n_skipped} pair do văn bản không chunk được (passage rỗng).")
    print(f"Đã ghi {n_written} pairs ra {out_path} (nhiều pickle batch nối tiếp, batch_size={batch_size}).")
    return n_written


def load_pairs_streaming(path):
    """Đọc lại toàn bộ batch đã ghi bởi build_training_pairs_from_pool_streaming, gộp thành 1 list
    và shuffle. Chỉ dùng khi RAM đủ chỗ chứa toàn bộ pairs cùng lúc — nếu không, train trực tiếp
    theo batch từ file thay vì gọi hàm này."""
    pairs = []
    with open(path, "rb") as f:
        while True:
            try:
                pairs.extend(pickle.load(f))
            except EOFError:
                break
    random.shuffle(pairs)
    return pairs


# ============================================================================================
# GROUP BUILDER (MỚI) — thay cho việc trả về list CẶP PHẲNG rồi shuffle toàn cục.
#
# LÝ DO: RerankDataset cũ là list (question, chunk_id, chunk_text, label) rồi DataLoader
# shuffle=True -> 1 positive và NEG_PER_POS negative của CÙNG 1 câu hỏi bị rải ra các batch khác
# nhau, model không bao giờ được so sánh trực tiếp chúng trong cùng 1 lần forward. Bản mới gom
# thành GROUP: mỗi group = (question, positive_text, [negative_text] * neg_per_pos) và DataLoader
# shuffle Ở CẤP GROUP -> mọi phần tử của 1 group LUÔN nằm chung 1 batch, tỉ lệ pos:neg trong mỗi
# batch cố định đúng 1:neg_per_pos (trước đây tỉ lệ này dao động ngẫu nhiên theo từng batch).
#
# ĐÃ SỬA (LISTWISE): cấu trúc group ở đây giờ là ĐIỀU KIỆN BẮT BUỘC chứ không còn là tối ưu "cho
# đẹp" nữa — Cell 13 dùng softmax cross-entropy TRÊN TỪNG GROUP (xem LISTWISE_TEMPERATURE, Cell 2),
# tức mẫu số của softmax chính là tập {positive + các negative CỦA CÙNG CÂU HỎI} trong group. Vì
# vậy: (1) positive PHẢI ở vị trí 0 của group (quy ước này Cell 12 dựa vào để đặt target = 0),
# (2) mỗi group phải có ÍT NHẤT 1 negative, nếu không loss của group đó bằng 0 và không sinh
# gradient (Cell 12 tự loại các group như vậy), và (3) không được trộn passage của 2 câu hỏi khác
# nhau vào cùng 1 group.
#
# ĐÃ SỬA (cách gán negative cho từng positive) — xem NEG_GROUP_SHARING (Cell 2):
# Bản CŨ: mine neg_per_pos * n_pos negative rồi CẮT thành n_pos KHỐI LIÊN TIẾP. Vì
# _sample_negatives_for_qid trả về theo thứ tự [hard..., easy...], khối đầu (gold thứ 1) nuốt gần
# hết hard negative, khối sau (gold thứ 2, 3...) chỉ còn easy negative -> mất cân bằng KHÔNG chủ
# đích, trong khi lúc inference MỌI gold đều phải vượt cùng những candidate khó trong top-30
# stage-1.
# Bản MỚI: (1) _sample_negatives_for_qid shuffle kết quả (thứ tự không còn mang thông tin hard/easy)
# và (2) mặc định NEG_GROUP_SHARING="shared" — mine đúng neg_per_pos negative rồi DÙNG CHUNG cho
# MỌI gold của câu hỏi, nên mỗi gold đều cạnh tranh với cùng một tập negative cùng độ khó. Chế độ
# "block" (chia round-robin) được giữ lại để rollback/so sánh. Số group và tổng số pair KHÔNG đổi.

def _groups_for_sample(pools, sample, neg_per_pos, neg_sharing=None):
    """Trả về (list[group], n_skipped) cho MỘT câu hỏi.

    group = (question, pos_text, [neg_text, ...]) — tối đa neg_per_pos negative.
    Positive/negative có text rỗng bị bỏ qua; nếu positive rỗng thì bỏ luôn cả group đó.

    ĐÃ SỬA — cách gán negative cho từng gold (xem NEG_GROUP_SHARING, Cell 2):

      * "shared" (MẶC ĐỊNH — Hướng B): mine ĐÚNG neg_per_pos negative CHO CẢ CÂU HỎI và dùng CHUNG
        bộ đó cho MỌI gold. Lý do: lúc inference, TẤT CẢ gold của câu hỏi cùng cạnh tranh trên MỘT
        candidate pool (top-30 stage-1) — nên mọi gold đều phải được luyện để thắng CÙNG một tập
        negative, với CÙNG tỉ lệ hard/easy. Bản cũ (cắt khối liên tiếp trên list [hard...,easy...])
        làm gold đầu tiên "chiến đấu" với gần hết hard negative còn gold thứ hai chỉ gặp easy
        negative -> gold sau không đủ tự tin để vượt candidate khó lúc inference.

      * "block": mine neg_per_pos * n_pos negative rồi chia cho từng gold theo ROUND-ROBIN (stride
        n_pos) thay vì cắt khối liên tiếp, nên hard/easy được rải đều cho mọi gold (kết hợp với việc
        _sample_negatives_for_qid đã shuffle, đây là bản vá tối thiểu của cơ chế cũ). Mỗi gold thấy
        negative KHÁC NHAU -> đa dạng hơn, nhưng mỗi gold chỉ thấy 1 phần pool.

    Số group (= số gold hợp lệ) và tổng số pair sinh ra GIỐNG HỆT bản cũ ở cả 2 chế độ.
    """
    if neg_sharing is None:
        neg_sharing = globals().get("NEG_GROUP_SHARING", "shared")

    qid, question, gold = sample["qid"], sample["question"], set(sample["answers"])
    pool_entry = pools.get(qid)
    if pool_entry is None:
        return [], 0

    n_skipped = 0

    # ---- Positive: giữ nguyên cách lấy của bản cũ ----
    pos_texts = []
    for g, global_idx in pool_entry["positives"]:
        cid = short_chunk_ids[global_idx]
        short_text = short_chunk_text_of[cid]
        if not short_text or not short_text.strip():
            n_skipped += 1
            continue
        pos_texts.append(short_text)
    if not pos_texts:
        return [], n_skipped

    # ---- Negative ----
    neg_pool = pool_entry["neg_pool"]
    n_pos = len(pool_entry["positives"]) or len(gold) or 1
    n_neg_request = neg_per_pos if neg_sharing == "shared" else neg_per_pos * n_pos
    negs = _sample_negatives_for_qid(qid, neg_pool, gold, n_neg_request)

    neg_texts = []
    for d, global_idx in negs:
        cid = short_chunk_ids[global_idx]
        short_text = short_chunk_text_of[cid]
        if not short_text or not short_text.strip():
            n_skipped += 1
            continue
        neg_texts.append(short_text)

    if not neg_texts:
        # Không có negative nào -> group sẽ không sinh gradient (Cell 12 loại) -> bỏ luôn.
        return [], n_skipped

    groups = []
    if neg_sharing == "shared":
        # Hướng B: MỌI gold dùng CHUNG một bộ negative (cùng tỉ lệ hard/easy, cùng độ khó).
        shared_block = neg_texts[:neg_per_pos]
        for pos_text in pos_texts:
            groups.append((question, pos_text, shared_block))
    else:
        # "block": chia round-robin (stride n_pos) -> hard/easy rải đều cho mọi gold.
        n_group = len(pos_texts)
        for k, pos_text in enumerate(pos_texts):
            block = neg_texts[k::n_group][:neg_per_pos]
            if len(block) < neg_per_pos:
                # Pool bị lọc bớt (text rỗng / pool nhỏ) -> bù thêm từ phần còn lại, không lặp lại
                # phần tử đã có trong block.
                in_block = set(map(id, block))
                block = block + [t for t in neg_texts if id(t) not in in_block][
                    : neg_per_pos - len(block)]
            if not block:
                block = neg_texts[:neg_per_pos]
            if not block:
                continue
            groups.append((question, pos_text, block))
    return groups, n_skipped


def build_training_groups_from_pool(pools, samples, neg_per_pos=NEG_PER_POS,
                                     neg_sharing=None,
                                     desc="Resample negative theo GROUP (TẦNG 2 - rẻ)"):
    """Bản group-based của build_training_pairs_from_pool: trả về list[(question, pos_text,
    [neg_text] * neg_per_pos)] đã shuffle Ở CẤP GROUP (không shuffle phần tử bên trong group —
    Cell 12 luôn đặt positive ở vị trí 0 để gán label)."""
    groups = []
    n_skipped = 0
    for sample in tqdm(samples, desc=desc):
        g, sk = _groups_for_sample(pools, sample, neg_per_pos, neg_sharing=neg_sharing)
        groups.extend(g)
        n_skipped += sk
    random.shuffle(groups)
    if n_skipped:
        print(f"Bỏ qua {n_skipped} passage do văn bản không chunk được (passage rỗng).")
    return groups


def build_training_groups_from_pool_streaming(pools, samples, out_path, neg_per_pos=NEG_PER_POS,
                                               neg_sharing=None, batch_size=500,
                                               desc="Resample negative theo GROUP (TẦNG 2 - streaming)"):
    """Bản streaming của build_training_groups_from_pool (ghi pickle batch nối tiếp ra đĩa thay vì
    giữ toàn bộ list group trong RAM). batch_size tính theo SỐ GROUP (mỗi group ~ 1+neg_per_pos
    passage), nên 500 group ≈ 3000 passage — tương đương batch_size=2000-3000 pair của bản cũ.
    Đọc lại bằng load_groups_streaming()."""
    buf, n_written, n_skipped = [], 0, 0
    with open(out_path, "wb") as fout:
        for sample in tqdm(samples, desc=desc):
            g, sk = _groups_for_sample(pools, sample, neg_per_pos, neg_sharing=neg_sharing)
            buf.extend(g)
            n_skipped += sk
            if len(buf) >= batch_size:
                pickle.dump(buf, fout)
                n_written += len(buf)
                buf = []
                gc.collect()
        if buf:
            pickle.dump(buf, fout)
            n_written += len(buf)

    if n_skipped:
        print(f"Bỏ qua {n_skipped} passage do văn bản không chunk được (passage rỗng).")
    print(f"Đã ghi {n_written} group ra {out_path} "
          f"(~{n_written * (1 + neg_per_pos)} pair, batch_size={batch_size} group).")
    return n_written


def load_groups_streaming(path):
    """Đọc lại toàn bộ group đã ghi bởi build_training_groups_from_pool_streaming, gộp thành 1 list
    và shuffle Ở CẤP GROUP."""
    groups = []
    with open(path, "rb") as f:
        while True:
            try:
                groups.extend(pickle.load(f))
            except EOFError:
                break
    random.shuffle(groups)
    return groups


# ---- Sinh training GROUP cho epoch đầu ----
# Ở epoch đầu tiên, _pool_neg_scores còn RỖNG (TẦNG 1.5 chưa được gọi lần nào) -> negative epoch 1
# tự động fallback về random.sample() trên neg_pool, giống hệt hành vi gốc. Từ epoch 2 trở đi, Cell
# 14 gọi refresh_pool_scores() trước khi rebuild_train_loader() nên negative sẽ được chọn theo rank
# TẦNG 1.5 (checkpoint vừa fine-tune ở epoch trước).
#
# ĐÃ SỬA: dùng build_training_groups_from_pool*() thay cho build_training_pairs_from_pool*() —
# đơn vị dữ liệu giờ là GROUP (1 positive + NEG_PER_POS negative CÙNG câu hỏi), xem Cell 12.
# Hai hàm *_pairs_* cũ vẫn được GIỮ LẠI phía trên (không còn được gọi ở đâu) để tiện so sánh/rollback.
# Đổi tên file cache (train_groups_7.pkl) để không load nhầm cache ĐỊNH DẠNG CŨ (list cặp phẳng).
# groups = build_training_groups_from_pool(fit_candidate_pools, fit_samples, neg_per_pos=NEG_PER_POS)

# --- Nhánh streaming (đang bật, giống bản cũ) ---
# ĐÃ SỬA: đổi tên cache để không load nhầm file sinh bởi cơ chế chia negative CŨ (cắt khối liên tiếp).
GROUPS_CACHE_PATH = "cache/train_groups_7list33.pkl"
build_training_groups_from_pool_streaming(fit_candidate_pools, fit_samples, GROUPS_CACHE_PATH,
                                           neg_per_pos=NEG_PER_POS)
train_groups = load_groups_streaming(GROUPS_CACHE_PATH)

print(f"Chế độ gán negative cho từng gold: NEG_GROUP_SHARING = {NEG_GROUP_SHARING!r}")
print("Tổng số training group:", len(train_groups))
print(f"Mỗi group = 1 positive + {NEG_PER_POS} negative -> tổng số pair ≈",
      sum(1 + len(negs) for _, _, negs in train_groups))
_q, _pos, _negs = train_groups[0]
print("Ví dụ group | câu hỏi:", _q[:80])
print("  positive:", len(_pos), "ký tự |", _pos[:80].replace("\n", " "))
print("  số negative:", len(_negs), "| negative đầu tiên:", _negs[0][:80].replace("\n", " "))
if _false_neg_stats["n_checked_qid"]:
    print(f"[Xác nhận chéo false-negative] Đã kiểm tra {_false_neg_stats['n_checked_qid']} câu hỏi "
          f"có điểm TẦNG 1.5, loại tổng cộng {_false_neg_stats['n_suspected_total']} candidate "
          f"nghi ngờ false-negative (cả hybrid rank gốc lẫn điểm reranker hiện tại đều đồng thuận).")
else:
    print("[Xác nhận chéo false-negative] Chưa có điểm TẦNG 1.5 cho câu hỏi nào (bình thường ở lần "
          "build đầu tiên, trước khi refresh_pool_scores() chạy lần đầu ở Cell 14) -> chưa lọc gì.")


Resample negative theo GROUP (TẦNG 2 - streaming): 100%|██████████| 6300/6300 [00:11<00:00, 559.66it/s]


Đã ghi 6866 group ra /mnt/mmlab2024nas/trantran/train_groups_7list3.pkl (~75526 pair, batch_size=500 group).
Chế độ gán negative cho từng gold: NEG_GROUP_SHARING = 'shared'
Tổng số training group: 6866
Mỗi group = 1 positive + 10 negative -> tổng số pair ≈ 75526
Ví dụ group | câu hỏi: Tập trung xây dựng và chỉnh đốn Đảng trong sạch khi thực hiện chương trình phát 
  positive: 1534 ký tự | --- Số: 96/NQ-CP Hà Nội, ngày 01 tháng 8 năm 2022 NGHỊ QUYẾT BAN HÀNH CHƯƠNG TRÌ
  số negative: 10 | negative đầu tiên: THỦ TƯỚNG CHÍNH PHỦ --- --- Số: 809/QĐ-TTg Hà Nội, ngày 12 tháng 7 năm 2022 QUYẾ
[Xác nhận chéo false-negative] Chưa có điểm TẦNG 1.5 cho câu hỏi nào (bình thường ở lần build đầu tiên, trước khi refresh_pool_scores() chạy lần đầu ở Cell 14) -> chưa lọc gì.


## 5. Fine-tune BAAI/bge-reranker-v2-m3 (cross-encoder) — LISTWISE LOSS

BAAI/bge-reranker-v2-m3 (kiến trúc XLM-RoBERTa) là model **sequence-classification**: cặp
(query, document) được đưa thẳng vào tokenizer dưới dạng 1 câu ghép `[query, document]`, model
trả về **1 logit duy nhất** `s = f(q, a)` thể hiện độ liên quan.

**Hàm loss (ĐÃ ĐỔI: pointwise BCE → listwise softmax cross-entropy).** Trước đây mỗi cặp
(q, a) được huấn luyện độc lập bằng `BCEWithLogitsLoss` với nhãn 1/0 — tức model bị ép đưa logit
của MỌI positive về `+∞` và MỌI negative về `-∞`, bất kể chúng thuộc câu hỏi nào. Điều đó không
khớp với thứ ta thực sự cần: một **thứ tự xếp hạng đúng trong nội bộ từng câu hỏi**.

Dữ liệu đã được tổ chức theo **group** = 1 positive `a⁺` và `N = NEG_PER_POS` hard-negative
`a₁⁻ … a_N⁻` của **cùng một câu hỏi** (Cell 10d). Loss mới tính softmax trên chính group đó:

$$\mathcal{L}(q) = -\log \frac{\exp\big(f(q, a^{+})/T\big)}
{\exp\big(f(q, a^{+})/T\big) + \sum_{j=1}^{N} \exp\big(f(q, a_j^{-})/T\big)}$$

với `T = LISTWISE_TEMPERATURE` (Cell 2). Đây chính là localized contrastive loss mà bản thân
BGE-reranker được huấn luyện gốc, tương đương cross-entropy `(N+1)` lớp với nhãn đúng luôn ở vị
trí 0. Model chỉ bị phạt khi một negative **vượt lên trên** positive của cùng câu hỏi, còn giá trị
tuyệt đối của logit thì được tự do — đúng bản chất bài toán ranking.

**Những gì thay đổi theo:**

| | Trước (BCE) | Sau (listwise) |
|---|---|---|
| Đơn vị loss | 1 cặp (q, a) | 1 group (q, a⁺, a⁻×N) |
| Số negative/positive | 5 | 15 (`NEG_PER_POS`) — mẫu số softmax càng rộng, tín hiệu tương phản càng mạnh |
| Batch | 2 group = 12 cặp/forward | 1 group = 16 cặp/forward, `GRAD_ACCUM_STEPS = 8` |
| LR | 2e-4 | 1e-4 (gradient không còn bị chia trung bình cho 6 cặp) |
| `sigmoid(logit)` | xác suất đã hiệu chỉnh | **chỉ còn dùng để xếp hạng**, không còn hiệu chỉnh |

Vì `sigmoid` là hàm đơn điệu, toàn bộ pipeline suy luận (`rerank_score`, `two_stage_retrieve`,
`refresh_pool_scores`) vẫn cho ra **đúng thứ tự** như cũ, không cần sửa. Chỉ có các ngưỡng tuyệt
đối (`RERANK_THRESHOLD = 0.99`) là mất ý nghĩa và phải tune lại — hiện `apply_rerank_filter` đang
tắt bước lọc theo ngưỡng nên mặc định không ảnh hưởng.


In [ ]:
# Cell 11 — Load BAAI/bge-reranker-v2-m3, bọc LoRA (train mới) HOẶC load checkpoint đã fine-tune (inference)
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

RERANKER_NAME = "BAAI/bge-reranker-v2-m3"
# ĐÃ SỬA: BAAI/bge-reranker-v2-m3 là model SEQUENCE-CLASSIFICATION (kiến
# trúc XLM-RoBERTa/encoder) — KHÁC HẲN Qwen3-Reranker (causal LM chấm điểm qua token "yes"/"no").
# MAX_LEN theo đúng max sequence length huấn luyện chính thức của model: 2304 token (256 cho query +
# 2048 cho passage) — dư dả cho 1 short-chunk (tối đa SHORT_CHUNK_CHARS ký tự
# Việt) cộng câu hỏi.
MAX_LEN = 2304

# ---- Cấu hình LoRA (chỉ áp dụng khi LOAD_PRETRAINED=False, tức train mới) ----
USE_LORA = True          # đặt False nếu muốn quay lại full fine-tune
LORA_R = 16               # rank adapter — 8/16/32 phổ biến, cao hơn = học được nhiều hơn nhưng tốn VRAM hơn
LORA_ALPHA = 32           # thường chọn = 2 * LORA_R
LORA_DROPOUT = 0.05
# ĐÃ SỬA: BAAI/bge-reranker-v2-m3 dùng kiến trúc XLM-RoBERTa (encoder) chứ không phải Qwen/Llama-style
# (decoder) như Qwen3-Reranker -> tên module attention khác hẳn, không còn q_proj/k_proj/v_proj/
# gate_proj/up_proj/down_proj. RobertaSelfAttention đặt tên "query"/"key"/"value" cho 3 phép chiếu
# attention — đây là các module LoRA cần nhắm tới.
LORA_TARGET_MODULES = ["query", "key", "value"]

# ---- Đường dẫn checkpoint đã lưu ở Cell 15 (dùng khi LOAD_PRETRAINED=True) ----
BEST_DIR         = "weights/vietnamese-v2-reranker-bge-7list33-lora-best"
FT_RERANKER_DIR  = "weights/vietnamese-v2-reranker-bge-7list33-lora-ft"
MERGED_DIR       = "weights/vietnamese-v2-reranker-bge-7list33-merged"

# ============================== BẬT/TẮT Ở ĐÂY ==============================
# True  -> KHÔNG train, chỉ load model đã fine-tune sẵn (ưu tiên MERGED_DIR, rồi tới
#          FT_RERANKER_DIR, rồi tới BEST_DIR) để chạy inference/submission.
# False -> khởi tạo LoRA MỚI từ base model để train ở Cell 13/14, Cell 15 sẽ GHI ĐÈ lên các
#          thư mục checkpoint ở trên.
LOAD_PRETRAINED = False
# =============================================================================

reranker_tokenizer = AutoTokenizer.from_pretrained(RERANKER_NAME)
reranker_model = AutoModelForSequenceClassification.from_pretrained(
    RERANKER_NAME, torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
    # SUA LAI (bo attn_implementation="eager"): ban dau ep "eager" de tranh RuntimeError
    # "expanded size ... non-singleton dimension 3" trong refresh_pool_scores(), nhung "eager"
    # tu materialize toan bo ma tran attention [batch, head, seq, seq] thay vi dung kernel toi
    # uu SDPA -> ton VRAM hon han, cong voi gradient_checkpointing_enable() phai RE-COMPUTE
    # forward luc backward -> gay CUDA OutOfMemoryError ngay o epoch 1 (xem run_7_fixed.log).
    # Quay lai attention mac dinh (sdpa) de train; bug shape o tren gio duoc xu ly rieng, AN
    # TOAN VE MEMORY, bang co che retry trong _forward_batch() ben duoi (khong can eager).
).to(DEVICE)

if LOAD_PRETRAINED:
    # ---- Chỉ inference: load lại checkpoint đã fine-tune, KHÔNG khởi tạo LoRA mới ----
    if os.path.isdir(MERGED_DIR):
        del reranker_model  # bỏ bản base vừa load ở trên, dùng thẳng bản đã merge cho nhẹ VRAM
        torch.cuda.empty_cache() if DEVICE == "cuda" else None
        reranker_model = AutoModelForSequenceClassification.from_pretrained(
            MERGED_DIR, torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
            # SUA LAI: bo attn_implementation="eager" (xem giai thich o khoi load base model tren) -
            # gay OOM khi train do ton VRAM hon nhieu; bug shape mismatch da duoc xu ly rieng.
        ).to(DEVICE)
        print(f"[LOAD_PRETRAINED=True] Đã load MERGED model (không cần peft) từ {MERGED_DIR}")
    elif os.path.isdir(FT_RERANKER_DIR):
        reranker_model = PeftModel.from_pretrained(reranker_model, FT_RERANKER_DIR).to(DEVICE)
        print(f"[LOAD_PRETRAINED=True] Đã load LoRA adapter từ {FT_RERANKER_DIR}")
    elif os.path.isdir(BEST_DIR):
        reranker_model = PeftModel.from_pretrained(reranker_model, BEST_DIR).to(DEVICE)
        print(f"[LOAD_PRETRAINED=True] Đã load LoRA adapter (best checkpoint) từ {BEST_DIR}")
    else:
        raise FileNotFoundError(
            "LOAD_PRETRAINED=True nhưng không tìm thấy checkpoint nào trong "
            f"{MERGED_DIR} / {FT_RERANKER_DIR} / {BEST_DIR}. "
            "Đặt LOAD_PRETRAINED=False nếu muốn train từ đầu."
        )
    reranker_model.eval()
else:
    # ---- Train mới: bọc LoRA lên base model, Cell 13/14/15 sẽ train và GHI ĐÈ checkpoint cũ ----
    if USE_LORA:
        lora_config = LoraConfig(
            task_type=TaskType.SEQ_CLS,   # ĐÃ SỬA: sequence-classification, không phải CAUSAL_LM
            r=LORA_R,
            lora_alpha=LORA_ALPHA,
            lora_dropout=LORA_DROPOUT,
            target_modules=LORA_TARGET_MODULES,
            bias="none",
        )
        reranker_model = get_peft_model(reranker_model, lora_config)
        reranker_model.print_trainable_parameters()
    print("[LOAD_PRETRAINED=False] Đã khởi tạo LoRA mới từ base model, sẵn sàng train ở Cell 13/14.")

# ĐÃ SỬA: BAAI/bge-reranker-v2-m3 chấm điểm qua 1 logit hồi quy duy nhất (không còn prompt "yes"/"no"
# kiểu Qwen3-Reranker) -> bỏ toàn bộ PREFIX/SUFFIX/TOKEN_TRUE_ID/TOKEN_FALSE_ID/TASK_INSTRUCTION/
# format_instruction. Input giờ là 1 CẶP [query, document] đưa thẳng vào tokenizer (tokenizer hỗ trợ
# sẵn kiểu input list[[text_a, text_b], ...]), đúng cách dùng chính thức của
# BAAI/bge-reranker-v2-m3.

def process_inputs(pairs, max_length=MAX_LEN):
    """pairs: list[[query, document], ...]. Trả về dict tensor đã chuyển sang DEVICE."""
    inputs = reranker_tokenizer(
        pairs, padding=True, pad_to_multiple_of=8, truncation=True, max_length=max_length, return_tensors="pt",
    )
    return {k: v.to(DEVICE) for k, v in inputs.items()}

@torch.no_grad()
def _forward_batch(pairs):
    """1 lan forward reranker_model tren 1 batch pairs, co retry khi dinh bug cache
    attention-mask cua duong SDPA (RuntimeError 'expanded size ... non-singleton dimension 3').
    Model dung attention mac dinh (sdpa) de tiet kiem VRAM khi train; bug shape hiem khi xay ra
    voi sdpa nhung neu co, retry (roi fallback forward tung phan tu mot) se tu xu ly duoc.
    """
    inputs = process_inputs(pairs)
    try:
        return reranker_model(**inputs).logits.view(-1)
    except RuntimeError as e:
        msg = str(e)
        if "non-singleton dimension" not in msg and "expanded size" not in msg:
            raise
        # Bug cache attention-mask: giai phong cache CUDA roi thu lai NGUYEN batch 1 lan.
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
        try:
            return reranker_model(**inputs).logits.view(-1)
        except RuntimeError:
            # Van loi -> forward TUNG PHAN TU MOT (cham nhung chac chan khong vo shape nua),
            # de khong mat toan bo epoch/refresh dang chay do 1 batch loi.
            outs = []
            for j in range(len(pairs)):
                single = process_inputs([pairs[j]])
                outs.append(reranker_model(**single).logits.view(-1))
            return torch.cat(outs, dim=0)


@torch.no_grad()
def rerank_score(query, docs, batch_size=8):
    # Trả về list điểm liên quan (sigmoid của logit hồi quy) cho từng doc/chunk, KHÔNG tính gradient
    # (dùng khi infer). sigmoid() đưa điểm về [0, 1], giữ tương thích với RERANK_THRESHOLD hiện có
    # (Cell 2) — vốn được thiết kế cho điểm dạng xác suất.
    scores = []
    reranker_model.eval()
    for i in range(0, len(docs), batch_size):
        batch_docs = docs[i:i + batch_size]
        pairs = [[query, d] for d in batch_docs]
        logits = _forward_batch(pairs)  # SUA: bọc forward qua _forward_batch (co retry an toan)
        probs = torch.sigmoid(logits)
        scores.extend(probs.float().cpu().tolist())
    return scores


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 393/393 [00:04<00:00, 98.02it/s] 


[LOAD_PRETRAINED=True] Đã load MERGED model (không cần peft) từ /mnt/mmlab2024nas/trantran/vietnamese-v2-reranker-bge-7list3-merged


In [24]:
# Cell 12 — Dataset / DataLoader cho fine-tuning (GROUP-BASED, phục vụ LISTWISE LOSS)
#
# ĐÃ SỬA (LISTWISE): collate_group_fn giờ trả về THÊM `group_sizes` — số passage của từng group
# trong batch, theo đúng thứ tự đã flatten. Cell 13 cần thông tin này để cắt vector logit phẳng
# thành từng group rồi mới áp softmax cross-entropy trên mỗi group (mẫu số của softmax = đúng tập
# {1 positive + các negative CỦA CÙNG CÂU HỎI}). Với BCE cũ thì không cần biết ranh giới group nên
# chỉ có (inputs, labels).
#
# `labels` VẪN ĐƯỢC TRẢ VỀ (vector [1,0,0,...] khớp 1-1 với thứ tự flatten) vì 2 lý do: (a) dùng cho
# số hạng BCE phụ tuỳ chọn khi LISTWISE_BCE_WEIGHT > 0 (Cell 2), và (b) tiện log/debug. Nếu
# LISTWISE_BCE_WEIGHT = 0 thì labels không tham gia vào loss.
#
# Group có số negative KHÁC NHAU là hợp lệ (hiếm — khi neg_pool bị lọc gần hết): Cell 13 pad phần
# thiếu bằng -inf trước khi softmax nên các vị trí pad có xác suất đúng bằng 0, không làm nhiễu loss.
from torch.utils.data import Dataset, DataLoader

GROUP_SIZE = 1 + NEG_PER_POS   # = 16 passage/group (1 positive + 15 negative)


class RerankGroupDataset(Dataset):
    def __init__(self, groups):
        # groups: list[(question, pos_text, list[neg_text])]
        # ĐÃ THÊM (LISTWISE): loại thẳng các group KHÔNG có negative nào. Với BCE cũ, một group chỉ
        # có positive vẫn sinh gradient (ép logit đó lên cao). Với softmax cross-entropy thì group
        # 1 phần tử luôn cho loss = -log(1) = 0 và gradient = 0 -> chỉ tốn 1 lần forward vô ích.
        self.groups = [g for g in groups if len(g[2]) >= 1]
        n_dropped = len(groups) - len(self.groups)
        if n_dropped:
            print(f"[Listwise] Bỏ {n_dropped} group không có negative nào (loss softmax = 0, "
                  f"không sinh gradient).")

    def __len__(self):
        return len(self.groups)

    def __getitem__(self, idx):
        question, pos_text, neg_texts = self.groups[idx]
        # Vị trí 0 LUÔN là positive — collate_group_fn và listwise_loss (Cell 13, target = 0) đều
        # dựa vào quy ước này.
        return question, [pos_text] + list(neg_texts)


def collate_group_fn(batch):
    """batch: list[(question, [passage_0 (positive), passage_1..N (negative)])].

    Trả về (inputs, labels, group_sizes):
      - inputs: dict tensor đã tokenize toàn bộ cặp [query, passage] đã flatten group-by-group.
      - labels: LongTensor [1,0,...,0, 1,0,...,0, ...] khớp 1-1 với thứ tự flatten (chỉ dùng cho
        số hạng BCE phụ tuỳ chọn).
      - group_sizes: LongTensor số passage của từng group, để Cell 13 cắt lại logit theo group.
    """
    pairs, labels, group_sizes = [], [], []
    for question, passages in batch:
        group_sizes.append(len(passages))
        for j, passage in enumerate(passages):
            pairs.append([question, passage])
            labels.append(1 if j == 0 else 0)
    inputs = process_inputs(pairs)
    return (inputs,
            torch.tensor(labels, dtype=torch.long),
            torch.tensor(group_sizes, dtype=torch.long))


# ---- VRAM/step: batch_size của DataLoader đếm theo GROUP, không phải theo pair ----
# GROUP_BATCH_SIZE = 1: mỗi lần forward xử lý ĐÚNG 1 group (1 positive + NEG_PER_POS negative CỦA
# CÙNG câu hỏi) — cần thiết vì softmax listwise (Cell 13/27) phải thấy TOÀN BỘ logit của group đó
# cùng lúc để tính đúng mẫu số. NEG_PER_POS tăng 7 -> 15 (Cell 2) nên group giờ = 16 passage/forward
# (thay vì 8) -> VRAM/forward tăng theo. Số câu hỏi/optimizer step vẫn được bù bằng GRAD_ACCUM_STEPS
# (Cell 13). Nếu OOM: giảm NEG_PER_POS (Cell 2) hoặc tăng GRAD_ACCUM_STEPS để bù lại batch hiệu dụng.
GROUP_BATCH_SIZE = 2
BATCH_SIZE = GROUP_BATCH_SIZE * GROUP_SIZE   # = 16 pair/forward; giữ tên biến cũ cho các cell khác

train_loader = DataLoader(
    RerankGroupDataset(train_groups), batch_size=GROUP_BATCH_SIZE, shuffle=True,
    collate_fn=collate_group_fn,
)
print(f"Group size = {GROUP_SIZE} passage (1 pos + {NEG_PER_POS} neg) | "
      f"{GROUP_BATCH_SIZE} group/forward = {BATCH_SIZE} pair/forward | loss = listwise softmax CE "
      f"(T={LISTWISE_TEMPERATURE}, bce_weight={LISTWISE_BCE_WEIGHT})")
print("Số batch mỗi epoch:", len(train_loader))


Group size = 11 passage (1 pos + 10 neg) | 2 group/forward = 22 pair/forward | loss = listwise softmax CE (T=1.0, bce_weight=0.0)
Số batch mỗi epoch: 3433


In [ ]:
# Cell 13 — Training loop (LoRA fine-tuning, gradient checkpointing để tiết kiệm thêm VRAM)
# + Resample negative mỗi epoch (tránh học tủ 1 bộ negative cố định)
# + ĐÃ THÊM: mỗi epoch (từ epoch 2) refresh known-positive score rồi tính lại
#   FALSE_NEG_RERANK_THRESHOLD ĐỘNG (percentile p10, làm mượt EMA — xem Cell 10c3) — GIỜ CHỈ CÒN
#   DÙNG LÀM NGƯỠNG DỰ PHÒNG toàn cục (câu hỏi chưa có known-positive riêng ở epoch hiện tại).
#   Ngưỡng THẬT SỰ lọc false-negative của mỗi câu hỏi được tính RIÊNG theo positive của CHÍNH câu
#   hỏi đó (local_false_neg_threshold, Cell 10c3) — vì loss listwise không ép sigmoid(logit) nhất
#   quán GIỮA các câu hỏi, một ngưỡng tuyệt đối chung cho mọi câu hỏi là không công bằng.
# + Eval trên dev sau mỗi epoch, chọn checkpoint theo recall trước, precision sau (không dùng F2)
#   (two_stage_retrieve dùng RERANK_THRESHOLD mặc định 0.99 trong lúc train — sẽ tune lại kỹ hơn
#   ở Cell 19b SAU KHI train xong, không ảnh hưởng tới việc chọn checkpoint ở bước này)
# + Flush gradient còn sót lại cuối mỗi epoch (tránh mất tối đa GRAD_ACCUM_STEPS-1 batch cuối epoch)
#
# LƯU Ý khi dùng LoRA:
#   - Chỉ ~0.5-1% tham số được train -> LR cần cao hơn nhiều so với full fine-tune (1e-5 -> 2e-4).
#   - reranker_model.parameters() vẫn truyền cả model (base bị freeze), optimizer sẽ tự bỏ qua
#     các tham số có requires_grad=False vì chúng không có .grad.
import bitsandbytes as bnb
import torch.nn.functional as F


# ============================================================================================
# HÀM LOSS MỚI — LISTWISE (softmax cross-entropy trong từng group), thay cho pointwise BCE.
#
#     L(q) = -log[ exp(s_pos / T) / ( exp(s_pos / T) + sum_j exp(s_neg_j / T) ) ]
#
# trong đó s = logit của cross-encoder, T = LISTWISE_TEMPERATURE (Cell 2), và mẫu số chạy trên
# ĐÚNG các passage của CÙNG MỘT CÂU HỎI (1 positive + NEG_PER_POS hard-negative, xem Cell 10d/12).
#
# Vì sao hợp với bài toán này hơn BCE:
#   - Đề bài chấm recall/precision trên top-5 XẾP HẠNG, tức chỉ quan tâm THỨ TỰ TƯƠNG ĐỐI trong
#     nội bộ mỗi câu hỏi. BCE lại ép logit về giá trị tuyệt đối 0/1 chung cho mọi câu hỏi — một
#     ràng buộc chặt hơn mức cần thiết, và mâu thuẫn trực tiếp với hard-negative (candidate thật sự
#     khá liên quan nhưng không phải gold) vì BCE bắt model gán cho chúng xác suất ~0.
#   - Với listwise, hard-negative chỉ cần bị xếp THẤP HƠN positive của chính câu hỏi đó; model
#     không bị phạt vì cho nó điểm cao ở thang tuyệt đối. Nhờ vậy pipeline hard-negative mining
#     (TẦNG 1.5, Cell 10c2/10d) không còn "đánh nhau" với hàm loss như trước.
#   - Gradient của softmax CE tự động tập trung vào negative ĐANG bị xếp cao nhất (trọng số
#     softmax lớn nhất) — tức tự động hard-negative weighting ngay trong loss, không cần chỉnh tay.
#
# Xử lý group có kích thước KHÁC NHAU: pad bằng -inf rồi mới softmax -> exp(-inf) = 0, vị trí pad
# không đóng góp vào mẫu số và không nhận gradient. An toàn tuyệt đối vì mỗi group luôn có ít nhất
# 1 phần tử thật ở vị trí 0 (positive).
def listwise_loss(logits, group_sizes, temperature=None, bce_weight=None):
    """logits: tensor 1-D đã flatten group-by-group (positive ở đầu mỗi group).
    group_sizes: tensor/list số passage của từng group, cùng thứ tự với logits.
    Trả về scalar loss đã trung bình hoá theo SỐ GROUP (= số câu hỏi trong batch).
    """
    if temperature is None:
        temperature = LISTWISE_TEMPERATURE
    if bce_weight is None:
        bce_weight = LISTWISE_BCE_WEIGHT

    # float32 cho phần tính loss: logits vào đây đang là bfloat16 (autocast) — bf16 chỉ có ~8 bit
    # mantissa nên logsumexp/softmax dễ mất chính xác, trong khi chi phí ép về fp32 ở đây không đáng
    # kể (vector chỉ vài chục phần tử).
    logits = logits.float()
    sizes = [int(n) for n in group_sizes]

    # Cắt vector phẳng thành từng group rồi pad -inf về cùng độ dài -> ma trận [n_group, max_size].
    parts = torch.split(logits, sizes)
    padded = torch.nn.utils.rnn.pad_sequence(
        parts, batch_first=True, padding_value=float("-inf")
    )

    # Nhãn đúng LUÔN ở vị trí 0 của mỗi group (quy ước của RerankGroupDataset, Cell 12).
    target = torch.zeros(padded.size(0), dtype=torch.long, device=padded.device)
    loss = F.cross_entropy(padded / temperature, target)

    # (Tuỳ chọn) cộng thêm BCE pointwise để giữ tính hiệu chỉnh của sigmoid(logit) — chỉ cần khi
    # muốn lọc theo RERANK_THRESHOLD tuyệt đối. Mặc định LISTWISE_BCE_WEIGHT = 0.0 -> bỏ qua hoàn
    # toàn, không tốn tính toán nào.
    if bce_weight > 0:
        labels = torch.zeros_like(logits)
        offsets = torch.tensor(
            [sum(sizes[:i]) for i in range(len(sizes))], device=logits.device, dtype=torch.long
        )
        labels[offsets] = 1.0
        loss = loss + bce_weight * F.binary_cross_entropy_with_logits(logits, labels)
    return loss


@torch.no_grad()
def listwise_batch_accuracy(logits, group_sizes):
    """Tỉ lệ group mà positive (vị trí 0) đang được xếp CAO NHẤT — chỉ để theo dõi trong tqdm.
    Với listwise, giá trị loss tuyệt đối khó đọc (phụ thuộc số negative), còn accuracy này đọc
    được ngay: 1/(1+NEG_PER_POS) = mức đoán mò, càng gần 1.0 càng tốt."""
    sizes = [int(n) for n in group_sizes]
    parts = torch.split(logits.float(), sizes)
    return float(np.mean([1.0 if int(p.argmax()) == 0 else 0.0 for p in parts]))


EPOCHS = 3
# ĐÃ SỬA (LISTWISE): hạ LR 2e-4 -> 1e-4 (LoRA). Không phải vì batch đổi, mà vì ĐỘ LỚN GRADIENT đổi:
# BCE cũ lấy trung bình trên 12 CẶP/forward, nên gradient đóng góp của mỗi câu hỏi bị chia nhỏ;
# listwise CE lấy trung bình trên số GROUP (= số câu hỏi) trong batch, mỗi group đóng góp nguyên
# một số hạng -log(softmax) -> gradient trên mỗi tham số lớn hơn đáng kể ở cùng LR. Giữ 2e-4 dễ gây
# dao động/phân kỳ ở những step đầu. Nếu loss giảm quá chậm, có thể nâng dần về 1.5e-4.
LR = 1e-4 if USE_LORA else 5e-6
# ĐÃ SỬA theo dataloader GROUP-BASED (Cell 12): BATCH_SIZE giờ = GROUP_BATCH_SIZE * GROUP_SIZE
# = 1 * GROUP_SIZE = 16 pair/forward (NEG_PER_POS 7 -> 15 làm group 8 -> 16 passage, Cell 2/12).
# GRAD_ACCUM_STEPS giữ nguyên = 8 -> vẫn 8 CÂU HỎI/optimizer step như trước (mỗi group = 1 câu hỏi
# vì GROUP_BATCH_SIZE = 1), chỉ khác là mỗi câu hỏi giờ đóng góp 16 pair thay vì 8 -> batch hiệu
# dụng tăng từ 64 lên 8 * 16 = 128 pair/step. Không cần chỉnh lại LR vì listwise CE lấy trung bình
# theo SỐ GROUP (= số câu hỏi/step), không theo số pair. Nếu OOM ở group 16 passage/forward: giảm
# NEG_PER_POS (Cell 2) hoặc hạ GROUP_BATCH_SIZE (Cell 12, đã ở mức sàn 1 — hết chỗ giảm) — trong
# trường hợp đó cân nhắc tăng GRAD_ACCUM_STEPS để bù lại batch hiệu dụng.
GRAD_ACCUM_STEPS = 4

def apply_rerank_filter(ranked, top_k_final=TOP_K_FINAL,
                         rerank_top_n=None, rerank_threshold=None, rerank_fallback_k=None):
    """Lọc kết quả sau rerank theo mục 2.2.2: xét top-N (mặc định 10) document điểm reranker cao
    nhất, chỉ giữ candidate có điểm > threshold (mặc định 0.99). Nếu không candidate nào vượt
    ngưỡng, fallback lấy top-K theo điểm (mặc định 2) để tránh trả về tập rỗng. Luôn cắt thêm
    top_k_final ở cuối để không bao giờ vượt quá ràng buộc cứng của đề bài (tối đa 5).
    ranked: list[(doc_id, score)] đã sort giảm dần theo điểm reranker.
    Đọc rerank_top_n/threshold/fallback_k từ config global (RERANK_*) nếu không truyền vào, để
    Cell 19b (tune threshold) có thể chỉnh lại giá trị mặc định mà không cần định nghĩa lại hàm.

    TẠM THỜI TẮT lọc theo threshold/fallback (xem comment bên dưới): ưu tiên đơn giản hoá, chỉ cắt
    tối đa top_k_final document_id từ top-N ranked theo điểm reranker, không loại candidate nào
    theo ngưỡng nữa. Muốn bật lại threshold thì uncomment khối code phía dưới."""
    if rerank_top_n is None:
        rerank_top_n = RERANK_TOP_N
    if rerank_threshold is None:
        rerank_threshold = RERANK_THRESHOLD
    if rerank_fallback_k is None:
        rerank_fallback_k = RERANK_FALLBACK_K

    top_n_ranked = ranked[:rerank_top_n]

    # ---- TẠM THỜI COMMENT: lọc theo threshold (mục 2.2.2) ----
    # filtered = [d for d, sc in top_n_ranked if sc > rerank_threshold]
    # if not filtered:
    #     filtered = [d for d, _ in top_n_ranked[:rerank_fallback_k]]

    # ---- Ưu tiên hiện tại: bỏ qua threshold/fallback, chỉ cắt tối đa top_k_final ----
    filtered = [d for d, _ in top_n_ranked]
    return filtered[:top_k_final]


# Cache Stage-1 (retrieve + chọn short-chunk) cho eval lặp lại trên CÙNG dev_samples qua
# nhiều epoch: phần này KHÔNG phụ thuộc trọng số reranker đang train (chỉ phụ thuộc BM25 index +
# câu hỏi), nên kết quả giống hệt nhau ở mọi epoch -> tính 1 lần ở epoch đầu tiên, từ epoch 2 trở đi
# chỉ còn phải chạy lại rerank_score() (forward pass GPU, phần DUY NHẤT thực sự cần recompute vì
# dùng trọng số model MỚI). Giúp eval cuối mỗi epoch nhanh hơn hẳn (bỏ được phần BM25 + chọn chunk
# lặp lại vô ích).
_eval_stage1_cache = {}  # (question, stage1_name, top_k_stage1, chunks_per_doc, lambda) -> (flat_chunk_docs, flat_short_texts)

def get_reranked_candidates(question, stage1_fn=hybrid_retrieve, top_k_stage1=TOP_K_STAGE1,
                             use_stage1_cache=False, chunks_per_doc=None, hybrid_lam=None):
    """Stage-1 (stage1_fn) -> Stage-2 rerank (BAAI/bge-reranker-v2-m3), trả về list[(doc_id, score)] đã sort
    giảm dần theo điểm reranker, CHƯA áp threshold/fallback/top_k_final (đó là việc của
    apply_rerank_filter). Tách riêng phần này ra khỏi two_stage_retrieve() để có thể CACHE kết quả
    rerank — bước tốn GPU nhất trong pipeline — và tái sử dụng khi chỉ cần thử nhiều threshold/top_n/
    fallback_k khác nhau (xem Cell 19b: tune threshold) mà không phải chạy lại forward pass reranker
    nhiều lần cho cùng 1 câu hỏi.

    ĐÃ SỬA: representative short-chunk được chọn bằng CÙNG scoring với Stage-1. Với pipeline
    mặc định (hybrid_retrieve), score là hybrid = lambda * BM25_norm + (1-lambda) * Dense_norm.
    Vì vậy training mining, Stage-1 và Stage-2 chunk selection dùng cùng tín hiệu hybrid.

    use_stage1_cache=True: cache (flat_chunk_docs, flat_texts) theo (question, top_k_stage1,
    chunks_per_doc) trong _eval_stage1_cache — dùng khi eval LẶP LẠI cùng stage1_fn/top_k_stage1/
    tập câu hỏi qua nhiều lần gọi (vd. sau mỗi epoch trong training loop): Stage-1 chỉ cần tính
    đúng 1 lần ở lần eval đầu tiên.

    chunks_per_doc: số SHORT-chunk tối đa lấy ra cho MỖI document ứng viên. None -> dùng
    MAX_CHUNKS_PER_DOC_FOR_RERANK. Các short-chunk được đưa THẲNG vào reranker, không expand.
    Giảm số này khi eval để bớt số chunk phải rerank mỗi document."""
    if chunks_per_doc is None:
        chunks_per_doc = MAX_CHUNKS_PER_DOC_FOR_RERANK
    stage1_name = getattr(stage1_fn, "__name__", str(stage1_fn))
    if stage1_fn is hybrid_retrieve:
        if hybrid_lam is None:
            hybrid_lam = LAMBDA_HYBRID
        cache_key = (question, stage1_name, top_k_stage1, chunks_per_doc, float(hybrid_lam))
    else:
        cache_key = (question, stage1_name, top_k_stage1, chunks_per_doc, None)
    if use_stage1_cache and cache_key in _eval_stage1_cache:
        flat_chunk_docs, flat_texts = _eval_stage1_cache[cache_key]
    else:
        candidates = [d for d, _ in stage1_fn(question, top_k=top_k_stage1)]
        if len(candidates) == 0:
            if use_stage1_cache:
                _eval_stage1_cache[cache_key] = ([], [])
            return []

        # Chọn SHORT-chunk bằng cùng scoring với Stage-1.
        if stage1_fn is hybrid_retrieve:
            chunk_scores = hybrid_scores_all(question, lam=hybrid_lam)
        elif stage1_fn is dense_retrieve:
            chunk_scores = dense_scores_all(question)
        else:
            chunk_scores = bm25_scores_all(question)

        flat_chunk_docs = []   # doc_id tương ứng với từng short-chunk
        flat_texts = []       # short-chunk text đưa THẲNG vào reranker
        for d in candidates:
            for cid, ctext in select_relevant_chunks(d, chunk_scores, top_n=chunks_per_doc):
                flat_chunk_docs.append(d)
                flat_texts.append(ctext)

        if use_stage1_cache:
            _eval_stage1_cache[cache_key] = (flat_chunk_docs, flat_texts)

    if not flat_texts:
        return []

    scores = rerank_score(question, flat_texts)

    doc_best_score = {}
    for d, sc in zip(flat_chunk_docs, scores):
        if d not in doc_best_score or sc > doc_best_score[d]:
            doc_best_score[d] = sc

    return sorted(doc_best_score.items(), key=lambda x: x[1], reverse=True)


def two_stage_retrieve(question, stage1_fn=hybrid_retrieve, top_k_stage1=TOP_K_STAGE1, top_k_final=TOP_K_FINAL,
                        rerank_top_n=None, rerank_threshold=None, rerank_fallback_k=None,
                        use_stage1_cache=False, chunks_per_doc=None, hybrid_lam=None):
    """Stage-1: hybrid_retrieve (hoặc bm25_retrieve/dense_retrieve) -> top_k_stage1 candidate doc_id.
    Stage-2: với mỗi candidate document, chọn tối đa chunks_per_doc SHORT-chunk liên quan nhất
    theo cùng scoring với Stage-1 — mặc định MAX_CHUNKS_PER_DOC_FOR_RERANK short-chunk/document.
    KHÔNG expand, KHÔNG ghép chunk lân cận; rerank TẤT CẢ các short-chunk đó cùng lúc (1 batch lớn), lấy điểm CAO NHẤT trong các chunk của mỗi văn
    bản làm điểm đại diện cho văn bản đó, rồi ÁP DỤNG BỘ LỌC THRESHOLD (mục 2.2.2): chỉ giữ lại các
    candidate "tự tin cao" trong top-10 (điểm > 0.99), fallback top-2 nếu không có candidate nào đạt
    ngưỡng, để vừa tăng precision vừa đảm bảo không trả về tập rỗng.
    (Wrapper mỏng quanh get_reranked_candidates() + apply_rerank_filter(), giữ nguyên hành vi/API
    như trước — các cell downstream (eval, ablation, submission) không cần sửa gì.)

    use_stage1_cache=True: truyền thẳng xuống get_reranked_candidates() để cache phần Stage-1 (BM25
    + chọn chunk) qua nhiều lần gọi liên tiếp trên cùng tập câu hỏi (xem eval cuối mỗi epoch bên dưới).
    chunks_per_doc: xem docstring get_reranked_candidates — None giữ nguyên hành vi cũ."""
    ranked = get_reranked_candidates(question, stage1_fn=stage1_fn, top_k_stage1=top_k_stage1,
                                      use_stage1_cache=use_stage1_cache, chunks_per_doc=chunks_per_doc,
                                      hybrid_lam=hybrid_lam)
    return apply_rerank_filter(
        ranked, top_k_final=top_k_final,
        rerank_top_n=rerank_top_n, rerank_threshold=rerank_threshold, rerank_fallback_k=rerank_fallback_k,
    )

# ============================================================================================
# Stage-2 dùng chung đúng một logic cho training/eval/inference: chọn short-chunk và rerank trực tiếp.
# Không còn bất kỳ nhánh EVAL-ONLY hay logic long-chunk nào.
# ============================================================================================


def rebuild_train_loader(epoch_seed):
    # ĐÃ SỬA: dùng lại fit_candidate_pools đã mine/cache sẵn ở Cell 10c (TẦNG 1 - HYBRID, ĐẮT, chỉ
    # chạy 1 lần) -> mỗi epoch chỉ còn gọi TẦNG 1.5 (refresh_pool_scores, RẺ — xem lời gọi ngay
    # TRƯỚC rebuild_train_loader() trong vòng lặp epoch bên dưới) rồi TẦNG 2
    # (build_training_pairs_from_pool, chọn negative theo rank TẦNG 1.5, xem Cell 10d), KHÔNG tính
    # lại bm25.get_scores()/dense cho fit_samples nữa. Nhờ vậy dù restart kernel, chỉ cần
    # mine_candidate_pools() load lại cache TẦNG 1 một lần là đủ cho toàn bộ quá trình train — TẦNG
    # 1.5 sẽ tự "làm nóng lại" difficulty của candidate theo checkpoint mới nhất mỗi epoch.
    random.seed(SEED + epoch_seed)
    # ĐÃ SỬA: build GROUP (1 positive + NEG_PER_POS negative cùng câu hỏi) thay vì cặp phẳng, và
    # shuffle ở CẤP GROUP -> giữ nguyên tính chất "mỗi epoch một bộ negative khác", đồng thời đảm
    # bảo positive/negative của cùng 1 câu hỏi luôn nằm chung 1 lần forward (xem Cell 12).
    groups = build_training_groups_from_pool(fit_candidate_pools, fit_samples, neg_per_pos=NEG_PER_POS,
                                              desc=f"Resample negative theo group (epoch seed={epoch_seed})")
    loader = DataLoader(RerankGroupDataset(groups), batch_size=GROUP_BATCH_SIZE, shuffle=True,
                        collate_fn=collate_group_fn)
    return loader

# --- Bật gradient checkpointing để giảm activation memory (vẫn hữu ích dù dùng LoRA) ---
reranker_model.gradient_checkpointing_enable()
reranker_model.enable_input_require_grads()  # cần thiết khi dùng checkpointing (kể cả với LoRA)
# ĐÃ SỬA: bỏ dòng reranker_model.config.use_cache = False — chỉ áp dụng cho causal LM (Qwen3-Reranker
# cũ dùng KV-cache khi generate); BAAI/bge-reranker-v2-m3 là encoder (sequence-classification), không có
# khái niệm use_cache nên không cần dòng này nữa.

# --- Chỉ train các tham số LoRA (requires_grad=True), base model đã bị freeze bởi get_peft_model ---
trainable_params = [p for p in reranker_model.parameters() if p.requires_grad]
n_trainable = sum(p.numel() for p in trainable_params)
n_total = sum(p.numel() for p in reranker_model.parameters())
print(f"Training {n_trainable:,} / {n_total:,} params ({100*n_trainable/n_total:.2f}%)")

optimizer = bnb.optim.AdamW8bit(trainable_params, lr=LR, weight_decay=0.01)

import math
# ceil (khong phai //): khop voi vong lap training, vi cuoi moi epoch con 1 step flush gradient du neu len(train_loader) % GRAD_ACCUM_STEPS != 0.
steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
num_training_steps = steps_per_epoch * EPOCHS
# Warmup tuyen tinh 10% so step dau (LR tang 0 -> LR), sau do giam tuyen tinh (linear, khong phai cosine) ve 0.
from transformers import get_linear_schedule_with_warmup
WARMUP_RATIO = 0.1
num_warmup_steps = int(WARMUP_RATIO * num_training_steps)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=num_warmup_steps, num_training_steps=max(1, num_training_steps)
)
print(f"Scheduler: linear warmup {num_warmup_steps}/{num_training_steps} step (ratio={WARMUP_RATIO}), sau do giam tuyen tinh ve 0")

best_recall = -1.0
best_precision = -1.0
best_epoch = -1     # epoch (1-indexed) ứng với checkpoint đã lưu ở BEST_DIR — dùng để in/kiểm tra
                    # lại sau này, tránh mù mờ không rõ BEST_DIR đang tương ứng epoch nào.
global_step = 0

for epoch in range(EPOCHS):
    if epoch == 0:
        # Epoch đầu tiên: dùng train_loader đã build sẵn ở Cell 13 (negative epoch 1 fallback về
        # random.sample() vì TẦNG 1.5 chưa refresh lần nào — xem Cell 10d).
        epoch_loader = train_loader
    else:
        # TẦNG 1.5 (MỚI): rerank lại toàn bộ neg_pool bằng checkpoint VỪA fine-tune xong ở epoch
        # trước — RẺ (không quét lại corpus/BM25/dense), rồi mới TẦNG 2 resample negative theo rank
        # mới đó (hard-negative "động", bám theo model đang train thay vì rank BM25/dense tĩnh ban đầu).
        torch.cuda.empty_cache()
        refresh_pool_scores(fit_candidate_pools, samples=fit_samples,
                             desc=f"Refresh pool scores TẦNG 1.5 (trước epoch {epoch + 1})")
        # ĐÃ THÊM: refresh điểm reranker hiện tại trên known-positive (Cell 10c3) rồi tính lại
        # FALSE_NEG_RERANK_THRESHOLD ĐỘNG theo percentile thấp (p10) của các điểm đó. ĐÃ SỬA: giá
        # trị này GIỜ CHỈ còn dùng làm NGƯỠNG DỰ PHÒNG toàn cục (fallback_thr) cho những câu hỏi
        # CHƯA có known-positive score riêng ở epoch hiện tại — ngưỡng THẬT SỰ áp dụng cho từng câu
        # hỏi được tính RIÊNG bởi local_false_neg_threshold() (gọi bên trong _suspected_false_
        # negatives, Cell 10d) ngay tại thời điểm build training pairs, không cần ghi đè gì thêm ở
        # đây. Rẻ (chỉ 1-3 forward pass/câu hỏi), an toàn để gọi mỗi epoch ngay sau
        # refresh_pool_scores(). None (vd. chưa đủ known-positive) -> tự động GIỮ NGUYÊN threshold
        # dự phòng hiện tại, không ghi đè.
        refresh_pool_pos_scores(fit_candidate_pools, samples=fit_samples,
                                 desc=f"Refresh positive scores TẦNG 1.5 (trước epoch {epoch + 1})")
        dynamic_thr = compute_dynamic_false_neg_threshold(low_percentile=10)
        if dynamic_thr is not None:
            FALSE_NEG_RERANK_THRESHOLD = dynamic_thr
            print(f"[Epoch {epoch + 1}] Dynamic FALSE_NEG_RERANK_THRESHOLD (EMA, DỰ PHÒNG toàn cục) = "
                  f"{FALSE_NEG_RERANK_THRESHOLD:.4f} (p10 của "
                  f"{sum(len(v) for v in _pool_pos_scores.values())} known-positive scores) — "
                  f"đa số câu hỏi sẽ dùng threshold RIÊNG (local_false_neg_threshold, Cell 10c3), "
                  f"giá trị này chỉ áp dụng cho câu hỏi thiếu known-positive riêng ở epoch này.")
        epoch_loader = rebuild_train_loader(epoch_seed=epoch)

    reranker_model.train()
    running_loss = 0.0
    running_acc = 0.0
    optimizer.zero_grad()
    pbar = tqdm(epoch_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    # ĐÃ SỬA (LISTWISE): collate_group_fn (Cell 12) trả về THÊM group_sizes — ranh giới giữa các
    # group trong vector logit đã flatten. Bắt buộc phải có với listwise vì softmax chỉ được phép
    # chạy TRONG từng group (1 positive + negative của CÙNG câu hỏi), không phải trên cả batch.
    for step, (inputs, labels, group_sizes) in enumerate(pbar):
        # BAAI/bge-reranker-v2-m3 (sequence-classification) trả về 1 logit duy nhất cho mỗi cặp
        # [query, passage]. logits.view(-1) trải phẳng theo đúng thứ tự pairs mà collate_group_fn
        # đã sinh (group-by-group, positive ở vị trí ĐẦU mỗi group) -> listwise_loss chỉ cần cắt
        # lại theo group_sizes rồi áp cross-entropy với target = 0 cho mọi group.
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            logits = reranker_model(**inputs).logits.view(-1)
        # Tính loss NGOÀI autocast: listwise_loss tự ép fp32 để logsumexp/softmax không mất chính
        # xác ở bf16 (forward nặng vẫn chạy bf16 như cũ, nên không tốn thêm VRAM đáng kể).
        loss = listwise_loss(logits, group_sizes)

        (loss / GRAD_ACCUM_STEPS).backward()

        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1

        running_loss += loss.item()
        # acc = tỉ lệ group mà positive đang đứng đầu; mức đoán mò = 1/GROUP_SIZE
        # (= 1/16 = 0.0625 với NEG_PER_POS = 15). Đây là chỉ số dễ đọc hơn giá trị loss tuyệt đối.
        running_acc += listwise_batch_accuracy(logits.detach(), group_sizes)
        if step % 20 == 0:
            pbar.set_postfix(loss=running_loss / (step + 1), acc=running_acc / (step + 1))

    if (step + 1) % GRAD_ACCUM_STEPS != 0:
        torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        global_step += 1

    reranker_model.eval()
    # Eval chỉ giảm số SHORT-chunk/document nếu muốn; inference thật mặc định vẫn dùng 3 short-chunk/document.
    # Short-chunk được đưa thẳng vào reranker, không expand.
    EVAL_CHUNKS_PER_DOC = 1
    epoch_metrics = evaluate(
        lambda q: two_stage_retrieve(q, stage1_fn=hybrid_retrieve, top_k_stage1=TOP_K_STAGE1, top_k_final=TOP_K_FINAL,
                                      use_stage1_cache=True, chunks_per_doc=EVAL_CHUNKS_PER_DOC),
        dev_samples, top_k=TOP_K_FINAL, desc=f"eval dev (epoch {epoch+1})"
    )
    epoch_recall = epoch_metrics["recall"]
    epoch_precision = epoch_metrics["precision"]
    print(f"Epoch {epoch+1}: listwise_loss(avg)={running_loss/len(epoch_loader):.4f} "
          f"top1_acc={running_acc/len(epoch_loader):.4f} (đoán mò={1.0/GROUP_SIZE:.3f}) | "
          f"dev recall={epoch_recall:.4f} precision={epoch_precision:.4f}")

    is_better = (epoch_recall, epoch_precision) > (best_recall, best_precision)
    if is_better:
        best_recall = epoch_recall
        best_precision = epoch_precision
        best_epoch = epoch + 1
        # Với LoRA, save_pretrained chỉ lưu adapter (vài chục MB), không lưu lại base model
        reranker_model.save_pretrained(BEST_DIR)
        reranker_tokenizer.save_pretrained(BEST_DIR)
        print(f"  -> recall cải thiện (hoặc recall bằng nhưng precision tốt hơn), "
              f"đã lưu adapter tốt nhất tại {BEST_DIR} (epoch={best_epoch}, recall={best_recall:.4f}, "
              f"precision={best_precision:.4f})")

print(f"Fine-tuning xong. Best dev recall = {best_recall} (epoch {best_epoch}) | "
      f"precision tương ứng = {best_precision} | adapter tốt nhất tại: {BEST_DIR}")
print(f"LƯU Ý: biến `reranker_model` trong bộ nhớ hiện đang là trọng số EPOCH CUỐI ({EPOCHS}), "
      f"KHÔNG PHẢI epoch tốt nhất ({best_epoch}) — Cell 15 ngay sau đây sẽ tự nạp lại đúng "
      f"checkpoint tốt nhất từ {BEST_DIR} trước khi lưu/merge, không cần bạn làm gì thêm ở đây.")


In [ ]:
# Cell 14 — Nạp lại đúng checkpoint TỐT NHẤT (BEST_DIR) rồi lưu/merge, đảm bảo FT_RERANKER_DIR
# và MERGED_DIR luôn khớp với epoch có dev recall cao nhất, KHÔNG PHẢI epoch cuối cùng của vòng lặp.
#
# LÝ DO BẮT BUỘC PHẢI NẠP LẠI: sau khi Cell 14 chạy xong toàn bộ EPOCHS, biến `reranker_model` trong
# bộ nhớ đang mang trọng số của EPOCH CUỐI CÙNG — vì trong training loop, việc "epoch nào tốt hơn"
# (biến is_better) chỉ quyết định có GHI FILE ra BEST_DIR hay không, chứ không hề đổi lại trọng số
# đang có trong biến `reranker_model`. Nếu bỏ qua bước nạp lại này, cell save/merge bên dưới sẽ lưu
# nhầm epoch cuối (có thể tệ hơn best) vào FT_RERANKER_DIR/MERGED_DIR — đây chính là lỗi gặp phải
# trước đó khi Cell 18 cho ra recall thấp hơn hẳn "Best dev recall" đã in ra ở Cell 14.
from peft import PeftModel
import gc

print(f"Đang nạp lại checkpoint TỐT NHẤT từ {BEST_DIR} (epoch={best_epoch}, recall={best_recall:.4f}, "
      f"precision={best_precision:.4f}) vào reranker_model, thay cho trọng số epoch cuối ({EPOCHS}) "
      f"đang có trong bộ nhớ...")

# ---- Dọn dẹp triệt để trước khi load lại, tránh VRAM còn sót do optimizer/trainable_params (Cell 14)
# vẫn giữ tham chiếu tới các tensor LoRA adapter cũ (nhỏ, nhưng dọn cho chắc, đặc biệt nếu VRAM sát
# giới hạn). `globals().get(...)` để không lỗi NameError nếu biến chưa từng tồn tại (vd. chạy lại
# Cell 15 nhiều lần trong cùng session). ----
for _var in ["reranker_model", "optimizer", "trainable_params", "scheduler"]:
    if _var in globals():
        del globals()[_var]
gc.collect()
torch.cuda.empty_cache() if DEVICE == "cuda" else None

reranker_model = AutoModelForSequenceClassification.from_pretrained(
    RERANKER_NAME, torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32
).to(DEVICE)

if USE_LORA:
    # Trường hợp thường gặp (mặc định notebook): BEST_DIR chỉ chứa adapter LoRA -> gắn vào base model.
    reranker_model = PeftModel.from_pretrained(reranker_model, BEST_DIR).to(DEVICE)
else:
    # Trường hợp full fine-tune (USE_LORA=False): BEST_DIR chứa TOÀN BỘ trọng số model (không phải
    # adapter) -> load thẳng, không qua PeftModel (PeftModel.from_pretrained sẽ lỗi vì BEST_DIR
    # không có adapter_config.json trong trường hợp này).
    del reranker_model
    torch.cuda.empty_cache() if DEVICE == "cuda" else None
    reranker_model = AutoModelForSequenceClassification.from_pretrained(
        BEST_DIR, torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32
    ).to(DEVICE)

reranker_model.eval()
print(f"Đã nạp xong checkpoint tốt nhất (epoch {best_epoch}) — từ đây trở đi, mọi cell dùng "
      f"`reranker_model`/`rerank_score()` (Cell 18 eval, Cell 20/21 submission, ...) đều dùng ĐÚNG "
      f"checkpoint tốt nhất, không còn bị lệch sang epoch cuối cùng nữa.")

# ---- Lưu lại adapter LoRA (giờ đã là adapter TỐT NHẤT, không phải epoch cuối) ----
reranker_model.save_pretrained(FT_RERANKER_DIR)   # chỉ lưu adapter LoRA (vài chục MB) nếu USE_LORA=True
reranker_tokenizer.save_pretrained(FT_RERANKER_DIR)
print("Đã lưu adapter/model fine-tune (best checkpoint) tại:", FT_RERANKER_DIR)

# ---- Tuỳ chọn: merge LoRA vào base model để có 1 model độc lập, inference nhanh như bản gốc ----
MERGE_LORA = True
if USE_LORA and MERGE_LORA:
    merged_model = reranker_model.merge_and_unload()
    merged_model.save_pretrained(MERGED_DIR)
    reranker_tokenizer.save_pretrained(MERGED_DIR)
    print("Đã merge LoRA vào base model và lưu tại:", MERGED_DIR)
    print("-> Dùng AutoModelForSequenceClassification.from_pretrained(MERGED_DIR) để load lại, không cần cài peft.")

## 6. Pipeline 2 tầng: Stage-1 (candidates) → Stage-2 (rerank + lọc) → top-5

Stage-1 mặc định dùng **hybrid_retrieve** (BM25 + Vietnamese_Embedding_v2 (AITeamVN), mục 2.2.1) để lấy
top-k ứng viên. Stage-2 dùng **BAAI/bge-reranker-v2-m3** (đã fine-tune LoRA, không đổi) để rerank,
trả về điểm liên quan CE_θ(q, a) ∈ [0, 1] cho từng candidate.

**Lọc theo threshold (mục 2.2.2).** Sau khi rerank, top-k thô của BM25/dense có thể vẫn còn
candidate không liên quan. Để tăng precision, ta áp dụng lọc: chỉ xét **top-10** candidate
điểm reranker cao nhất, giữ lại những candidate có điểm **> 0.99**. Nếu không candidate nào
vượt ngưỡng này, fallback lấy **top-2** theo điểm reranker để tránh trả về tập rỗng. Ngưỡng
0.99, top-10 và fallback top-2 được **tune trên tập validation (dev)** — xem Cell 19b. Kết quả
sau lọc luôn được cắt thêm ở `top_k_final` để đảm bảo không bao giờ vượt quá 5 document_id
(ràng buộc cứng của đề bài). Chiến lược này giúp tăng precision đáng kể chỉ với chi phí giảm
nhẹ recall, cải thiện F2 tổng thể.


In [27]:
# # Cell 18 — Đánh giá pipeline 2 tầng (Hybrid stage-1 + fine-tuned reranker) trên tập dev (offline)
# two_stage_metrics = evaluate(
#     lambda q: two_stage_retrieve(q, stage1_fn=hybrid_retrieve, top_k_stage1=TOP_K_STAGE1, top_k_final=TOP_K_FINAL),
#     dev_samples, top_k=TOP_K_FINAL, desc="2-stage Hybrid(BM25+Dense) + bge-reranker-v2-m3-FT (dev)"
# )

# print("2-stage (fine-tuned):", two_stage_metrics)


In [28]:
# # Cell 18b — Tune bộ lọc Stage-2 (threshold, top-10, fallback top-K) trên tập VALIDATION (dev)
# # Cố định stage-1 = hybrid_retrieve (đã fix lambda ở Cell 8c) + reranker đã fine-tune, chỉ grid
# # search THRESHOLD (giữ rerank_top_n=10, fallback_k=2 cố định theo đúng mục 2.2.2).
# #
# # ĐÃ SỬA tiêu chí chọn threshold: KHÔNG dùng F2-score nữa. Chọn theo ĐÚNG tiêu chí chấm điểm của đề
# # bài — Recall TRƯỚC, Precision SAU (giống hệt cách chọn checkpoint reranker ở Cell 14): so sánh
# # recall trung bình giữa các threshold trước, threshold nào recall cao hơn thắng; nếu recall BẰNG
# # NHAU thì mới so sánh precision để chọn threshold có precision cao hơn. Dùng tuple (recall,
# # precision) so sánh theo thứ tự từ điển (lexicographic) để thể hiện đúng thứ tự ưu tiên này.
# #
# # TỐI ƯU: apply_rerank_filter() (Cell 14) chỉ là hậu xử lý (post-hoc) trên điểm đã rerank — không
# # phụ thuộc gì vào stage-1/GPU. Bản gốc gọi two_stage_retrieve() (chạy lại CẢ stage-1 lẫn forward
# # pass reranker trên GPU) cho MỖI threshold trong THRESHOLD_GRID, tức lặp lại bước tốn GPU nhất của
# # pipeline 8 LẦN/câu hỏi một cách dư thừa. Ở đây ta rerank 1 LẦN/câu hỏi bằng get_reranked_candidates()
# # (Cell 14), cache lại danh sách đã sort theo điểm reranker, rồi mỗi threshold chỉ cần gọi
# # apply_rerank_filter() (rẻ, chạy trên CPU, không đụng GPU/stage-1 nữa).
# #
# # LƯU Ý: hiện tại phần lọc threshold bên trong apply_rerank_filter() đang bị COMMENT (ưu tiên chỉ cắt
# # top_k_final, xem Cell 14) -> threshold_search_results bên dưới sẽ cho kết quả GIỐNG NHAU ở mọi
# # threshold cho tới khi bạn bật lại phần lọc đó. Cell này vẫn hữu ích để tune lại threshold khi cần,
# # nhưng đừng ngạc nhiên nếu các dòng in ra hiện tại trông "phẳng" (không đổi theo threshold).

# # ---- Precompute rerank 1 LẦN cho toàn bộ dev_samples (bước tốn GPU nhất), cache lại rồi mới quét
# #      threshold — xem ghi chú TỐI ƯU ở trên. ----
# _dev_ranked_candidates = [
#     get_reranked_candidates(s["question"], stage1_fn=hybrid_retrieve, top_k_stage1=TOP_K_STAGE1)
#     for s in tqdm(dev_samples, desc="Precompute reranked candidates (dev, dùng chung cho mọi threshold)")
# ]


# def pipeline_metrics_at_threshold(threshold, ranked_lists, samples=dev_samples,
#                                    top_n=RERANK_TOP_N, fallback_k=RERANK_FALLBACK_K):
#     """Tính recall/precision trung bình trên dev_samples ứng với 1 threshold, bằng cách áp
#     apply_rerank_filter() (CPU, rẻ) lên các danh sách đã rerank sẵn (ranked_lists) — không recompute
#     stage-1/rerank. KHÔNG còn tính f2 — lựa chọn threshold dựa trên (recall, precision), không dựa
#     trên f2 nữa (xem ghi chú ĐÃ SỬA ở đầu cell)."""
#     recalls, precisions = [], []
#     for s, ranked in zip(samples, ranked_lists):
#         pred = apply_rerank_filter(
#             ranked, top_k_final=TOP_K_FINAL,
#             rerank_top_n=top_n, rerank_threshold=threshold, rerank_fallback_k=fallback_k,
#         )
#         r, p = recall_precision(pred, s["answers"])
#         recalls.append(r); precisions.append(p)
#     recall, precision = float(np.mean(recalls)), float(np.mean(precisions))
#     return {"recall": recall, "precision": precision}

# THRESHOLD_GRID = [0.5, 0.7, 0.9, 0.95, 0.97, 0.99, 0.995, 0.999]
# threshold_search_results = {
#     t: pipeline_metrics_at_threshold(t, _dev_ranked_candidates) for t in tqdm(THRESHOLD_GRID, desc="Tuning threshold")
# }

# for t, m in sorted(threshold_search_results.items()):
#     print(f"  threshold={t:.3f} -> recall={m['recall']:.4f} precision={m['precision']:.4f}")

# # ĐÃ SỬA: chọn threshold theo (recall, precision) — recall trước, precision sau (không dùng F2).
# # max() trên tuple so sánh recall trước; chỉ khi recall bằng nhau mới xét tới precision.
# RERANK_THRESHOLD = max(
#     threshold_search_results,
#     key=lambda t: (threshold_search_results[t]["recall"], threshold_search_results[t]["precision"]),
# )
# print(f"\n-> Chọn threshold={RERANK_THRESHOLD:.3f} "
#       f"(recall={threshold_search_results[RERANK_THRESHOLD]['recall']:.4f}, "
#       f"precision={threshold_search_results[RERANK_THRESHOLD]['precision']:.4f}) cho bộ lọc Stage-2.")


## 7. Sinh file `submission.json` cho Public/Private Test

Đúng định dạng đề bài yêu cầu:
```json
{"<qid>": {"answer": ["177504", "740"]}}
```
Luôn cắt tối đa 5 document_id (nếu vượt quá, đề bài chấm 0 điểm cho câu đó).


In [ ]:
# Cell 19 — Sinh submission cho một tệp test bất kỳ (public hoặc private)
def generate_submission(test_path, stage1_fn=hybrid_retrieve, top_k_stage1=TOP_K_STAGE1, top_k_final=TOP_K_FINAL):
    test_raw = load_json(test_path)
    submission = {}
    for qid, v in tqdm(test_raw.items(), desc=f"Predicting {os.path.basename(test_path)}"):
        question = v["question"]
        pred_ids = two_stage_retrieve(
            question, stage1_fn=stage1_fn, top_k_stage1=top_k_stage1, top_k_final=top_k_final
        )
        assert len(pred_ids) <= 5, "Không được vượt quá 5 document_id!"
        submission[str(qid)] = {"answer": pred_ids}
    return submission

def save_submission(submission, out_dir="outputs"):
    json_path = os.path.join(out_dir, "submission_7list33.json")
    zip_path = os.path.join(out_dir, "submission_7list33.zip")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(submission, f, ensure_ascii=False, indent=2)
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(json_path, arcname="submission_7list33.json")
    print("Đã lưu:", json_path, "và", zip_path)


In [30]:
# Cell 20 — Chạy dự đoán cho Private Test (đổi sang PRIVATE_TEST_PATH khi vào giai đoạn Private)
if os.path.exists(PRIVATE_TEST_PATH):
    private_submission = generate_submission(PRIVATE_TEST_PATH)
    save_submission(private_submission)
    # xem thử vài kết quả đầu tiên
    for qid in list(private_submission.keys())[:3]:
        print(qid, private_submission[qid])
else:
    print(f"Chưa có file {PRIVATE_TEST_PATH} — cập nhật DATA_DIR khi đề bài công bố dữ liệu chính thức.")


Predicting private-official.json: 100%|██████████| 2080/2080 [21:05<00:00,  1.64it/s]

Đã lưu: /mnt/mmlab2024nas/trantran/submission_7list3.json và /mnt/mmlab2024nas/trantran/submission_7list3.zip
44474 {'answer': ['200618', '292631', '222640', '115380', '253656']}
77938 {'answer': ['52552', '298075', '210883', '35448', '146911']}
33616 {'answer': ['245154', '122159', '98892', '68399', '43981']}
